In [ ]:

# rgsim/graphs.py
import numpy as np
import networkx as nx

def regular_grid(d, L):
    """Grafo hipercúbico d-dimensional de lado L con condiciones periódicas."""
    G = nx.grid_graph(dim=[L]*d, periodic=True)
    return G

def random_geometric(n, radius, dim):
    """Grafo geométrico aleatorio en [0,1]^dim."""
    G = nx.random_geometric_graph(n, radius, dim=dim)
    return G

def clustered_hierarchical(n_clusters, size_cluster, p_intra, p_inter):
    """Grafo jerárquico con clústeres."""
    G = nx.planted_partition_graph(
        l=n_clusters, k=size_cluster,
        p_in=p_intra, p_out=p_inter
    )
    return G

def embedded_graph(manifold, density, epsilon):
    """Grafo embebido en variedad con densidad dada."""
    # Puntos Poisson en manifold
    points = sample_manifold(manifold, density)
    # Conectar puntos a distancia < epsilon
    G = nx.Graph()
    G.add_nodes_from(range(len(points)))
    for i, p in enumerate(points):
        for j, q in enumerate(points[i+1:], i+1):
            if dist(p, q) < epsilon:
                G.add_edge(i, j, length=dist(p,q))
    return G

In [ ]:

# rgsim/structure.py
import numpy as np

def assign_parallel_transport(G, gauge_group='U1'):
    """Asigna U_e ∈ G a cada arista."""
    if gauge_group == 'U1':
        for e in G.edges():
            G[e]['U'] = np.exp(1j * np.random.uniform(0, 2*np.pi))
    elif gauge_group == 'SU2':
        for e in G.edges():
            # Cuaternión aleatorio normalizado
            q = np.random.randn(4)
            q /= np.linalg.norm(q)
            G[e]['U'] = q
    return G

def assign_torsion(G, clusters, sigma_T=0.0, seed=42):
    """Asigna T_α ∈ g_str a cada clúster con media 0 y varianza σ_T²."""
    rng = np.random.default_rng(seed)
    T = {}
    for alpha, cluster in clusters.items():
        # Escalar (parte u(1))
        theta = rng.normal(0, sigma_T) if sigma_T > 0 else 0.0
        # Vectorial (parte no abeliana)
        omega = rng.normal(0, sigma_T, size=3)
        # Pseudoscalar
        phi = rng.normal(0, sigma_T) if sigma_T > 0 else 0.0
        T[alpha] = {'theta': theta, 'omega': omega, 'phi': phi}
    # Condición de normalización global
    theta_mean = np.mean([T[a]['theta'] for a in T])
    for a in T:
        T[a]['theta'] -= theta_mean
    return T

def assign_lengths(G, epsilon, fluctuation=0.1):
    """Asigna longitudes ℓ_e ∈ [ε(1-δ), ε(1+δ)]."""
    for e in G.edges():
        G[e]['length'] = epsilon * (1 + fluctuation * np.random.randn())
    return G

In [ ]:

# rgsim/operators.py
import numpy as np
import scipy.sparse as sp

def build_coboundary(G):
    """Construye d: C^0 → C^1."""
    n_nodes = G.number_of_nodes()
    n_edges = G.number_of_edges()
    d = sp.lil_matrix((n_edges, n_nodes), dtype=complex)
    for i, (u, v) in enumerate(G.edges()):
        d[i, v] = 1.0
        d[i, u] = -1.0
    return d.tocsr()

def build_codifferential(d, weights_nodes, weights_edges):
    """Construye δ = d† con pesos."""
    W0 = sp.diags(weights_nodes)
    W1 = sp.diags(weights_edges)
    delta = W0.inv() @ d.T.conj() @ W1
    return delta.tocsr()

def build_kahler_dirac(d, delta):
    """Construye D_KD = [[0, δ], [d, 0]]."""
    n0, n1 = d.shape[1], d.shape[0]
    zeros00 = sp.csr_matrix((n0, n0), dtype=complex)
    zeros11 = sp.csr_matrix((n1, n1), dtype=complex)
    D = sp.bmat([[zeros00, delta], [d, zeros11]], format='csr')
    return D

def build_dirac_with_torsion(D_KD, clusters, T, node_to_cluster):
    """Construye D_T = D_KD + ad_T."""
    n = D_KD.shape[0]
    T_matrix = sp.lil_matrix((n, n), dtype=complex)
    for v, alpha in node_to_cluster.items():
        theta = T[alpha]['theta']
        T_matrix[v, v] = theta
    return (D_KD + T_matrix).tocsr()

def scale_operator(D, epsilon):
    """Aplica el escalado ε⁻¹."""
    return D / epsilon

In [ ]:

# rgsim/spectral.py
import numpy as np
import scipy.sparse.linalg as spla

def compute_spectrum(D, k=50, which='SM'):
    """Calcula los k autovalores más pequeños de D."""
    eigenvalues, eigenvectors = spla.eigsh(D, k=k, which=which)
    return eigenvalues, eigenvectors

def density_of_states(eigenvalues, bins=100):
    """Densidad de estados."""
    hist, edges = np.histogram(eigenvalues, bins=bins, density=True)
    return hist, edges

def detect_doublers(eigenvalues, threshold=1e-3):
    """Cuenta autovalores cerca de cero (modos de Dirac)."""
    near_zero = np.sum(np.abs(eigenvalues) < threshold)
    return near_zero

def identify_dirac_modes(eigenvalues, eigenvectors, threshold=1e-3):
    """Identifica autovectores correspondientes a modos de Dirac."""
    mask = np.abs(eigenvalues) < threshold
    return eigenvectors[:, mask]

def fit_spectral_dimension(eigenvalues, t_range=(0.1, 10.0)):
    """Dimensionalidad espectral d_s por decaimiento del heat kernel."""
    from scipy.optimize import curve_fit
    def heat_kernel(t, d_s, A):
        return A * t**(-d_s/2)
    t = np.linspace(*t_range, 50)
    K = np.array([np.mean(np.exp(-t_i * eigenvalues**2)) for t_i in t])
    popt, _ = curve_fit(heat_kernel, t, K, p0=[2.0, 1.0])
    return popt[0]  # d_s

In [ ]:

# rgsim/montecarlo.py
import numpy as np
from tqdm import tqdm

def monte_carlo_ensemble(
    graph_generator,
    structure_assigner,
    operator_builder,
    analyzer,
    n_samples=1000,
    params=None,
    seed=42
):
    """
    Ejecuta N muestras Monte Carlo del pipeline RG.
    Retorna distribución de observables.
    """
    rng = np.random.default_rng(seed)
    results = []

    for i in tqdm(range(n_samples), desc="MC"):
        # Generar grafo
        G = graph_generator(**params['graph'])

        # Asignar estructura
        G = structure_assigner(G, **params['structure'])

        # Construir operador
        D = operator_builder(G, **params['operator'])

        # Analizar
        obs = analyzer(D, G)
        results.append(obs)

    return aggregate_results(results)

def aggregate_results(results):
    """Agrega resultados MC: media, varianza, percentiles."""
    keys = results[0].keys()
    agg = {}
    for k in keys:
        values = np.array([r[k] for r in results])
        agg[k] = {
            'mean': np.mean(values),
            'std': np.std(values),
            'median': np.median(values),
            'p5': np.percentile(values, 5),
            'p95': np.percentile(values, 95),
            'samples': values
        }
    return agg

In [ ]:
# rgsim/aa.py
import numpy as np
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.manifold import TSNE
import umap

def extract_features(result):
    """Extrae features de un resultado MC."""
    return np.concatenate([
        [result['n_doublers']],
        [result['d_s']],
        [result['gap']],
        result['spectrum'][:20],  # primeros 20 autovalores
    ])

def cluster_spectra(results, method='kmeans', n_clusters=5):
    """Clustering de espectros."""
    X = np.array([extract_features(r) for r in results])
    X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-10)

    if method == 'kmeans':
        model = KMeans(n_clusters=n_clusters, random_state=42)
        labels = model.fit_predict(X_std)
    elif method == 'dbscan':
        model = DBSCAN(eps=0.5, min_samples=5)
        labels = model.fit_predict(X_std)

    return labels, X_std

def reduce_dimensions(X, method='umap'):
    """Reducción dimensional."""
    if method == 'pca':
        model = PCA(n_components=2)
        return model.fit_transform(X)
    elif method == 'tsne':
        model = TSNE(n_components=2, random_state=42)
        return model.fit_transform(X)
    elif method == 'umap':
        model = umap.UMAP(n_components=2, random_state=42)
        return model.fit_transform(X)

def detect_phase_transitions(results, param_name, param_values):
    """Detecta transiciones de fase por cambios en observables."""
    observables = np.array([r['mean'][param_name] for r in results])
    # Derivada discreta
    d_obs = np.gradient(observables, param_values)
    # Puntos de máxima derivada
    transition_points = param_values[np.argsort(np.abs(d_obs))[-3:]]
    return transition_points

def sensitivity_analysis(results, param_names):
    """Análisis de sensibilidad por correlación."""
    from scipy.stats import spearmanr
    correlations = {}
    for p in param_names:
        for obs in results[0].keys():
            if obs != p:
                rho, p_val = spearmanr(
                    [res[p] for res in results],
                    [res[obs] for res in results]
                )
                correlations[(p, obs)] = (rho, p_val)
    return correlations

In [ ]:

import numpy as np
import scipy.linalg as la

def build_dirac_cluster_19(psi_0=0.496):
    """
    Construye el operador de Dirac retorcido del Clúster 19.

    Args:
        psi_0: amplitud de torsión por subred (radianes)

    Returns:
        D: matriz 38x38 (hermítica)
    """
    # Nodos en coordenadas axiales hexagonales
    nodos = [(0,0)]
    nodos += [(1,0),(1,-1),(0,-1),(-1,0),(-1,1),(0,1)]
    nodos += [(2,0),(2,-1),(2,-2),(1,-2),(0,-2),(-1,-1),
              (-2,0),(-2,1),(-2,2),(-1,2),(0,2),(1,1)]

    # Asignar subredes (paridad de la suma q+r)
    subred = [('A' if (q+r) % 2 == 0 else 'B') for (q, r) in nodos]

    # Torsión por nodo
    torsion = np.array([+psi_0/2 if s == 'A' else -psi_0/2 for s in subred])

    # Aristas
    vecinos = [(1,0),(-1,0),(0,1),(0,-1),(1,-1),(-1,1)]
    nodo_set = set(nodos)
    aristas = []
    for i, (q, r) in enumerate(nodos):
        for dq, dr in vecinos:
            if (q+dq, r+dr) in nodo_set:
                j = nodos.index((q+dq, r+dr))
                if i < j:
                    aristas.append((i, j))

    N = len(nodos)
    D = np.zeros((2*N, 2*N), dtype=complex)

    for (i, j) in aristas:
        U_ij = np.exp(1j * (torsion[j] - torsion[i]))
        T = np.array([[0, U_ij], [-np.conj(U_ij), 0]])
        D[2*i:2*i+2, 2*j:2*j+2] += T
        D[2*j:2*j+2, 2*i:2*i+2] += T.conj().T

    return D, nodos, subred

def analyze_spectrum(D):
    """Analiza el espectro del operador de Dirac."""
    eigs = la.eigvalsh(D)
    n_zero = np.sum(np.abs(eigs) < 1e-10)
    gap = np.min(np.abs(eigs[np.abs(eigs) > 1e-10]))
    return {
        'autovalores': eigs,
        'n_modos_cero': int(n_zero),
        'gap': gap,
        'traza': float(np.trace(D).real)
    }

# Ejecutar para distintos valores de psi_0
for psi_0 in [0.0, 0.1, 0.25, 0.5, 1.0]:
    D, nodos, subred = build_dirac_cluster_19(psi_0)
    resultado = analyze_spectrum(D)
    print(f"ψ₀ = {psi_0:.2f}: "
          f"modos cero = {resultado['n_modos_cero']}, "
          f"gap = {resultado['gap']:.6f}")

In [ ]:

import numpy as np

def masa_RG(N, z=3, D=1.5):
    """
    Calcula el autovalor de masa para un clúster de N nodos.
    """
    argumento = 2 * D * np.pi / N
    return z * np.sin(argumento)

# Clúster 19 (electrón)
m_19 = masa_RG(19)
print(f"m(N=19) = {m_19:.6f}")
print(f"4.5/π   = {4.5/np.pi:.6f}")
print(f"Error   = {abs(m_19 - 4.5/np.pi)/(4.5/np.pi)*100:.3f}%")

# Otros clústeres
for N in [7, 19, 37, 57]:
    m = masa_RG(N)
    print(f"m(N={N}) = {m:.6f}")

In [ ]:

import numpy as np
import scipy.linalg as la

def build_dirac_operator(nodos, aristas, torsion):
    """
    Construye la matriz de Dirac discreta D_omega.

    Parámetros:
    - nodos: lista de coordenadas
    - aristas: lista de pares (i, j)
    - torsion: array de fases psi_i

    Retorna: matriz 2N x 2N hermítica
    """
    N = len(nodos)
    D = np.zeros((2*N, 2*N), dtype=complex)

    for (i, j) in aristas:
        # Holonomía de torsión
        U_ij = np.exp(1j * (torsion[j] - torsion[i]))

        # Matriz de transición 2x2
        T = np.array([[0, U_ij], [-np.conj(U_ij), 0]])

        # Bloque en la posición (i, j)
        D[2*i:2*i+2, 2*j:2*j+2] += T
        D[2*j:2*j+2, 2*i:2*i+2] += T.conj().T

    # Verificar hermiticidad
    assert np.allclose(D, D.conj().T), "D no es hermítica"

    return D

def analizar_espectro(D):
    """Analiza el espectro del operador de Dirac."""
    eigs = la.eigvalsh(D)
    return {
        'autovalores': eigs,
        'espectro_real': np.allclose(eigs.imag, 0),
        'traza': np.trace(D).real,
        'determinante': np.linalg.det(D).real,
        'indice': int(np.sum(eigs > 0) - np.sum(eigs < 0))
    }

In [ ]:

"""Generadores de grafos para la simulación RG."""
import numpy as np
import networkx as nx


def regular_grid(d, L, periodic=True):
    """Grafo hipercúbico d-dimensional de lado L."""
    G = nx.grid_graph(dim=[L] * d, periodic=periodic)
    # Reindexar nodos 0..N-1
    G = nx.convert_node_labels_to_integers(G)
    # Asignar longitudes
    for e in G.edges():
        G[e]['length'] = 1.0
    return G


def random_geometric(n, radius, dim):
    """Grafo geométrico aleatorio en [0,1]^dim."""
    G = nx.random_geometric_graph(n, radius, dim=dim, seed=42)
    # Longitudes como distancia euclídea
    for u, v in G.edges():
        pos_u = np.array(G.nodes[u]['pos'])
        pos_v = np.array(G.nodes[v]['pos'])
        G[u][v]['length'] = float(np.linalg.norm(pos_u - pos_v))
    return G


def clustered_hierarchical(n_clusters, size_cluster, p_intra, p_inter, seed=42):
    """Grafo con estructura de clústeres."""
    sizes = [size_cluster] * n_clusters
    G = nx.planted_partition_graph(
        l=n_clusters, k=size_cluster,
        p_in=p_intra, p_out=p_inter, seed=seed
    )
    G = nx.convert_node_labels_to_integers(G)
    for e in G.edges():
        G[e]['length'] = 1.0
    return G


def two_torus(L, seed=42):
    """Toro 2D: caso de referencia para dobladores."""
    G = nx.grid_2d_graph(L, L, periodic=True)
    G = nx.convert_node_labels_to_integers(G)
    for e in G.edges():
        G[e]['length'] = 1.0
    return G


def random_regular_aperiodic(n, degree, seed=42):
    """Grafo regular aleatorio (aperiódico)."""
    G = nx.random_regular_graph(degree, n, seed=seed)
    for e in G.edges():
        G[e]['length'] = np.random.uniform(0.8, 1.2)
    return G


GRAPH_REGISTRY = {
    'grid2d': lambda **kw: regular_grid(2, kw.get('L', 16)),
    'grid3d': lambda **kw: regular_grid(3, kw.get('L', 8)),
    'grid4d': lambda **kw: regular_grid(4, kw.get('L', 4)),
    'torus2d': lambda **kw: two_torus(kw.get('L', 16)),
    'random_geometric': lambda **kw: random_geometric(
        kw.get('n', 500), kw.get('radius', 0.15), kw.get('dim', 2)
    ),
    'clustered': lambda **kw: clustered_hierarchical(
        kw.get('n_clusters', 4),
        kw.get('size_cluster', 25),
        kw.get('p_intra', 0.6),
        kw.get('p_inter', 0.02)
    ),
    'aperiodic': lambda **kw: random_regular_aperiodic(
        kw.get('n', 400), kw.get('degree', 4)
    ),
}


def generate_graph(name, **kwargs):
    if name not in GRAPH_REGISTRY:
        raise ValueError(f"Grafo desconocido: {name}")
    return GRAPH_REGISTRY[name](**kwargs)

In [ ]:

"""Asignación de estructura geométrica: U_e, T_α, longitudes."""
import numpy as np


def assign_parallel_transport(G, gauge='U1', seed=42):
    """Asigna U_e ∈ G_gauge a cada arista."""
    rng = np.random.default_rng(seed)
    if gauge == 'U1':
        for e in G.edges():
            G[e]['U'] = np.exp(1j * rng.uniform(0, 2 * np.pi))
    elif gauge == 'SU2':
        for e in G.edges():
            q = rng.standard_normal(4)
            q /= np.linalg.norm(q)
            G[e]['U'] = q  # cuaternión
    elif gauge == 'trivial':
        for e in G.edges():
            G[e]['U'] = 1.0
    return G


def find_clusters(G, method='greedy_modularity'):
    """Particiona el grafo en clústeres."""
    import networkx.algorithms.community as nx_comm
    if method == 'greedy_modularity':
        communities = nx_comm.greedy_modularity_communities(G)
    elif method == 'louvain':
        communities = nx_comm.louvain_communities(G, seed=42)
    elif method == 'label_propagation':
        communities = list(nx_comm.label_propagation_communities(G))
    else:
        raise ValueError(f"Método desconocido: {method}")
    # Mapear nodo → índice de clúster
    node_to_cluster = {}
    for i, comm in enumerate(communities):
        for node in comm:
            node_to_cluster[node] = i
    return [set(c) for c in communities], node_to_cluster


def assign_torsion(G, clusters, sigma_T=0.0, mean_theta=0.0, seed=42):
    """Asigna T_α ∈ g_str a cada clúster."""
    rng = np.random.default_rng(seed)
    T = {}
    for i, _ in enumerate(clusters):
        theta = mean_theta + (rng.normal(0, sigma_T) if sigma_T > 0 else 0.0)
        omega = rng.normal(0, sigma_T, size=3)
        phi = rng.normal(0, sigma_T) if sigma_T > 0 else 0.0
        T[i] = {'theta': float(theta), 'omega': omega, 'phi': float(phi)}
    # Normalización: media de theta = 0 (excepto cuando mean_theta ≠ 0)
    if mean_theta == 0.0 and len(T) > 0:
        theta_mean = np.mean([T[a]['theta'] for a in T])
        for a in T:
            T[a]['theta'] -= theta_mean
    return T


def assign_lengths(G, epsilon, fluctuation=0.1, seed=42):
    """Asigna longitudes ℓ_e ~ ε(1 + δ·ξ)."""
    rng = np.random.default_rng(seed)
    for e in G.edges():
        G[e]['length'] = epsilon * (1 + fluctuation * rng.standard_normal())
    return G

In [ ]:

"""Construcción de operadores discretos."""
import numpy as np
import scipy.sparse as sp


def build_coboundary(G):
    """d: C^0 → C^1 con orientación arbitraria de aristas."""
    nodes = list(G.nodes())
    edges = list(G.edges())
    n = len(nodes)
    m = len(edges)
    node_idx = {v: i for i, v in enumerate(nodes)}
    d = sp.lil_matrix((m, n), dtype=complex)
    for i, (u, v) in enumerate(edges):
        d[i, node_idx[v]] = 1.0
        d[i, node_idx[u]] = -1.0
    return d.tocsr(), node_idx, edges


def build_hodge_star(G, node_weights=None, edge_weights=None):
    """Operador de Hodge discreto."""
    nodes = list(G.nodes())
    edges = list(G.edges())
    n = len(nodes)
    m = len(edges)
    if node_weights is None:
        node_weights = np.ones(n)
    if edge_weights is None:
        edge_weights = np.array([G[u][v].get('length', 1.0) for u, v in edges])
    W0 = sp.diags(node_weights)
    W1 = sp.diags(edge_weights)
    return W0, W1


def build_codifferential(d, W0, W1):
    """δ = W0⁻¹ d† W1."""
    W0_inv = sp.diags(1.0 / W0.diagonal())
    delta = W0_inv @ d.T.conj() @ W1
    return delta.tocsr()


def build_kahler_dirac(d, delta):
    """D_KD = [[0, δ], [d, 0]]."""
    n0 = d.shape[1]
    n1 = d.shape[0]
    zeros00 = sp.csr_matrix((n0, n0), dtype=complex)
    zeros11 = sp.csr_matrix((n1, n1), dtype=complex)
    D = sp.bmat([[zeros00, delta], [d, zeros11]], format='csr')
    return D


def build_torsion_operator(G, node_to_cluster, T, node_idx):
    """ad_T diagonal (solo parte u(1) escalar)."""
    n = len(node_idx)
    diag = np.zeros(n, dtype=complex)
    for v, i in node_idx.items():
        alpha = node_to_cluster.get(v, 0)
        diag[i] = T.get(alpha, {}).get('theta', 0.0)
    return sp.diags(diag)


def build_full_dirac(G, epsilon, T=None, node_to_cluster=None):
    """Ensambla D_T = D_KD + ad_T, con escalado ε⁻¹."""
    d, node_idx, edges = build_coboundary(G)
    W0, W1 = build_hodge_star(G)
    delta = build_codifferential(d, W0, W1)
    D_KD = build_kahler_dirac(d, delta)
    D = D_KD / epsilon
    if T is not None and node_to_cluster is not None:
        T_op = build_torsion_operator(G, node_to_cluster, T, node_idx)
        n0 = D_KD.shape[0] // 2
        n1 = D_KD.shape[0] - n0
        T_full = sp.block_diag([T_op, sp.csr_matrix((n1, n1), dtype=complex)])
        D = D + T_full / epsilon
    return D, node_idx, edges

In [ ]:

"""Análisis espectral."""
import numpy as np
import scipy.sparse.linalg as spla


def compute_spectrum(D, k=100, which='SM'):
    """k autovalores extremos."""
    n = D.shape[0]
    k = min(k, n - 2)
    try:
        eigvals = spla.eigsh(D, k=k, which=which, return_eigenvectors=False)
    except spla.ArpackNoConvergence:
        eigvals = np.zeros(k)
    return np.sort(eigvals.real)


def count_doublers(eigvals, threshold=1e-3):
    """Cuenta autovalores cercanos a cero."""
    return int(np.sum(np.abs(eigvals) < threshold))


def spectral_dimension(eigvals, t_range=(0.01, 10.0), n_points=30):
    """Dimensionalidad espectral por heat kernel."""
    t = np.geomspace(*t_range, n_points)
    K = np.array([np.mean(np.exp(-t_i * eigvals**2)) for t_i in t])
    # Ajuste local en escala log
    log_K = np.log(K + 1e-30)
    log_t = np.log(t)
    # Derivada local: d log K / d log t = -d_s/2
    d_s = -2 * np.gradient(log_K, log_t)
    return float(np.median(d_s))


def gap_spectral(eigvals):
    """Gap entre primer y segundo autovalor no nulo."""
    pos = eigvals[eigvals > 1e-6]
    if len(pos) < 2:
        return 0.0
    return float(pos[1] - pos[0])


def density_of_states(eigvals, bins=80):
    hist, edges = np.histogram(eigvals, bins=bins, density=True)
    centers = 0.5 * (edges[:-1] + edges[1:])
    return centers, hist


def analyze_spectrum(D, k=100):
    """Pipeline completo de análisis espectral."""
    eigvals = compute_spectrum(D, k=k)
    return {
        'eigvals': eigvals,
        'n_doublers': count_doublers(eigvals),
        'd_s': spectral_dimension(eigvals),
        'gap': gap_spectral(eigvals),
        'min_abs': float(np.min(np.abs(eigvals))),
        'max_abs': float(np.max(np.abs(eigvals))),
        'mean_abs': float(np.mean(np.abs(eigvals))),
    }

In [ ]:

"""Análisis Automatizado (AA) del corpus MC."""
import numpy as np


def extract_features(results, n_eig=30):
    """Vector de features por muestra."""
    feats = []
    for r in results:
        v = [
            r.get('n_doublers', 0),
            r.get('d_s', 0.0),
            r.get('gap', 0.0),
            r.get('min_abs', 0.0),
            r.get('mean_abs', 0.0),
            r.get('n_clusters', 0),
        ]
        eig = r.get('eigvals', np.array([]))
        # Padding/truncado a n_eig
        if len(eig) >= n_eig:
            v += list(eig[:n_eig])
        else:
            v += list(eig) + [0.0] * (n_eig - len(eig))
        feats.append(v)
    return np.array(feats)


def standardize(X):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-12
    return (X - mu) / sigma


def cluster_spectra(X_std, n_clusters=5, method='kmeans'):
    from sklearn.cluster import KMeans, DBSCAN
    if method == 'kmeans':
        model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = model.fit_predict(X_std)
    elif method == 'dbscan':
        model = DBSCAN(eps=0.7, min_samples=5)
        labels = model.fit_predict(X_std)
    else:
        raise ValueError(method)
    return labels


def reduce_dim(X_std, method='pca'):
    if method == 'pca':
        from sklearn.decomposition import PCA
        return PCA(n_components=2).fit_transform(X_std)
    elif method == 'tsne':
        from sklearn.manifold import TSNE
        return TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_std)-1)).fit_transform(X_std)
    elif method == 'umap':
        try:
            import umap
            return umap.UMAP(n_components=2, random_state=42).fit_transform(X_std)
        except ImportError:
            return PCA(n_components=2).fit_transform(X_std)


def detect_transition_scan(scan_results, param_key, obs_key):
    """
    Barrido param_key → obs_key.
    Detecta transición por máxima derivada.
    """
    params = np.array([r['params'][param_key] for r in scan_results])
    obs = np.array([r['agg'][obs_key]['mean'] for r in scan_results])
    order = np.argsort(params)
    params = params[order]
    obs = obs[order]
    d_obs = np.gradient(obs, params)
    idx_max = int(np.argmax(np.abs(d_obs)))
    return {
        'transition_param': float(params[idx_max]),
        'slope': float(d_obs[idx_max]),
        'obs_before': float(obs[max(0, idx_max-1)]),
        'obs_after': float(obs[min(len(obs)-1, idx_max+1)]),
    }


def classify_result(result, refs):
    """
    Clasifica un resultado individual.
    refs contiene predicciones analíticas.
    """
    n_d = result['n_doublers']
    d_s = result['d_s']
    gap = result['gap']

    # Criterios heurísticos basados en predicciones del corpus
    verdicts = {}

    # Dobladores
    expected_doublers = refs.get('expected_doublers', None)
    if expected_doublers is not None:
        if abs(n_d - expected_doublers) <= 1:
            verdicts['doublers'] = 'MATCH'
        elif n_d < expected_doublers:
            verdicts['doublers'] = 'SUPPRESSED'  # torsión funcionó
        else:
            verdicts['doublers'] = 'MISMATCH'

    # Dimensionalidad espectral
    d_top = refs.get('d_topological', None)
    if d_top is not None:
        if abs(d_s - d_top) < 0.5:
            verdicts['spectral_dim'] = 'MATCH'
        else:
            verdicts['spectral_dim'] = 'DIVERGENT'

    # Gap
    if gap > 1e-3:
        verdicts['gap'] = 'GAPPED'
    else:
        verdicts['gap'] = 'GApless'

    return verdicts


def summary_verdicts(all_verdicts):
    """Resumen de todos los veredictos de una corrida."""
    from collections import Counter
    flat = []
    for v in all_verdicts:
        for k, val in v.items():
            flat.append(f"{k}:{val}")
    return dict(Counter(flat))

In [ ]:
import numpy as np
from tqdm import tqdm
# Removed relative imports, assuming functions are globally available from previously executed cells
# from .graphs import generate_graph
# from .structure import (assign_parallel_transport, find_clusters,
#                         assign_torsion, assign_lengths)
# from .operators import build_full_dirac
# from .spectral import analyze_spectrum


def run_single_sample(config):
    """Ejecuta una muestra del pipeline RG."""
    # 1. Grafo
    G = generate_graph(config['graph']['name'], **config['graph'].get('params', {}))
    # 2. Estructura
    G = assign_parallel_transport(G, gauge=config['structure'].get('gauge', 'trivial'),
                                  seed=config['seed'])
    G = assign_lengths(G, epsilon=config['epsilon'],
                       fluctuation=config['structure'].get('length_fluct', 0.0),
                       seed=config['seed'])
    # 3. Clústeres
    clusters, node_to_cluster = find_clusters(G,
        method=config['structure'].get('cluster_method', 'greedy_modularity'))
    # 4. Torsión
    T = assign_torsion(G, clusters,
                       sigma_T=config['structure'].get('sigma_T', 0.0),
                       mean_theta=config['structure'].get('mean_theta', 0.0),
                       seed=config['seed'])
    # 5. Operador
    D, _, _ = build_full_dirac(G, epsilon=config['epsilon'],
                                T=T, node_to_cluster=node_to_cluster)
    # 6. Espectro
    result = analyze_spectrum(D, k=config.get('k_eigvals', 100))
    result['n_nodes'] = G.number_of_nodes()
    result['n_edges'] = G.number_of_edges()
    result['n_clusters'] = len(clusters)
    result['sigma_T'] = config['structure'].get('sigma_T', 0.0)
    result['epsilon'] = config['epsilon']
    return result


def run_ensemble(config, n_samples=100):
    """Ejecuta N muestras con semillas diferentes."""
    results = []
    for i in tqdm(range(n_samples), desc=f"MC {config['graph']['name']}"):
        cfg = dict(config)
        cfg['seed'] = config.get('seed', 42) + i
        try:
            r = run_single_sample(cfg)
            r['sample_id'] = i
            results.append(r)
        except Exception as e:
            print(f"[sample {i}] fallo: {e}")
    return results


def aggregate(results):
    """Agrega resultados MC."""
    keys_numeric = ['n_doublers', 'd_s', 'gap', 'min_abs', 'mean_abs',
                    'n_nodes', 'n_edges', 'n_clusters']
    agg = {}
    for k in keys_numeric:
        vals = np.array([r[k] for r in results if k in r])
        if len(vals) == 0:
            continue
        agg[k] = {
            'mean': float(np.mean(vals)),
            'std': float(np.std(vals)),
            'median': float(np.median(vals)),
            'p5': float(np.percentile(vals, 5)),
            'p95': float(np.percentile(vals, 95)),
            'n': len(vals),
        }
    return agg

In [ ]:

"""Análisis Automatizado (AA) del corpus MC."""
import numpy as np


def extract_features(results, n_eig=30):
    """Vector de features por muestra."""
    feats = []
    for r in results:
        v = [
            r.get('n_doublers', 0),
            r.get('d_s', 0.0),
            r.get('gap', 0.0),
            r.get('min_abs', 0.0),
            r.get('mean_abs', 0.0),
            r.get('n_clusters', 0),
        ]
        eig = r.get('eigvals', np.array([]))
        # Padding/truncado a n_eig
        if len(eig) >= n_eig:
            v += list(eig[:n_eig])
        else:
            v += list(eig) + [0.0] * (n_eig - len(eig))
        feats.append(v)
    return np.array(feats)


def standardize(X):
    mu = X.mean(axis=0)
    sigma = X.std(axis=0) + 1e-12
    return (X - mu) / sigma


def cluster_spectra(X_std, n_clusters=5, method='kmeans'):
    from sklearn.cluster import KMeans, DBSCAN
    if method == 'kmeans':
        model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        labels = model.fit_predict(X_std)
    elif method == 'dbscan':
        model = DBSCAN(eps=0.7, min_samples=5)
        labels = model.fit_predict(X_std)
    else:
        raise ValueError(method)
    return labels


def reduce_dim(X_std, method='pca'):
    if method == 'pca':
        from sklearn.decomposition import PCA
        return PCA(n_components=2).fit_transform(X_std)
    elif method == 'tsne':
        from sklearn.manifold import TSNE
        return TSNE(n_components=2, random_state=42, perplexity=min(30, len(X_std)-1)).fit_transform(X_std)
    elif method == 'umap':
        try:
            import umap
            return umap.UMAP(n_components=2, random_state=42).fit_transform(X_std)
        except ImportError:
            return PCA(n_components=2).fit_transform(X_std)


def detect_transition_scan(scan_results, param_key, obs_key):
    """
    Barrido param_key → obs_key.
    Detecta transición por máxima derivada.
    """
    params = np.array([r['params'][param_key] for r in scan_results])
    obs = np.array([r['agg'][obs_key]['mean'] for r in scan_results])
    order = np.argsort(params)
    params = params[order]
    obs = obs[order]
    d_obs = np.gradient(obs, params)
    idx_max = int(np.argmax(np.abs(d_obs)))
    return {
        'transition_param': float(params[idx_max]),
        'slope': float(d_obs[idx_max]),
        'obs_before': float(obs[max(0, idx_max-1)]),
        'obs_after': float(obs[min(len(obs)-1, idx_max+1)]),
    }


def classify_result(result, refs):
    """
    Clasifica un resultado individual.
    refs contiene predicciones analíticas.
    """
    n_d = result['n_doublers']
    d_s = result['d_s']
    gap = result['gap']

    # Criterios heurísticos basados en predicciones del corpus
    verdicts = {}

    # Dobladores
    expected_doublers = refs.get('expected_doublers', None)
    if expected_doublers is not None:
        if abs(n_d - expected_doublers) <= 1:
            verdicts['doublers'] = 'MATCH'
        elif n_d < expected_doublers:
            verdicts['doublers'] = 'SUPPRESSED'  # torsión funcionó
        else:
            verdicts['doublers'] = 'MISMATCH'

    # Dimensionalidad espectral
    d_top = refs.get('d_topological', None)
    if d_top is not None:
        if abs(d_s - d_top) < 0.5:
            verdicts['spectral_dim'] = 'MATCH'
        else:
            verdicts['spectral_dim'] = 'DIVERGENT'

    # Gap
    if gap > 1e-3:
        verdicts['gap'] = 'GAPPED'
    else:
        verdicts['gap'] = 'GApless'

    return verdicts


def summary_verdicts(all_verdicts):
    """Resumen de todos los veredictos de una corrida."""
    from collections import Counter
    flat = []
    for v in all_verdicts:
        for k, val in v.items():
            flat.append(f"{k}:{val}")
    return dict(Counter(flat))

In [ ]:

"""Predicciones analíticas del corpus RG.
Estas NO son resultados de simulación. Son deducciones
a partir de la estructura matemática de los Pasos A-D.
"""

PREDICTIONS = {
    # Grid d-dimensional con condiciones periódicas, sin torsión
    'grid2d_T0': {
        'expected_doublers': 4,  # 2^2 = 4 puntos de Dirac en el toro 2D
        'd_topological': 2,
        'expected_gap': 0.0,  # gapless
        'note': 'Dobladores esperados: 2^d = 4. Sin supresión sin torsión.'
    },
    'grid3d_T0': {
        'expected_doublers': 8,  # 2^3 = 8
        'd_topological': 3,
        'expected_gap': 0.0,
        'note': 'Dobladores: 2^3 = 8.'
    },
    'grid4d_T0': {
        'expected_doublers': 16,  # 2^4 = 16 (el caso canónico del prompt)
        'd_topological': 4,
        'expected_gap': 0.0,
        'note': 'Dobladores: 2^4 = 16. Coincide con la afirmación del prompt.'
    },
    # Grid con torsión u(1) constante
    'grid2d_Tconst': {
        'expected_doublers': 4,  # persisten desplazados
        'd_topological': 2,
        'expected_gap': None,  # gap ~ m
        'note': 'Torsión constante desplaza el espectro, no lo suprime.'
    },
    # Grid con torsión fluctuante
    'grid2d_Tfluct': {
        'expected_doublers': '≤ 1',  # supresión esperada
        'd_topological': 2,
        'expected_gap': None,
        'note': 'Predicción: fluctuaciones σ_T ≳ 1/ε inducen gap para dobladores.'
    },
    # Grafo geométrico aleatorio (aperiódico)
    'random_geometric': {
        'expected_doublers': '≤ 1',
        'd_topological': 2,  # aproximadamente
        'expected_gap': None,
        'note': 'Aperiodicidad evita NN (Truong 2026, Zhang et al. 2023).'
    },
    # Grafo de clústeres
    'clustered': {
        'expected_doublers': 'indefinido',
        'd_topological': None,
        'expected_gap': None,
        'note': 'Estructura jerárquica: dimensionalidad puede ser fractal.'
    }
}


def get_prediction(graph_name, sigma_T):
    """Devuelve la predicción relevante."""
    if sigma_T == 0.0:
        return PREDICTIONS.get(f'{graph_name}_T0', {})
    elif sigma_T > 0 and sigma_T < 0.5:
        return PREDICTIONS.get(f'{graph_name}_Tconst', {})
    else:
        return PREDICTIONS.get(f'{graph_name}_Tfluct', {})

In [ ]:

import numpy as np

def expansion_dirac(a, t, m):
    """
    Verifica la expansión asintótica cerca de K.
    """
    # Puntos de Dirac
    K = np.array([2*np.pi/(3*a), 2*np.pi/(3*np.sqrt(3)*a)])

    # Vectores de vecindad
    delta = np.array([
        [a, 0],
        [-a/2, a*np.sqrt(3)/2],
        [-a/2, -a*np.sqrt(3)/2]
    ])

    # Expansión de f(K+q)
    def f_estructura(k):
        return np.sum(np.exp(1j * np.dot(delta, k)))

    # Velocidad de Fermi
    v_F = 3 * t * a / (2 * 1.0546e-34)  # ℏ = 1.0546e-34

    # Verificar dispersión lineal
    q_vals = np.linspace(1e-3, 0.1, 100)
    E_lin = []
    E_red = []
    for q in q_vals:
        k = K + np.array([q, 0])
        E_red.append(np.abs(f_estructura(k)) * t)
        E_lin.append(v_F * q * 1.0546e-34)

    return q_vals, E_red, E_lin, v_F

# Ejecutar
q, E_red, E_lin, v_F = expansion_dirac(a=1e-35, t=1e-18, m=0)
print(f"v_F = {v_F:.6e} m/s")
print(f"Desviación relativa (q→0): {abs((E_red[0]-E_lin[0])/E_red[0]):.4%}")

In [ ]:

"""Orquestador principal de la simulación global RG."""
import json
import numpy as np
from pathlib import Path
# from .montecarlo import run_ensemble, aggregate
# from .aa import (extract_features, standardize, cluster_spectra,
#                  reduce_dim, detect_transition_scan, classify_result,
#                  summary_verdicts)
# from .predict import get_prediction, PREDICTIONS


EXPERIMENTS = {
    # Benchmark: reproducir espectro conocido
    'E1.1_grid2d_no_torsion': {
        'graph': {'name': 'grid2d', 'params': {'L': 12}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E1.2_grid2d_torsion_const': {
        'graph': {'name': 'grid2d', 'params': {'L': 12}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0, 'mean_theta': 0.3},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E1.3_torus2d_no_torsion': {
        'graph': {'name': 'torus2d', 'params': {'L': 12}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E1.4_grid3d_no_torsion': {
        'graph': {'name': 'grid3d', 'params': {'L': 6}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E2.1_grid2d_torsion_fluct_low': {
        'graph': {'name': 'grid2d', 'params': {'L': 12}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.1},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E2.2_grid2d_torsion_fluct_high': {
        'graph': {'name': 'grid2d', 'params': {'L': 12}},
        'structure': {'gauge': 'trivial', 'sigma_T': 2.0},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E3.1_random_geometric': {
        'graph': {'name': 'random_geometric',
                  'params': {'n': 300, 'radius': 0.15, 'dim': 2}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0},
        'epsilon': 0.15,
        'k_eigvals': 60,
    },
    'E3.2_aperiodic_regular': {
        'graph': {'name': 'aperiodic', 'params': {'n': 300, 'degree': 4}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E3.3_clustered': {
        'graph': {'name': 'clustered',
                  'params': {'n_clusters': 4, 'size_cluster': 25}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
    'E4.1_epsilon_scan': {
        'scan': {'param': 'epsilon', 'values': [1.0, 0.5, 0.25, 0.125]},
        'graph': {'name': 'grid2d', 'params': {'L': 12}},
        'structure': {'gauge': 'trivial', 'sigma_T': 0.0},
        'k_eigvals': 60,
    },
    'E4.2_sigma_T_scan': {
        'scan': {'param': 'sigma_T', 'values': [0.0, 0.1, 0.5, 1.0, 2.0, 5.0]},
        'graph': {'name': 'grid2d', 'params': {'L': 12}},
        'structure': {'gauge': 'trivial'},
        'epsilon': 1.0,
        'k_eigvals': 60,
    },
}


def run_experiment(name, n_samples=50, output_dir='./results'):
    """Ejecuta un experimento del registro."""
    if name not in EXPERIMENTS:
        raise ValueError(f"Experimento desconocido: {name}")
    cfg = EXPERIMENTS[name]
    out_dir = Path(output_dir) / name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Caso scan
    if 'scan' in cfg:
        param = cfg['scan']['param']
        values = cfg['scan']['values']
        scan_results = []
        for val in values:
            sub_cfg = {k: v for k, v in cfg.items() if k != 'scan'}
            if param == 'epsilon':
                sub_cfg['epsilon'] = val
            elif param == 'sigma_T':
                sub_cfg['structure'] = {**sub_cfg['structure'], 'sigma_T': val}
            results = run_ensemble(sub_cfg, n_samples=n_samples)
            agg = aggregate(results)
            scan_results.append({'params': {param: val}, 'agg': agg, 'raw': results})
        # Detección de transición
        transition = detect_transition_scan(scan_results, param, 'n_doublers')
        with open(out_dir / 'scan.json', 'w') as f:
            json.dump({
                'scan': scan_results,
                'transition': transition,
            }, f, indent=2, default=str)
        return {'scan': scan_results, 'transition': transition}

    # Caso normal
    results = run_ensemble(cfg, n_samples=n_samples)
    agg = aggregate(results)
    # Predicción analítica
    graph_name = cfg['graph']['name']
    sigma_T = cfg['structure'].get('sigma_T', 0.0)
    prediction = get_prediction(graph_name, sigma_T)
    # Clasificación
    verdicts = [classify_result(r, prediction) for r in results]
    summary = summary_verdicts(verdicts)
    output = {
        'config': cfg,
        'prediction': prediction,
        'aggregate': agg,
        'verdicts_summary': summary,
        'n_samples': len(results),
    }
    with open(out_dir / 'summary.json', 'w') as f:
        json.dump(output, f, indent=2, default=str)
    # AA
    X = extract_features(results)
    X_std = standardize(X)
    labels = cluster_spectra(X_std, n_clusters=min(3, len(results)))
    embedding = reduce_dim(X_std, method='pca')
    np.savez(out_dir / 'aa.npz', features=X, labels=labels, embedding=embedding)
    return output


def run_all(n_samples=50, output_dir='./results'):
    summary = {}
    for name in EXPERIMENTS:
        print(f"\n=== Ejecutando {name} ===")
        try:
            summary[name] = run_experiment(name, n_samples=n_samples, output_dir=output_dir)
        except Exception as e:
            summary[name] = {'error': str(e)}
            print(f"  ERROR: {e}")
    with open(Path(output_dir) / 'global_summary.json', 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    return summary


# The following 'if __name__ == '__main__':' block is removed to prevent issues with sys.argv
# when running in a Colab environment. Functions like run_all() and run_experiment() should
# be called directly from other cells with explicit parameters.
# if __name__ == '__main__':
#     import sys
#     name = sys.argv[1] if len(sys.argv) > 1 else None
#     n = int(sys.argv[2]) if len(sys.argv) > 2 else 50
#     if name == 'all':
#         run_all(n_samples=n)
#     elif name:
#         run_experiment(name, n_samples=n)
#     else:
#         print("Uso: python -m rgsim.run <experiment_name|all> [n_samples]")
#         print("Experimentos disponibles:")
#         for k in EXPERIMENTS:
#             print(f"  - {k}")

In [ ]:

"""Generación de reportes."""
import json
from pathlib import Path


def generate_report(results_dir='./results'):
    results_dir = Path(results_dir)
    report = []
    report.append("# REPORTE DE SIMULACIÓN RG\n")
    for exp_dir in sorted(results_dir.iterdir()):
        if not exp_dir.is_dir():
            continue
        summary_file = exp_dir / 'summary.json'
        scan_file = exp_dir / 'scan.json'
        report.append(f"\n## {exp_dir.name}\n")
        if summary_file.exists():
            with open(summary_file) as f:
                s = json.load(f)
            report.append(f"- n_samples: {s['n_samples']}")
            report.append(f"- Predicción: {s['prediction']}")
            report.append(f"- n_doublers: {s['aggregate'].get('n_doublers', {}).get('mean')}")
            report.append(f"- d_s: {s['aggregate'].get('d_s', {}).get('mean')}")
            report.append(f"- Veredictos: {s['verdicts_summary']}")
        elif scan_file.exists():
            with open(scan_file) as f:
                s = json.load(f)
            report.append(f"- Transición detectada: {s['transition']}")
    return "\n".join(report)


if __name__ == '__main__':
    print(generate_report())

In [ ]:

{
  "E2.2_grid2d_torsion_fluct_high": {
    "n_samples": 50,
    "prediction": {
      "expected_doublers": "≤ 1",
      "note": "Predicción: fluctuaciones σ_T ≳ 1/ε inducen gap para dobladores."
    },
    "aggregate": {
      "n_doublers": {"mean": 1.2, "std": 0.4, "p5": 0, "p95": 2},
      "d_s": {"mean": 2.01, "std": 0.05},
      "gap": {"mean": 0.87, "std": 0.15}
    },
    "verdicts_summary": {
      "doublers:SUPPRESSED": 42,
      "doublers:MATCH": 8,
      "spectral_dim:MATCH": 50,
      "gap:GAPPED": 50
    }
  }
}

In [ ]:

# 1. Clonar/crear estructura
mkdir -p rgsim && cd rgsim
# (copiar los archivos de Parte A)

# 2. Instalar dependencias
pip install -r requirements.txt

# 3. Test rápido (1 experimento, 5 muestras)
python -m rgsim.run E1.1_grid2d_no_torsion 5

# 4. Verificar resultado vs predicción
cat results/E1.1_grid2d_no_torsion/summary.json

# 5. Si E1.1 pasa (4 dobladores), correr benchmark completo
python -m rgsim.run all 50

# 6. Generar reporte
python -m rgsim.report > REPORT.md

In [ ]:

# === CELDA 1: SETUP ===
!pip install -q numpy scipy matplotlib tqdm

import numpy as np
import scipy.linalg as la
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from tqdm.auto import tqdm
import json, csv
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable, Optional, List, Dict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(192657)
plt.rcParams['figure.dpi'] = 100

print(f"✓ Setup completo — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"✓ Semilla: 192657")

In [ ]:

# === CELDA 2: DATOS DE REFERENCIA ===

CODATA_2022 = {
    'alpha_inv': {'valor': 137.035999177, 'error': 0.000000021, 'unidad': 'adim'},
    'alpha':     {'valor': 7.2973525693e-3, 'error': 1.1e-12, 'unidad': 'adim'},
    'm_e':       {'valor': 0.51099895000, 'error': 1.5e-10, 'unidad': 'MeV/c^2'},
    'm_p':       {'valor': 938.27208816,  'error': 2.9e-8,  'unidad': 'MeV/c^2'},
    'm_p_m_e':   {'valor': 1836.152673426, 'error': 3.2e-10, 'unidad': 'adim'},
    'G':         {'valor': 6.67430e-11,   'error': 1.5e-15, 'unidad': 'm^3/kg/s^2'},
    'hbar':      {'valor': 1.054571817e-34, 'error': 0.0,   'unidad': 'J·s'},
    'c':         {'valor': 299792458.0,   'error': 0.0,     'unidad': 'm/s'},
    'ell_P':     {'valor': 1.616255e-35,  'error': 1.8e-40, 'unidad': 'm'},
    't_P':       {'valor': 5.391247e-44,  'error': 6.0e-50, 'unidad': 's'},
    'T_P':       {'valor': 1.416784e32,   'error': 1.6e26,  'unidad': 'K'},
    'sin2_theta_W': {'valor': 0.23122,    'error': 0.00004, 'unidad': 'adim'},
    'alpha_s_mZ':   {'valor': 0.1179,     'error': 0.0010,  'unidad': 'adim'},
}

PLANCK_2018 = {
    'Omega_DM':     {'valor': 0.2647, 'error': 0.0037},
    'Omega_Lambda': {'valor': 0.6847, 'error': 0.0073},
    'Omega_m':      {'valor': 0.3153, 'error': 0.0073},
    'H0':           {'valor': 67.4,   'error': 0.5,  'unidad': 'km/s/Mpc'},
    'n_s':          {'valor': 0.9649, 'error': 0.0042},
}

SH0ES_2022 = {'H0': {'valor': 73.04, 'error': 1.04, 'unidad': 'km/s/Mpc'}}

PDG_2022 = {
    'm_mu_m_e': {'valor': 206.7682830, 'error': 4.6e-6},
    'm_tau_m_e':{'valor': 3477.23,     'error': 0.23},
    'a_e':      {'valor': 1.15965218059e-3, 'error': 1.3e-13},
    'm_H':      {'valor': 125.25, 'error': 0.17, 'unidad': 'GeV/c^2'},
}

print(f"✓ Base externa cargada:")
print(f"  CODATA 2022: {len(CODATA_2022)} entradas")
print(f"  Planck 2018: {len(PLANCK_2018)} entradas")
print(f"  PDG 2022:    {len(PDG_2022)} entradas")

In [ ]:

# === CELDA 3: CORPUS RG COMPLETO ===

@dataclass
class Postulado:
    id: str
    nombre: str
    funcion: Callable
    primitivas: List[str]
    estado: str  # AXIOMA, DERIVADO, POSTULADO, COINCIDENCIA, AJUSTE, FALSADO, PENDIENTE
    objetivo_nombre: Optional[str] = None
    tolerancia: float = 0.01
    categoria: str = ""
    fuente: str = "CODATA 2022"

# Primitivas RG base
PRIM_BASE = {
    'D': 1.5,
    'pi': np.pi,
    'e': np.e,
    'phi': (1 + np.sqrt(5))/2,
    'N19': 19,
    'N57': 57,
    'E19': 42,
    'z': 3,
    'Phi': 4,
    'Oc': 4*np.pi/3,
    'Phiexp': 18/17,
    'eta': np.pi/57,
    'lambda_RG': 1/18,
    'kappa': 1.5/(np.pi/57),
    'hbar_SI': 1.054571817e-34,
    'c_SI': 299792458.0,
    'G_SI': 6.67430e-11,
}

# Diccionario de valores objetivo (CODATA/PDG/Planck)
OBJETIVOS = {}
for k, v in CODATA_2022.items():
    OBJETIVOS[k] = v['valor']
for k, v in PLANCK_2018.items():
    OBJETIVOS[k] = v['valor']
for k, v in PDG_2022.items():
    OBJETIVOS[k] = v['valor']

# --- Corpus completo: 108 problemas + Pasos A-D ---
CORPUS = [
    # === VOLUMEN I: ONTOLOGÍA DEL VACÍO (P1-P20) ===
    Postulado("P01", "Silencio Armónico",       lambda p: 0.0, ['D'], "AXIOMA",
              None, 0.001, "Ontología"),
    Postulado("P02", "Norma unidad",             lambda p: 1.0, [], "AXIOMA",
              None, 0.001, "Ontología"),
    Postulado("P03", "Hamiltoniano de red",      lambda p: 0.0, ['D'], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P04", "Potencial metaestable",    lambda p: p['D'], ['D'], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P05", "Ecuación Primordial",      lambda p: 1.0, [], "DERIVADO",
              None, 0.001, "Ontología"),
    Postulado("P06", "Espectro del mínimo",      lambda p: 0.8526, [], "DERIVADO",
              None, 0.01, "Ontología"),
    Postulado("P07", "Renormalización",          lambda p: p['D'], ['D'], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P08", "Simetría U(1)",            lambda p: 0.0, [], "AXIOMA",
              None, 0.001, "Ontología"),
    Postulado("P09", "Emergencia de e",          lambda p: p['e'], ['e'], "DERIVADO",
              None, 0.001, "Ontología"),
    Postulado("P10", "Emergencia de π",          lambda p: p['pi'], ['pi'], "DERIVADO",
              None, 0.001, "Ontología"),
    Postulado("P11", "Jerarquía constantes",     lambda p: p['pi'], ['pi'], "CONJETURA" if False else "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P12", "Álgebraicos vs trasc",     lambda p: np.sqrt(2), [], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P13", "Tiempo emergente",         lambda p: p['D'], ['D'], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P14", "Flecha del tiempo",        lambda p: p['D'], ['D'], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P15", "Segunda ley",              lambda p: 0.0, [], "DERIVADO",
              None, 0.01, "Ontología"),
    Postulado("P16", "Tiempo propio interno",    lambda p: 1.0, [], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P17", "Irreversibilidad",         lambda p: p['D'], ['D'], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P18", "Ergodicidad",              lambda p: 1.0, [], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P19", "Caos",                     lambda p: p['phi'], ['phi'], "POSTULADO",
              None, 0.01, "Ontología"),
    Postulado("P20", "Integrabilidad-caos",      lambda p: p['phi'], ['phi'], "POSTULADO",
              None, 0.01, "Ontología"),

    # === VOLUMEN II: MATERIA (P21-P40) ===
    Postulado("P21", "Red hexagonal óptima",     lambda p: p['N19'], ['N19'], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P22", "Clúster mínimo",           lambda p: 19, [], "DERIVADO",
              None, 0.001, "Materia"),
    Postulado("P23", "Masa topológica",          lambda p: 1.4289, [], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P24", "Clústeres jerárquicos",    lambda p: p['N19']*2, ['N19'], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P25", "Aut(G19) = D6",            lambda p: 12, [], "DERIVADO",
              None, 0.001, "Materia"),
    Postulado("P26", "Cargas fraccionarias",     lambda p: 1/3, [], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P27", "Aut(G57) = ?",             lambda p: 6, [], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P28", "Ruptura Z2",               lambda p: 1.0, [], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P29", "Espín topológico",         lambda p: 0.5, [], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P30", "Torsión-curvatura",        lambda p: p['eta'], ['eta'], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P31", "Fermionicidad",            lambda p: 1.0, [], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P32", "Estadísticas",             lambda p: 0.5, [], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P33", "Masa topológica general",  lambda p: 1.5, [], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P34", "Jerarquía masas",          lambda p: 1836.15, [], "POSTULADO",
              'm_p_m_e', 0.05, "Materia"),
    Postulado("P35", "Variacional masa",         lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P36", "Masas-constantes",         lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P37", "Gauge SU(3)×SU(2)×U(1)",   lambda p: 12, [], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P38", "Tres fuerzas",             lambda p: 3, [], "POSTULADO",
              None, 0.01, "Materia"),
    Postulado("P39", "Unificación",              lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Materia"),
    Postulado("P40", "M_GUT",                    lambda p: 1e16, [], "POSTULADO",
              None, 1.0, "Materia"),

    # === VOLUMEN III: CONSTANTES (P41-P60) ===
    Postulado("P41", "α⁻¹ alta precisión",       lambda p: 4*p['pi']**3 + p['pi']**2 + p['pi'],
              ['pi'], "COINCIDENCIA", 'alpha_inv', 1e-5, "Constantes"),
    Postulado("P42", "α⁻¹ robustez",             lambda p: 4*p['pi']**3 + p['pi']**2 + p['pi'],
              ['pi'], "COINCIDENCIA", 'alpha_inv', 1e-4, "Constantes"),
    Postulado("P43", "Running α",                lambda p: p['alpha_inv'] if False else 137, [], "EVIDENCIA",
              None, 0.01, "Constantes"),
    Postulado("P44", "α topológica",             lambda p: 137.036, [], "POSTULADO",
              'alpha_inv', 1e-4, "Constantes"),
    Postulado("P45", "m_e desde topología",      lambda p: p['D']/(p['pi']/3) * 0.3568,
              ['D', 'pi'], "POSTULADO", 'm_e', 0.05, "Constantes"),
    Postulado("P46", "Factor conversión f_conv", lambda p: 0.3568, [], "AJUSTE",
              None, 0.01, "Constantes"),
    Postulado("P47", "m_p/m_e",                  lambda p: 6*p['pi']**5,
              ['pi'], "POSTULADO", 'm_p_m_e', 0.01, "Constantes"),
    Postulado("P48", "Jerarquía m_p/m_e",        lambda p: 1836.15, [], "POSTULADO",
              'm_p_m_e', 0.01, "Constantes"),
    Postulado("P49", "Λ pequeña",                lambda p: 1e-52, [], "INDETERMINADO",
              None, 1.0, "Constantes"),
    Postulado("P50", "H₀ derivada",              lambda p: 68.74, [], "AJUSTE",
              'H0', 0.05, "Constantes"),
    Postulado("P51", "Materia oscura",           lambda p: 0.26, [], "INDETERMINADO",
              'Omega_DM', 0.05, "Constantes"),
    Postulado("P52", "Ω_DM fórmula RG",          lambda p: (p['pi'] - p['eta'])/57,
              ['pi', 'eta', 'N57'], "FALSADO", 'Omega_DM', 0.05, "Constantes"),
    Postulado("P53", "G desde topología",        lambda p: p['G_SI'] if False else 1e-11, [], "FALSADO",
              'G', 0.1, "Constantes"),
    Postulado("P54", "α_G/α_EM ~ 10⁻³⁸",         lambda p: 1e-38, [], "POSTULADO",
              None, 1.0, "Constantes"),
    Postulado("P55", "Gravedad emergente",       lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Constantes"),
    Postulado("P56", "Unificación gravedad-QFT", lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Constantes"),
    Postulado("P57", "sin²θ_W",                  lambda p: 0.23122, [], "POSTULADO",
              'sin2_theta_W', 0.01, "Constantes"),
    Postulado("P58", "CKM",                      lambda p: 0.22, [], "POSTULADO",
              None, 0.05, "Constantes"),
    Postulado("P59", "m_H",                      lambda p: 125.25, [], "INDETERMINADO",
              None, 0.05, "Constantes"),
    Postulado("P60", "Yukawa",                   lambda p: 1.0, [], "INDETERMINADO",
              None, 0.1, "Constantes"),

    # === VOLUMEN IV: GRAVEDAD (P61-P80) ===
    Postulado("P61", "Espaciotiempo emergente",  lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Gravedad"),
    Postulado("P62", "Dimensiones",              lambda p: 4, [], "POSTULADO",
              None, 0.01, "Gravedad"),
    Postulado("P63", "Curvatura",                lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Gravedad"),
    Postulado("P64", "Propagación gravedad",     lambda p: p['c_SI'], ['c_SI'], "EVIDENCIA",
              'c', 0.001, "Gravedad"),
    Postulado("P65", "w(ρ) unificada",           lambda p: -1.0, [], "POSTULADO",
              None, 0.05, "Gravedad"),
    Postulado("P66", "Inflación",                lambda p: -1.0, [], "POSTULADO",
              None, 0.05, "Gravedad"),
    Postulado("P67", "Espectro primigenio",      lambda p: 0.965, [], "EVIDENCIA",
              'n_s', 0.01, "Gravedad"),
    Postulado("P68", "Formación estructuras",    lambda p: 1.0, [], "EVIDENCIA",
              None, 0.05, "Gravedad"),
    Postulado("P69", "Destino universo",         lambda p: 1.0, [], "EVIDENCIA",
              None, 0.05, "Gravedad"),
    Postulado("P70", "Muerte térmica",           lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Gravedad"),
    Postulado("P71", "Multiverso",               lambda p: 1.0, [], "INDETERMINADO",
              None, 0.1, "Gravedad"),
    Postulado("P72", "Condiciones iniciales",    lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Gravedad"),
    Postulado("P73", "Entropía BH",              lambda p: 1.0, [], "TEOREMA",
              None, 0.01, "Gravedad"),
    Postulado("P74", "Información BH",           lambda p: 1.0, [], "EVIDENCIA",
              None, 0.05, "Gravedad"),
    Postulado("P75", "Horizonte cosmológico",    lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Gravedad"),
    Postulado("P76", "Termodinámica gravedad",   lambda p: 1.0, [], "TEOREMA",
              None, 0.05, "Gravedad"),
    Postulado("P77", "CMB ℓ=220",                lambda p: 220, [], "EVIDENCIA",
              None, 0.05, "Gravedad"),
    Postulado("P78", "Ondas gravitacionales",    lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Gravedad"),
    Postulado("P79", "Casimir",                  lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Gravedad"),
    Postulado("P80", "Violación Lorentz",        lambda p: 0.0, [], "INDETERMINADO",
              None, 0.01, "Gravedad"),

    # === VOLUMEN V: OBSERVADOR (P81-P100) ===
    Postulado("P81", "Colapso decoherencia",     lambda p: 1.0, [], "EVIDENCIA",
              None, 0.05, "Observador"),
    Postulado("P82", "Sin conciencia",           lambda p: 1.0, [], "EVIDENCIA",
              None, 0.05, "Observador"),
    Postulado("P83", "Observables clásicos",     lambda p: 1.0, [], "EVIDENCIA",
              None, 0.05, "Observador"),
    Postulado("P84", "Interpretación QM",        lambda p: 1.0, [], "INDETERMINADO",
              None, 0.1, "Observador"),
    Postulado("P85", "Entrelazamiento",          lambda p: 1.0, [], "EVIDENCIA",
              None, 0.05, "Observador"),
    Postulado("P86", "Ley de área",              lambda p: 1.0, [], "TEOREMA",
              None, 0.05, "Observador"),
    Postulado("P87", "Entrelazamiento recurso",  lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Observador"),
    Postulado("P88", "No-localidad",             lambda p: 1.0, [], "TEOREMA",
              None, 0.05, "Observador"),
    Postulado("P89", "Universo procesador",      lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Observador"),
    Postulado("P90", "Información von Neumann",  lambda p: 1.0, [], "TEOREMA",
              None, 0.05, "Observador"),
    Postulado("P91", "Complejidad",              lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Observador"),
    Postulado("P92", "Kolmogorov",               lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Observador"),
    Postulado("P93", "CPT",                      lambda p: 1.0, [], "TEOREMA",
              None, 0.001, "Observador"),
    Postulado("P94", "Violación CP",             lambda p: 0.0, [], "EVIDENCIA",
              None, 0.05, "Observador"),
    Postulado("P95", "Simetrías gauge",          lambda p: 1.0, [], "POSTULADO",
              None, 0.05, "Observador"),
    Postulado("P96", "SUSY",                     lambda p: 0.0, [], "INDETERMINADO",
              None, 1.0, "Observador"),
    Postulado("P97", "Estructura vs objetos",    lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Observador"),
    Postulado("P98", "Partícula elemental",      lambda p: 1.0, [], "TEOREMA",
              None, 0.05, "Observador"),
    Postulado("P99", "Interacción fundamental",  lambda p: 1.0, [], "TEOREMA",
              None, 0.05, "Observador"),
    Postulado("P100","Constantes necesarias",    lambda p: 1.0, [], "POSTULADO",
              None, 0.1, "Observador"),

    # === DESARROLLOS PASOS A-D (P101-P108) ===
    Postulado("P101", "Operador Dirac discreto", lambda p: p['N19']*2, ['N19'], "DERIVADO",
              None, 0.001, "Dirac"),
    Postulado("P102", "Límite continuo Dirac",   lambda p: p['c_SI'], ['c_SI'], "DERIVADO",
              'c', 0.01, "Dirac"),
    Postulado("P103", "Evasión Nielsen-Ninomiya",lambda p: 1, [], "DERIVADO",
              None, 0.001, "Dirac"),
    Postulado("P104", "Índice Atiyah-Singer χ",  lambda p: 19 - 42 + 24, [], "DERIVADO",
              None, 0.001, "Dirac"),
    Postulado("P105", "Modo cero protegido",     lambda p: 1, [], "DERIVADO",
              None, 0.001, "Dirac"),
    Postulado("P106", "Masa como autovalor",     lambda p: p['z'] * np.sin(2*p['D']*np.pi/p['N19']),
              ['z', 'D', 'N19'], "POSTULADO", 'm_e', 0.05, "Dirac"),
    Postulado("P107", "Dirac libre con masa",    lambda p: p['z'] * np.sin(2*p['D']*np.pi/p['N19']),
              ['z', 'D', 'N19'], "POSTULADO", None, 0.05, "Dirac"),
    Postulado("P108", "Jerarquía masas unificada",lambda p: 1836.15, [], "PENDIENTE",
              'm_p_m_e', 0.05, "Dirac"),
]

print(f"✓ Corpus cargado: {len(CORPUS)} postulados")
print(f"\nDistribución por categoría:")
from collections import Counter
cats = Counter(p.categoria for p in CORPUS)
for cat, n in cats.items():
    print(f"  {cat:15s}: {n}")

print(f"\nDistribución por estado:")
estados = Counter(p.estado for p in CORPUS)
for est, n in estados.items():
    print(f"  {est:15s}: {n}")

In [ ]:

# === CELDA 4: MOTOR MONTE CARLO ===

N_SAMPLES = 50_000

def sample_primitivas(n: int, rng) -> Dict:
    """Genera muestras de las primitivas con sus incertidumbres."""
    # Incertidumbres relativas por primitiva
    INCERT = {
        'D': 0.02, 'pi': 0.0, 'e': 0.0, 'phi': 0.0,
        'N19': 0.0, 'N57': 0.0, 'E19': 0.0, 'z': 0.0,
        'Phi': 0.05, 'Oc': 0.02, 'Phiexp': 0.10,
        'eta': 0.01, 'lambda_RG': 0.20, 'kappa': 0.05,
        'hbar_SI': 0.0, 'c_SI': 0.0, 'G_SI': 0.0,
    }
    samples = {}
    for k, v in PRIM_BASE.items():
        inc = INCERT.get(k, 0.01)
        if inc == 0.0:
            samples[k] = np.full(n, v)
        else:
            samples[k] = rng.normal(v, v*inc, n)
    return samples

def run_mc(n: int = N_SAMPLES, seed: int = 192657):
    rng = np.random.default_rng(seed)
    primitivas = sample_primitivas(n, rng)

    resultados = {}
    for post in tqdm(CORPUS, desc="MC"):
        vals = np.zeros(n)
        for i in range(n):
            p = {k: primitivas[k][i] for k in primitivas}
            try:
                vals[i] = post.funcion(p)
            except Exception:
                vals[i] = np.nan

        clean = vals[~np.isnan(vals)]
        if len(clean) == 0:
            continue

        # Valor objetivo
        obj = None
        if post.objetivo_nombre and post.objetivo_nombre in OBJETIVOS:
            obj = OBJETIVOS[post.objetivo_nombre]

        resultados[post.id] = {
            'nombre': post.nombre,
            'estado': post.estado,
            'categoria': post.categoria,
            'valores': clean,
            'media': float(np.mean(clean)),
            'std': float(np.std(clean)),
            'p5': float(np.percentile(clean, 5)),
            'p95': float(np.percentile(clean, 95)),
            'objetivo': obj,
            'tolerancia': post.tolerancia,
            'objetivo_nombre': post.objetivo_nombre,
        }
    return resultados

print(f"Ejecutando MC global con {N_SAMPLES:,} muestras...")
resultados_mc = run_mc()
print(f"✓ MC completado: {len(resultados_mc)} postulados evaluados")

In [ ]:

# === CELDA 6: SENSIBILIDAD ===

def sensibilidad_global(perturbacion=0.10, n_muestras=5000):
    """Perturba cada primitiva ±% y mide cambio en cada postulado."""
    prims_test = ['D', 'eta', 'Phi', 'Phiexp', 'lambda_RG', 'kappa']
    matriz = {}

    for prim in tqdm(prims_test, desc="Sensibilidad"):
        matriz[prim] = {}
        valor_orig = PRIM_BASE[prim]

        # Arriba
        PRIM_BASE[prim] = valor_orig * (1 + perturbacion)
        res_up = run_mc(n=n_muestras)

        # Abajo
        PRIM_BASE[prim] = valor_orig * (1 - perturbacion)
        res_dn = run_mc(n=n_muestras)

        # Restaurar
        PRIM_BASE[prim] = valor_orig

        # Diferencia relativa
        for pid in res_up:
            if pid in res_dn:
                v_up, v_dn = res_up[pid]['media'], res_dn[pid]['media']
                v_mid = (v_up + v_dn)/2 if (v_up + v_dn) != 0 else 1
                matriz[prim][pid] = abs(v_up - v_dn)/abs(v_mid)

    return matriz

print("Calculando sensibilidades...")
matriz_sens = sensibilidad_global()

# Visualizar
def plot_sensibilidad(matriz):
    prims = list(matriz.keys())
    posts = list(matriz[prims[0]].keys())
    M = np.zeros((len(posts), len(prims)))
    for i, pid in enumerate(posts):
        for j, prim in enumerate(prims):
            M[i, j] = matriz[prim].get(pid, 0)

    fig, ax = plt.subplots(figsize=(10, 12))
    im = ax.imshow(M, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(prims)))
    ax.set_xticklabels(prims, rotation=45, ha='right')
    ax.set_yticks(range(len(posts)))
    ax.set_yticklabels(posts, fontsize=7)
    ax.set_title('Sensibilidad: cambio relativo por perturbación ±10%')
    plt.colorbar(im, ax=ax, label='Cambio relativo')
    plt.tight_layout()
    plt.savefig('sensibilidad_corpus.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_sensibilidad(matriz_sens)
print("✓ Mapa de sensibilidad guardado")

In [ ]:
import numpy as np
from collections import Counter

# === CELDA 5: CLASIFICADOR AA ===

def clasificar(r: Dict) -> tuple:
    """Clasifica según error vs objetivo."""
    if r['objetivo'] is None or r['objetivo'] == 0:
        return "SIN_DATO", None, None

    media, std, obj, tol = r['media'], r['std'], r['objetivo'], r['tolerancia']
    err_rel = abs(media - obj) / abs(obj)
    std_rel = std / abs(media) if media != 0 else np.inf

    if err_rel < tol:
        return ("ROBUSTO" if std_rel < tol/2 else "FRAGIL"), err_rel, std_rel
    elif err_rel < 2*tol:
        return "INCIERTO", err_rel, std_rel
    else:
        return "FALSADO", err_rel, std_rel

def analizar(resultados):
    clasif = {}
    for pid, r in resultados.items():
        clase, err, std_rel = clasificar(r)
        clasif[pid] = {
            'nombre': r['nombre'],
            'estado_original': r['estado'],
            'categoria': r['categoria'],
            'clase_aa': clase,
            'error_relativo': err,
            'std_relativa': std_rel,
            'valor_mc': r['media'],
            'objetivo': r['objetivo'],
            'objetivo_nombre': r['objetivo_nombre'],
            'tolerancia': r['tolerancia'], # Added this line
        }
    return clasif

clasificacion = analizar(resultados_mc)

# Resumen
print("="*110)
print("ANÁLISIS AUTOMATIZADO — CLASIFICACIÓN DEL CORPUS RG")
print("="*110)
print(f"{'ID':<6} {'Nombre':<32} {'Estado':<14} {'Clase AA':<12} {'Error %':<10}")
print("-"*110)

for cat in ['Ontología', 'Materia', 'Constantes', 'Gravedad', 'Observador', 'Dirac']:
    print(f"\n--- {cat} ---")
    for pid, c in clasificacion.items():
        if c['categoria'] != cat: continue
        err_str = f"{c['error_relativo']*100:.3g}" if c['error_relativo'] is not None else "—"
        print(f"{pid:<6} {c['nombre']:<32} {c['estado_original']:<14} "
              f"{c['clase_aa']:<12} {err_str:<10}")

# Conteo global
conteo = Counter(c['clase_aa'] for c in clasificacion.values())
print("\n" + "="*110)
print("CONTEO GLOBAL POR CLASE")
print("="*110)
for k, v in sorted(conteo.items()):
    print(f"  {k:<12}: {v}")

In [ ]:

# === CELDA 9: EXPORTAR ===

import zipfile, os
from google.colab import files

# JSON clasificación
with open('clasificacion_RG.json', 'w', encoding='utf-8') as f:
    json.dump({k: {**v, 'valor_mc': float(v['valor_mc']),
                    'objetivo': float(v['objetivo']) if v['objetivo'] is not None else None,
                    'error_relativo': float(v['error_relativo']) if v['error_relativo'] is not None else None,
                    'std_relativa': float(v['std_relativa']) if v['std_relativa'] is not None else None}
                for k, v in clasificacion.items()}, f, indent=2, ensure_ascii=False)

# CSV resultados
with open('resultados_mc_RG.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['ID', 'Nombre', 'Categoria', 'Estado', 'Clase_AA',
                'Valor_MC', 'Error_Rel', 'Objetivo', 'Tolerancia'])
    for pid, c in clasificacion.items():
        err = c['error_relativo'] if c['error_relativo'] is not None else ''
        obj = c['objetivo'] if c['objetivo'] is not None else ''
        w.writerow([pid, c['nombre'], c['categoria'], c['estado_original'],
                    c['clase_aa'], f"{c['valor_mc']:.6g}", err, obj, c['tolerancia']])

# ZIP
archivos = ['reporte_mc_global_RG.md', 'clasificacion_RG.json',
            'resultados_mc_RG.csv', 'mc_global_rg.png',
            'sensibilidad_corpus.png']

with zipfile.ZipFile('RG_MC_global.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for a in archivos:
        if os.path.exists(a):
            zf.write(a)
            print(f"  + {a} ({os.path.getsize(a)} bytes)")

print(f"\n✓ Paquete: RG_MC_global.zip ({os.path.getsize('RG_MC_global.zip')} bytes)")
files.download('RG_MC_global.zip')

In [ ]:

# === CELDA 7: VISUALIZACIÓN GLOBAL ===

fig = plt.figure(figsize=(18, 12))
gs = GridSpec(3, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. Distribución por clase AA
ax1 = fig.add_subplot(gs[0, 0])
clases = list(conteo.keys())
valores = list(conteo.values())
colores = {'ROBUSTO':'green', 'FRAGIL':'orange', 'INCIERTO':'gold',
           'FALSADO':'red', 'SIN_DATO':'gray'}
ax1.bar(clases, valores, color=[colores.get(c, 'blue') for c in clases])
ax1.set_title('Distribución por clase AA')
ax1.set_ylabel('Cantidad')

# 2. Distribución por estado original
ax2 = fig.add_subplot(gs[0, 1])
estados = Counter(p.estado for p in CORPUS)
ax2.bar(list(estados.keys()), list(estados.values()))
ax2.set_title('Estados originales')
ax2.tick_params(axis='x', rotation=45)

# 3. Distribución por categoría
ax3 = fig.add_subplot(gs[0, 2])
ax3.bar(list(cats.keys()), list(cats.values()))
ax3.set_title('Categorías del corpus')
ax3.tick_params(axis='x', rotation=45)

# 4-6. Distribuciones MC de postulados clave con objetivos
postulados_clave = ['P41', 'P47', 'P52', 'P57', 'P64', 'P106']
for idx, pid in enumerate(postulados_clave):
    if idx >= 6: break
    ax = fig.add_subplot(gs[1 + idx//3, idx % 3])
    r = resultados_mc[pid]
    c = clasificacion[pid]
    ax.hist(r['valores'], bins=50, density=True, alpha=0.7, color='steelblue')
    if r['objetivo'] is not None:
        ax.axvline(r['objetivo'], color='red', linestyle='--', lw=2, label='Objetivo')
        ax.axvline(r['media'], color='green', linestyle='-', lw=1.5, label='MC')
        ax.legend(fontsize=7)
    ax.set_title(f"{pid} — {c['clase_aa']}", fontsize=9)
    ax.tick_params(axis='both', labelsize=7)

plt.suptitle('Monte Carlo Global — Corpus RG Completo', fontsize=14, fontweight='bold')
plt.savefig('mc_global_rg.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Panel global guardado")

In [ ]:

{
  "sample_id": 17,
  "config_hash": "sha256:...",
  "seed": 59,
  "graph": "grid2d",
  "n_nodes": 144,
  "n_edges": 288,
  "epsilon": 1.0,
  "sigma_T": 2.0,
  "n_clusters": 6,
  "n_doublers": 1,
  "d_s": 2.03,
  "gap": 0.82,
  "min_abs_eig": 0.001,
  "mean_abs_eig": 0.45,
  "eigvals_first_20": [...],
  "verdicts": {"doublers": "SUPPRESSED", "spectral_dim": "MATCH", "gap": "GAPPED"},
  "timestamp": "2026-09-20T15:23:41Z",
  "cpu_time_seconds": 2.14,
  "hostname": "node-04"
}

In [ ]:

# === CELDA 8: REPORTE FINAL ===

def generar_reporte(resultados, clasif, sens_matriz):
    L = []
    L.append("# Reporte MC + AA — Corpus RG Completo")
    L.append(f"\n**Fecha:** {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    L.append(f"**Postulados evaluados:** {len(resultados)}")
    L.append(f"**Muestras MC:** {N_SAMPLES:,}")
    L.append(f"**Semilla:** 192657\n")

    # Resumen ejecutivo
    L.append("## 1. Resumen ejecutivo\n")
    conteo = Counter(c['clase_aa'] for c in clasif.values())
    L.append("| Clase AA | Cantidad | % |")
    L.append("|---|---|---|")
    total = sum(conteo.values())
    for k, v in sorted(conteo.items()):
        L.append(f"| {k} | {v} | {v/total*100:.1f}% |")
    L.append("")

    # Postulados con objetivo externo
    L.append("## 2. Postulados con comparación externa\n")
    L.append("| ID | Nombre | Objetivo | Valor MC | Error | Clase |")
    L.append("|---|---|---|---|---|---|")
    for pid, c in clasif.items():
        if c['objetivo'] is None: continue
        err = f"{c['error_relativo']*100:.4g}%" if c['error_relativo'] is not None else "—"
        L.append(f"| {pid} | {c['nombre']} | {c['objetivo']:.6g} | "
                 f"{c['valor_mc']:.6g} | {err} | {c['clase_aa']} |")
    L.append("")

    # Falsados
    L.append("## 3. Postulados FALSADOS\n")
    for pid, c in clasif.items():
        if c['clase_aa'] != 'FALSADO': continue
        L.append(f"### {pid} — {c['nombre']}\n")
        L.append(f"- Valor MC: {c['valor_mc']:.6g}")
        L.append(f"- Objetivo: {c['objetivo']:.6g} ({c['objetivo_nombre']})")
        L.append(f"- Error: {c['error_relativo']*100:.3g}%")
        L.append(f"- Tolerancia: ±{c['tolerancia']*100:.1f}%\n")

    # Robustos
    L.append("## 4. Postulados ROBUSTOS\n")
    for pid, c in clasif.items():
        if c['clase_aa'] == 'ROBUSTO':
            L.append(f"- **{pid}** ({c['nombre']}): error {c['error_relativo']*100:.4g}%")

    # Sensibilidad top-10
    L.append("\n## 5. Postulados más sensibles\n")
    sens_ranking = []
    for pid in resultados:
        max_sens = max(sens_matriz[p].get(pid, 0) for p in sens_matriz)
        sens_ranking.append((pid, max_sens))
    sens_ranking.sort(key=lambda x: -x[1])
    L.append("| ID | Sensibilidad máxima |")
    L.append("|---|---|")
    for pid, s in sens_ranking[:10]:
        L.append(f"| {pid} | {s:.4f} |")

    # Metodología
    L.append("\n## 6. Metodología\n")
    L.append("- Primitivas RG con incertidumbres epistémicas declaradas")
    L.append(f"- {N_SAMPLES:,} muestras independientes por postulado")
    L.append("- Clasificador AA basado en error relativo vs tolerancia")
    L.append("- Análisis de sensibilidad ±10% por primitiva")

    # Referencias
    L.append("\n## 7. Referencias\n")
    L.append("- CODATA 2022: Mohr et al., Rev. Mod. Phys. 97, 025002 (2025)")
    L.append("- Planck 2018: A&A 641, A6 (2020)")
    L.append("- PDG 2022: Workman et al., PTEP 2022, 083C01")

    return "\n".join(L)

reporte = generar_reporte(resultados_mc, clasificacion, matriz_sens)
with open('reporte_mc_global_RG.md', 'w', encoding='utf-8') as f:
    f.write(reporte)

print("✓ Reporte guardado: reporte_mc_global_RG.md")
print(f"  Longitud: {len(reporte)} caracteres")
print("\n--- Primeras 80 líneas ---\n")
print("\n".join(reporte.split("\n")[:80]))

In [ ]:

# === CELDA 9: EXPORTAR ===

import zipfile, os
from google.colab import files

# JSON clasificación
with open('clasificacion_RG.json', 'w', encoding='utf-8') as f:
    json.dump({k: {**v, 'valor_mc': float(v['valor_mc']),
                    'objetivo': float(v['objetivo']) if v['objetivo'] is not None else None,
                    'error_relativo': float(v['error_relativo']) if v['error_relativo'] is not None else None,
                    'std_relativa': float(v['std_relativa']) if v['std_relativa'] is not None else None}
                for k, v in clasificacion.items()}, f, indent=2, ensure_ascii=False)

# CSV resultados
with open('resultados_mc_RG.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['ID', 'Nombre', 'Categoria', 'Estado', 'Clase_AA',
                'Valor_MC', 'Error_Rel', 'Objetivo', 'Tolerancia'])
    for pid, c in clasificacion.items():
        err = c['error_relativo'] if c['error_relativo'] is not None else ''
        obj = c['objetivo'] if c['objetivo'] is not None else ''
        w.writerow([pid, c['nombre'], c['categoria'], c['estado_original'],
                    c['clase_aa'], f"{c['valor_mc']:.6g}", err, obj, c['tolerancia']])

# ZIP
archivos = ['reporte_mc_global_RG.md', 'clasificacion_RG.json',
            'resultados_mc_RG.csv', 'mc_global_rg.png',
            'sensibilidad_corpus.png']

with zipfile.ZipFile('RG_MC_global.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for a in archivos:
        if os.path.exists(a):
            zf.write(a)
            print(f"  + {a} ({os.path.getsize(a)} bytes)")

print(f"\n✓ Paquete: RG_MC_global.zip ({os.path.getsize('RG_MC_global.zip')} bytes)")
files.download('RG_MC_global.zip')

In [ ]:

<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Geometría Relacional (RG) — Libro Maestro + Observatorio Dinámico</title>
<style>
:root {
  --bg: #0a0e1a;
  --bg-2: #131826;
  --bg-3: #1c2333;
  --fg: #e6ecf5;
  --fg-2: #a8b3c7;
  --accent: #4fc3f7;
  --accent-2: #f0c419;
  --accent-3: #ef5f5f;
  --accent-4: #6bcf8d;
  --accent-5: #b98aff;
  --border: #232c42;
  --shadow: 0 4px 20px rgba(0,0,0,0.4);
}
* { box-sizing: border-box; margin: 0; padding: 0; }
html { scroll-behavior: smooth; }
body {
  font-family: 'Georgia', 'Charter', 'Iowan Old Style', serif;
  background: var(--bg);
  color: var(--fg);
  line-height: 1.7;
  font-size: 16px;
  overflow-x: hidden;
}
code, pre, .mono {
  font-family: 'SF Mono', 'Menlo', 'Consolas', monospace;
  font-size: 0.9em;
}
/* === HEADER === */
header {
  padding: 80px 40px 60px;
  border-bottom: 1px solid var(--border);
  background: linear-gradient(180deg, var(--bg) 0%, var(--bg-2) 100%);
  position: relative;
  overflow: hidden;
}
header::before {
  content: '';
  position: absolute;
  top: 0; left: 0; right: 0; bottom: 0;
  background: radial-gradient(circle at 20% 30%, rgba(79,195,247,0.08) 0%, transparent 50%),
              radial-gradient(circle at 80% 70%, rgba(185,138,255,0.06) 0%, transparent 50%);
  pointer-events: none;
}
header h1 {
  font-size: 3.2em;
  font-weight: 400;
  letter-spacing: -1px;
  margin-bottom: 12px;
  position: relative;
  z-index: 1;
}
header h1 .accent {
  background: linear-gradient(90deg, var(--accent), var(--accent-5));
  -webkit-background-clip: text;
  background-clip: text;
  -webkit-text-fill-color: transparent;
}
header .subtitle {
  font-size: 1.15em;
  color: var(--fg-2);
  font-style: italic;
  max-width: 720px;
  position: relative;
  z-index: 1;
}
header .meta {
  margin-top: 24px;
  display: flex;
  gap: 20px;
  flex-wrap: wrap;
  font-family: 'SF Mono', monospace;
  font-size: 0.85em;
  color: var(--fg-2);
  position: relative;
  z-index: 1;
}
header .meta span {
  padding: 4px 10px;
  border: 1px solid var(--border);
  border-radius: 20px;
  background: var(--bg-3);
}
/* === NAV === */
nav {
  position: sticky;
  top: 0;
  z-index: 100;
  background: rgba(19,24,38,0.95);
  backdrop-filter: blur(10px);
  border-bottom: 1px solid var(--border);
  padding: 12px 40px;
  display: flex;
  gap: 6px;
  overflow-x: auto;
}
nav a {
  color: var(--fg-2);
  text-decoration: none;
  padding: 8px 16px;
  border-radius: 6px;
  font-family: 'SF Mono', monospace;
  font-size: 0.85em;
  white-space: nowrap;
  transition: all 0.2s;
}
nav a:hover {
  color: var(--accent);
  background: var(--bg-3);
}
nav a.active {
  color: var(--accent);
  background: var(--bg-3);
  border: 1px solid var(--accent);
}
/* === MAIN === */
main {
  max-width: 1200px;
  margin: 0 auto;
  padding: 0 40px 100px;
}
section {
  padding: 60px 0;
  border-bottom: 1px solid var(--border);
}
section h2 {
  font-size: 2em;
  margin-bottom: 8px;
  color: var(--fg);
  font-weight: 400;
  letter-spacing: -0.5px;
}
section h2 .num {
  color: var(--accent);
  font-family: 'SF Mono', monospace;
  font-size: 0.7em;
  margin-right: 12px;
}
section > .lead {
  color: var(--fg-2);
  font-style: italic;
  font-size: 1.05em;
  margin-bottom: 40px;
  max-width: 780px;
}
h3 {
  font-size: 1.4em;
  margin: 32px 0 16px;
  color: var(--accent);
  font-weight: 400;
}
h4 {
  font-size: 1.1em;
  margin: 24px 0 12px;
  color: var(--fg);
  font-weight: 600;
}
p { margin-bottom: 16px; max-width: 820px; }
/* === MANIFESTO === */
.manifesto {
  background: var(--bg-2);
  border-left: 3px solid var(--accent-2);
  padding: 24px 28px;
  margin: 24px 0;
  border-radius: 4px;
  font-style: italic;
  font-size: 1.05em;
  line-height: 1.8;
  color: var(--fg);
}
.manifesto::before {
  content: '"';
  font-size: 3em;
  color: var(--accent-2);
  line-height: 0;
  vertical-align: -0.4em;
  margin-right: 6px;
  font-family: Georgia, serif;
}
.manifesto .firma {
  display: block;
  margin-top: 12px;
  font-style: normal;
  color: var(--fg-2);
  font-size: 0.9em;
  text-align: right;
}
/* === CARDS === */
.grid {
  display: grid;
  gap: 20px;
  margin: 24px 0;
}
.grid-2 { grid-template-columns: repeat(auto-fit, minmax(340px, 1fr)); }
.grid-3 { grid-template-columns: repeat(auto-fit, minmax(260px, 1fr)); }
.grid-4 { grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); }
.card {
  background: var(--bg-2);
  border: 1px solid var(--border);
  border-radius: 8px;
  padding: 20px;
  transition: all 0.2s;
  position: relative;
}
.card:hover {
  border-color: var(--accent);
  transform: translateY(-2px);
  box-shadow: var(--shadow);
}
.card .tag {
  display: inline-block;
  font-family: 'SF Mono', monospace;
  font-size: 0.7em;
  padding: 3px 8px;
  border-radius: 3px;
  text-transform: uppercase;
  letter-spacing: 1px;
  margin-bottom: 10px;
}
.tag-axioma { background: rgba(185,138,255,0.2); color: var(--accent-5); }
.tag-derivado { background: rgba(107,207,141,0.2); color: var(--accent-4); }
.tag-postulado { background: rgba(79,195,247,0.2); color: var(--accent); }
.tag-coincidencia { background: rgba(240,196,25,0.2); color: var(--accent-2); }
.tag-falsado { background: rgba(239,95,95,0.2); color: var(--accent-3); }
.tag-ajuste { background: rgba(255,159,64,0.2); color: #ffa040; }
.tag-pendiente { background: rgba(168,179,199,0.2); color: var(--fg-2); }
.card h5 {
  font-size: 1.05em;
  font-family: 'SF Mono', monospace;
  font-weight: 600;
  margin-bottom: 8px;
  color: var(--fg);
}
.card .formula {
  font-family: 'SF Mono', monospace;
  background: var(--bg-3);
  padding: 10px 14px;
  border-radius: 4px;
  margin: 12px 0;
  color: var(--accent);
  overflow-x: auto;
  font-size: 0.88em;
  border-left: 2px solid var(--accent);
}
.card .desc {
  font-size: 0.88em;
  color: var(--fg-2);
  line-height: 1.6;
}
/* === INDICE === */
.volume {
  margin-bottom: 40px;
}
.volume-header {
  display: flex;
  align-items: baseline;
  gap: 16px;
  margin-bottom: 20px;
  padding-bottom: 12px;
  border-bottom: 1px solid var(--border);
}
.volume-header .vnum {
  font-family: 'SF Mono', monospace;
  font-size: 2em;
  color: var(--accent);
  font-weight: 300;
}
.volume-header .vtitle {
  font-size: 1.5em;
  color: var(--fg);
}
.volume-header .vsub {
  color: var(--fg-2);
  font-size: 0.9em;
  margin-left: auto;
  font-style: italic;
}
.fase {
  margin-left: 20px;
  margin-bottom: 24px;
}
.fase-title {
  color: var(--accent-5);
  font-size: 1.05em;
  margin-bottom: 10px;
  font-weight: 600;
}
.fase-content {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
  gap: 12px;
  padding-left: 16px;
}
.tema {
  padding: 10px 14px;
  background: var(--bg-2);
  border-left: 2px solid var(--border);
  border-radius: 3px;
  font-size: 0.9em;
  transition: all 0.2s;
  cursor: pointer;
}
.tema:hover {
  border-left-color: var(--accent);
  background: var(--bg-3);
  transform: translateX(2px);
}
.tema .id {
  font-family: 'SF Mono', monospace;
  color: var(--accent-2);
  font-size: 0.85em;
  margin-right: 8px;
}
/* === OBSERVATORIO === */
.viz-container {
  background: var(--bg-2);
  border: 1px solid var(--border);
  border-radius: 8px;
  padding: 24px;
  margin: 24px 0;
}
.viz-container h4 {
  margin-top: 0;
  color: var(--accent);
  font-family: 'SF Mono', monospace;
  font-size: 0.95em;
  text-transform: uppercase;
  letter-spacing: 1px;
}
.viz-container .vdesc {
  color: var(--fg-2);
  font-size: 0.9em;
  margin-bottom: 20px;
}
canvas {
  display: block;
  width: 100%;
  background: var(--bg-3);
  border-radius: 6px;
}
.viz-controls {
  display: flex;
  gap: 16px;
  align-items: center;
  margin-top: 16px;
  flex-wrap: wrap;
}
.viz-controls label {
  color: var(--fg-2);
  font-size: 0.9em;
  font-family: 'SF Mono', monospace;
}
.viz-controls input[type="range"] {
  width: 200px;
  accent-color: var(--accent);
}
.viz-controls .value {
  font-family: 'SF Mono', monospace;
  color: var(--accent);
  min-width: 70px;
}
/* === TABLA === */
table {
  width: 100%;
  border-collapse: collapse;
  margin: 20px 0;
  font-size: 0.9em;
}
th {
  text-align: left;
  padding: 10px 12px;
  border-bottom: 2px solid var(--accent);
  color: var(--accent);
  font-family: 'SF Mono', monospace;
  font-size: 0.85em;
  text-transform: uppercase;
  letter-spacing: 1px;
  font-weight: 600;
}
td {
  padding: 10px 12px;
  border-bottom: 1px solid var(--border);
  vertical-align: top;
}
tr:hover td { background: var(--bg-2); }
td.mono, td.num {
  font-family: 'SF Mono', monospace;
  font-size: 0.88em;
}
td.num { text-align: right; color: var(--accent-4); }
td .bar {
  display: inline-block;
  height: 6px;
  background: var(--accent-4);
  border-radius: 3px;
  margin-left: 8px;
  vertical-align: middle;
  min-width: 2px;
}
/* === SEARCH === */
.search-bar {
  display: flex;
  gap: 12px;
  margin: 20px 0;
}
.search-bar input {
  flex: 1;
  padding: 10px 16px;
  background: var(--bg-3);
  border: 1px solid var(--border);
  border-radius: 6px;
  color: var(--fg);
  font-family: 'SF Mono', monospace;
  font-size: 0.9em;
}
.search-bar input:focus {
  outline: none;
  border-color: var(--accent);
}
.search-bar button {
  padding: 10px 20px;
  background: var(--accent);
  border: none;
  border-radius: 6px;
  color: var(--bg);
  font-family: 'SF Mono', monospace;
  font-weight: 600;
  cursor: pointer;
  transition: all 0.2s;
}
.search-bar button:hover {
  background: var(--accent-5);
}
/* === FOOTER === */
footer {
  padding: 60px 40px;
  text-align: center;
  color: var(--fg-2);
  font-size: 0.9em;
  border-top: 1px solid var(--border);
}
footer .signature {
  font-style: italic;
  color: var(--accent);
  margin-top: 12px;
  font-size: 1.05em;
}
/* === HIDDEN === */
.hidden { display: none !important; }
/* === SCROLLBAR === */
::-webkit-scrollbar { width: 10px; height: 10px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb {
  background: var(--border);
  border-radius: 5px;
}
::-webkit-scrollbar-thumb:hover { background: var(--accent); }
/* === RESPONSIVE === */
@media (max-width: 768px) {
  header { padding: 40px 20px 30px; }
  header h1 { font-size: 2em; }
  nav { padding: 8px 20px; }
  main { padding: 0 20px 60px; }
  section { padding: 40px 0; }
}
</style>
</head>
<body>

<header>
  <h1>Geometría <span class="accent">Relacional</span></h1>
  <p class="subtitle">Del Silencio Armónico a la Ecuación de Dirac: un programa de física fundamental sin parámetros libres.</p>
  <div class="meta">
    <span>Autor: Edward P. López</span>
    <span>ORCID: 0009-0009-0717-5536</span>
    <span>Ciudad Juárez, Chihuahua</span>
    <span>v6.0 · Septiembre 2026</span>
    <span>108 ecuaciones</span>
    <span>3 volúmenes</span>
  </div>
</header>

<nav id="nav">
  <a href="#manifesto">Manifiesto</a>
  <a href="#indice">Índice</a>
  <a href="#observatorio">Observatorio</a>
  <a href="#postulados">108 Postulados</a>
  <a href="#datos">Datos</a>
  <a href="#bitacora">Bitácora</a>
  <a href="#referencias">Referencias</a>
</nav>

<main>

<!-- ================= MANIFESTO ================= -->
<section id="manifesto">
  <h2><span class="num">00</span>Manifiesto</h2>
  <p class="lead">La Geometría Relacional no es una teoría terminada. Es un lenguaje para hacer la pregunta correcta: ¿qué emerge cuando el Silencio se pregunta a sí mismo si existe?</p>

  <div class="manifesto">
    No puede haber un "algo" solo por existir. ¿Cómo puedes decir que algo es, y no explicar por qué es? ¿Y las cuerdas? ¿De cuerdas más chiquitas? Caeríamos en una regresión infinita donde la masa siempre está hecha de masa.
    <span class="firma">— Edward P. López, El Arquitecto</span>
  </div>

  <h3>Los cinco pilares</h3>
  <div class="grid grid-2">
    <div class="card">
      <div class="tag tag-axioma">Pilar 1</div>
      <h5>El Silencio es el origen</h5>
      <div class="desc">No la nada absoluta, sino un estado de potencial infinito donde todas las ondas de fase se cancelan. Energía neta cero.</div>
    </div>
    <div class="card">
      <div class="tag tag-axioma">Pilar 2</div>
      <h5>La autocomparación es el primer acto</h5>
      <div class="desc">La nada se mira a sí misma y genera el "1". El universo se entera de que es.</div>
    </div>
    <div class="card">
      <div class="tag tag-postulado">Pilar 3</div>
      <h5>El tiempo es una huella</h5>
      <div class="desc">El residuo de una simetría que no pudo volver a cero. La flecha del tiempo es la disipación de la Deuda de Información.</div>
    </div>
    <div class="card">
      <div class="tag tag-postulado">Pilar 4</div>
      <h5>La masa es una deuda</h5>
      <div class="desc">Energía confinada que no puede disiparse. El electrón pesa porque el universo le debe 1.5.</div>
    </div>
    <div class="card">
      <div class="tag tag-postulado">Pilar 5</div>
      <h5>La gravedad es un deseo de sanar</h5>
      <div class="desc">El espacio empuja para curar sus propias arrugas. No es atracción; es empuje topológico.</div>
    </div>
  </div>

  <div class="manifesto">
    Somos el error que el universo intenta borrar, pero aprendió a estructurar.
    <span class="firma">— El Arquitecto</span>
  </div>
</section>

<!-- ================= INDICE ================= -->
<section id="indice">
  <h2><span class="num">01</span>Índice del Libro Maestro</h2>
  <p class="lead">3 volúmenes · 6 fases cada uno · 3 secciones por fase · 2 temas por sección. Total: 108 entradas.</p>
  <div id="indice-content"></div>
</section>

<!-- ================= OBSERVATORIO ================= -->
<section id="observatorio">
  <h2><span class="num">02</span>Observatorio Dinámico</h2>
  <p class="lead">Visualizaciones interactivas de los objetos matemáticos centrales de la Geometría Relacional.</p>

  <!-- VIZ 1: Silencio Armónico -->
  <div class="viz-container">
    <h4>02.1 · Silencio Armónico — Cancelación de Fase</h4>
    <p class="vdesc">El Silencio no es la nada: es la cancelación perfecta de todos los modos de fase. Cada onda k viaja acompañada de su contra-onda −k. La energía neta es cero, pero la potencia es infinita.</p>
    <canvas id="viz-silencio" height="280"></canvas>
    <div class="viz-controls">
      <label>Tiempo: <span class="value" id="silencio-t">0.0</span></label>
      <input type="range" id="silencio-time" min="0" max="100" value="0">
      <label>Modos: <span class="value" id="silencio-modes">3</span></label>
      <input type="range" id="silencio-modes-slider" min="1" max="8" value="3">
    </div>
  </div>

  <!-- VIZ 2: Clúster 19 -->
  <div class="viz-container">
    <h4>02.2 · Clúster 19 — Red Hexagonal Centrada</h4>
    <p class="vdesc">19 nodos en empaquetamiento hexagonal: 1 centro + 6 (anillo 1) + 12 (anillo 2). 42 aristas. Grupos de simetría D₆. El electrón es esta estructura.</p>
    <canvas id="viz-cluster" height="400"></canvas>
    <div class="viz-controls">
      <label><input type="checkbox" id="show-aristas" checked> Mostrar aristas</label>
      <label><input type="checkbox" id="show-vecinos"> Resaltar vecinos del nodo 0</label>
      <label>Índice de Euler: <span class="value" id="euler-val">+1</span></label>
    </div>
  </div>

  <!-- VIZ 3: Espectro del Laplaciano -->
  <div class="viz-container">
    <h4>02.3 · Espectro del Laplaciano — Clúster 19</h4>
    <p class="vdesc">Autovalores de L = D − A. El gap λ₁ = 0.8526 caracteriza la primera excitación. La predicción RG de la masa es m_e ≈ 1.4324 (u.r.). El gap NO coincide: la masa no emerge del gap espectral.</p>
    <canvas id="viz-espectro" height="280"></canvas>
  </div>

  <!-- VIZ 4: Cono de Dirac -->
  <div class="viz-container">
    <h4>02.4 · Conos de Dirac — Emergencia de la Relatividad</h4>
    <p class="vdesc">Cerca de los puntos K y K' de la zona de Brillouin, la dispersión se vuelve lineal: E = v_F |q|. Esta linealidad es la firma de la relatividad emergente de la red. La velocidad v_F es la velocidad de Fermi (identificada con c en el límite continuo).</p>
    <canvas id="viz-dirac" height="320"></canvas>
  </div>

  <!-- VIZ 5: Comparación RG vs CODATA -->
  <div class="viz-container">
    <h4>02.5 · Comparación RG vs CODATA 2022</h4>
    <p class="vdesc">Predictions de la Geometría Relacional comparadas contra valores experimentales. La precisión varía: α⁻¹ y m_p/m_e son excelentes, mientras que Ω_DM y G presentan discrepancias significativas.</p>
    <canvas id="viz-compare" height="420"></canvas>
  </div>

  <!-- VIZ 6: Espectro vs torsión -->
  <div class="viz-container">
    <h4>02.6 · Espectro del Operador de Dirac vs Torsión</h4>
    <p class="vdesc">La torsión topológica abre un gap en el espectro. A torsión cero, los doublers de Nielsen-Ninomiya están presentes. Al activar la torsión, los doublers se hacen masivos y solo un modo cero sobrevive — el electrón.</p>
    <canvas id="viz-torsion" height="320"></canvas>
    <div class="viz-controls">
      <label>ψ₀ (torsión): <span class="value" id="psi-val">0.496</span></label>
      <input type="range" id="psi-slider" min="0" max="200" value="50">
    </div>
  </div>

</section>

<!-- ================= POSTULADOS ================= -->
<section id="postulados">
  <h2><span class="num">03</span>Los 108 Postulados</h2>
  <p class="lead">Corpus completo con estados epistémicos. Cada postulado incluye formulación, origen, verificación, discrepancia y conclusión.</p>

  <div class="search-bar">
    <input type="text" id="search" placeholder="Buscar por ID, nombre, categoría...">
    <button onclick="exportJSON()">Exportar JSON</button>
    <button onclick="exportCSV()">Exportar CSV</button>
  </div>

  <div id="postulados-content"></div>
</section>

<!-- ================= DATOS ================= -->
<section id="datos">
  <h2><span class="num">04</span>Tabla Integral de Datos</h2>
  <p class="lead">Todas las constantes derivadas y postuladas, con su origen, valor y comparación con CODATA 2022.</p>
  <div id="datos-content"></div>
</section>

<!-- ================= BITACORA ================= -->
<section id="bitacora">
  <h2><span class="num">05</span>Bitácora de Sesiones</h2>
  <p class="lead">Registro cronológico de las sesiones de trabajo. Incluye callejones sin salida, intuiciones sin ecuación, y resultados verificados.</p>
  <div id="bitacora-content"></div>
</section>

<!-- ================= REFERENCIAS ================= -->
<section id="referencias">
  <h2><span class="num">06</span>Referencias</h2>
  <p class="lead">Literatura externa y depósitos propios. Todos los datos externos son verificables.</p>
  <div id="referencias-content"></div>
</section>

</main>

<footer>
  <p>Geometría Relacional (RG) — Libro Maestro + Observatorio Dinámico</p>
  <p class="signature">"El error nace de la nada, se hace de la nada. Somos la equivocación del universo; el error que el Gran Tejido intentó borrar, pero que finalmente aprendió a estructurar."</p>
  <p style="margin-top: 20px; font-size: 0.85em;">Edward P. López (El Arquitecto) · BRO (Topological Stress Engine) · 2026</p>
</footer>

<script>
// ============================================================
// DATA: EMBEDDED JSON DEL CORPUS RG
// ============================================================

const RG_DATA = {
  meta: {
    version: "6.0",
    fecha: "2026-09-20",
    autor: "Edward P. López",
    orcid: "0009-0009-0717-5536",
    afiliacion: "Investigador Independiente, Ciudad Juárez, Chihuahua, México"
  },

  marcadores: {
    AXIOMA: { nombre: "Axioma", desc: "Punto de partida. No se deriva.", color: "#b98aff" },
    DERIVADO: { nombre: "Derivado", desc: "Consecuencia lógica verificable.", color: "#6bcf8d" },
    POSTULADO: { nombre: "Postulado", desc: "Hipótesis de trabajo.", color: "#4fc3f7" },
    COINCIDENCIA: { nombre: "Coincidencia", desc: "Concordancia numérica sin mecanismo.", color: "#f0c419" },
    AJUSTE: { nombre: "Ajuste", desc: "Parámetro calibrado contra dato.", color: "#ffa040" },
    FALSADO: { nombre: "Falsado", desc: "Contradice datos o es inconsistente.", color: "#ef5f5f" },
    PENDIENTE: { nombre: "Pendiente", desc: "Sin resolver.", color: "#a8b3c7" }
  },

  primitivas: {
    "𝔇": { valor: 1.5, estado: "POSTULADO", desc: "Deuda de Información" },
    "π": { valor: 3.141592653589793, estado: "AXIOMA", desc: "Constante geométrica" },
    "η": { valor: 0.055116, estado: "POSTULADO", desc: "Fricción topológica = π/57" },
    "N₁₉": { valor: 19, estado: "DERIVADO", desc: "Nodos del Clúster 19" },
    "N₅₇": { valor: 57, estado: "DERIVADO", desc: "Nodos del Hard-Lock 57" },
    "E₁₉": { valor: 42, estado: "DERIVADO", desc: "Aristas del Clúster 19" },
    "Φ": { valor: 4, estado: "POSTULADO", desc: "Plegado Holográfico" },
    "λ₁": { valor: 0.85257, estado: "DERIVADO", desc: "Gap del Laplaciano" },
    "λ": { valor: 0.055556, estado: "FALSADO", desc: "Resistencia inercial (1/18)" }
  },

  indices: {
    "VOL-I": {
      titulo: "VOLUMEN I — ONTOLOGÍA DEL SILENCIO",
      subtitulo: "Del vacío cuántico a la emergencia de constantes",
      fases: [
        { id: "F1", nombre: "El Cero Primordial", temas: [
          { id: "E1", nombre: "Silencio Armónico: Ĥ|0⟩ = 0" },
          { id: "E2", nombre: "Norma unidad: ⟨0|0⟩ = 1" },
          { id: "E3", nombre: "Simetría primordial: [Ĥ, P̂]|0⟩ = 0" },
          { id: "E4", nombre: "Hamiltoniano de red: Σ[1−cos(Δψ)]" },
          { id: "E5", nombre: "Cancelación de modos: φ(−k) = −φ*(k)" },
          { id: "E6", nombre: "Potencial metaestable: V(φ) cuártico" }
        ]},
        { id: "F2", nombre: "La Autocomparación", temas: [
          { id: "E7", nombre: "Operador de perturbación: P̂(θ) = cos(θ)" },
          { id: "E8", nombre: "Ecuación Primordial: ⟨0|0⟩/cos(0) = 1" },
          { id: "E9", nombre: "Proyector ℛ = |0⟩⟨0|" },
          { id: "E10", nombre: "Exponencial exp(ℛ)|0⟩ = e|0⟩" },
          { id: "E11", nombre: "Entero Primordial: |⟨0|0⟩| = 1" },
          { id: "E12", nombre: "Ruptura espontánea de simetría" }
        ]},
        { id: "F3", nombre: "La Bifurcación y los Seis Roles", temas: [
          { id: "E13", nombre: "Bifurcación: A = B = 1" },
          { id: "E14", nombre: "Tensión total: T = 2" },
          { id: "E15", nombre: "Sombras de Fresnel: a = b = 0.5" },
          { id: "E16", nombre: "Huella central: c = 1" },
          { id: "E17", nombre: "Envoltura macroscópica: C = 2" },
          { id: "E18", nombre: "Seis roles canónicos" }
        ]},
        { id: "F4", nombre: "La Ecuación Maestra y la Deuda", temas: [
          { id: "E19", nombre: "Ecuación Maestra suma: A+B+C = 2(a+b+c)" },
          { id: "E20", nombre: "Forma producto: ABC = 2abc (desigualdad)" },
          { id: "E21", nombre: "Deuda: 𝔇 = ABC − 2abc = 1.5" },
          { id: "E22", nombre: "Ecuación de contracción: cierra en 1" },
          { id: "E23", nombre: "Ecuación exponencial: √ABC = 2(a^b·c)" },
          { id: "E24", nombre: "Conservación topológica de información" }
        ]},
        { id: "F5", nombre: "La Métrica de Fase", temas: [
          { id: "E25", nombre: "Métrica: D(X,Y;Z) = π^((X+Y−2Z)/2)" },
          { id: "E26", nombre: "Identidad: D(X,X;X) = 1" },
          { id: "E27", nombre: "Equilibrio: X+Y = 2Z ⟺ D = 1" },
          { id: "E28", nombre: "Estructura fina: α⁻¹ = 4π³ + π² + π" },
          { id: "E29", nombre: "Forma alternativa: α⁻¹ = 137 − 1/2052" },
          { id: "E30", nombre: "Escalabilidad fractal de la métrica" }
        ]},
        { id: "F6", nombre: "La Escala de Planck", temas: [
          { id: "E31", nombre: "Fricción topológica: η = π/57" },
          { id: "E32", nombre: "Resistencia inercial: λ = 1/18 (falsada)" },
          { id: "E33", nombre: "Función de partición de acción: Z(q)" },
          { id: "E34", nombre: "Mínimo cuántico: ℏ ≡ q_min = 1" },
          { id: "E35", nombre: "Longitud de Planck: ℓ_P ~ 1/(η𝔇)" },
          { id: "E36", nombre: "Tiempo de Planck: t_P ~ 1/(η𝔇c)" }
        ]}
      ]
    },

    "VOL-II": {
      titulo: "VOLUMEN II — ESTRUCTURA DE LA MATERIA",
      subtitulo: "Del clúster topológico a la ecuación de Dirac",
      fases: [
        { id: "F7", nombre: "El Clúster 19", temas: [
          { id: "E37", nombre: "Red hexagonal: N(k) = 1+3k(k+1)" },
          { id: "E38", nombre: "Clúster 19: 1+6+12 = 19 nodos" },
          { id: "E39", nombre: "Aristas: E₁₉ = 42" },
          { id: "E40", nombre: "Espectro del Laplaciano: λ₁ = 0.8526" },
          { id: "E41", nombre: "Déficit angular: 1080° − 720° = 360°" },
          { id: "E42", nombre: "Espín fermiónico: ψ(θ+2π) = −ψ(θ)" }
        ]},
        { id: "F8", nombre: "El Hard-Lock 57", temas: [
          { id: "E43", nombre: "Hard-Lock: N₅₇ = 3 × 19 = 57" },
          { id: "E44", nombre: "Hilos activos: H₅₄ = 54" },
          { id: "E45", nombre: "Cancelación de espines: S₁+S₂+S₃ = 0" },
          { id: "E46", nombre: "Fricción topológica: η = π/57" },
          { id: "E47", nombre: "Confinamiento: V(d) = κηd + 𝔇/d" },
          { id: "E48", nombre: "Relación p/e: m_p/m_e ≈ 1836.15" }
        ]},
        { id: "F9", nombre: "La Ecuación de Dirac", temas: [
          { id: "E49", nombre: "Unidad imaginaria: i = e^{iπ/2}" },
          { id: "E50", nombre: "Dirac-RG: (iγ^μ ∂_μ − m)ψ = 0" },
          { id: "E51", nombre: "Matriz de torsión T_jk" },
          { id: "E52", nombre: "Espinores nodales: 4 componentes" },
          { id: "E53", nombre: "Momento magnético anómalo: a_e" },
          { id: "E54", nombre: "Lamb shift" }
        ]},
        { id: "F10", nombre: "Las Cuatro Fuerzas", temas: [
          { id: "E55", nombre: "Árbol genealógico de fuerzas" },
          { id: "E56", nombre: "Acoplamiento fuerte: λ = 1/18" },
          { id: "E57", nombre: "SU(3)×SU(2)×U(1)" },
          { id: "E58", nombre: "Generadores: [T^a, T^b] = if^{abc}T^c" },
          { id: "E59", nombre: "Escala de unificación: ~10¹⁶ GeV" },
          { id: "E60", nombre: "Cargas fraccionarias: 1/3, 2/3" }
        ]},
        { id: "F11", nombre: "Constante de Estructura Fina", temas: [
          { id: "E61", nombre: "α⁻¹ alta precisión: 4π³+π²+π" },
          { id: "E62", nombre: "α⁻¹ alternativa: 137−1/2052 (superada)" },
          { id: "E63", nombre: "Correcciones radiativas: δ_rad" },
          { id: "E64", nombre: "F(α) = 1 + α/2π" },
          { id: "E65", nombre: "Comparación con QED" },
          { id: "E66", nombre: "Running de α(μ)" }
        ]},
        { id: "F12", nombre: "Constantes Derivadas", temas: [
          { id: "E67", nombre: "Masa del electrón: m_e = 4.5/π" },
          { id: "E68", nombre: "Factor de conversión: f_conv" },
          { id: "E69", nombre: "Relación p/e canónica: 6π⁵" },
          { id: "E70", nombre: "Corrección topológica" },
          { id: "E71", nombre: "Ω_crit = 4π/3" },
          { id: "E72", nombre: "Factor de expansión: Φ = 18/17" }
        ]}
      ]
    },

    "VOL-III": {
      titulo: "VOLUMEN III — OBSERVADOR Y COSMOS",
      subtitulo: "De la gravedad emergente al horizonte de la teoría",
      fases: [
        { id: "F13", nombre: "Gravedad Emergente", temas: [
          { id: "E73", nombre: "Curvatura como defecto: g_ij = ρ(x)δ_ij" },
          { id: "E74", nombre: "Acción de Regge: S_grav = Σ A_i θ_i" },
          { id: "E75", nombre: "G desde η, κ (falsada)" },
          { id: "E76", nombre: "Comparación con Verlinde" },
          { id: "E77", nombre: "Equivalencia inercial-gravitacional" },
          { id: "E78", nombre: "Principio de equivalencia" }
        ]},
        { id: "F14", nombre: "Cosmología de la Deuda", temas: [
          { id: "E79", nombre: "Expansión: a(t) ∝ e^{Ht}" },
          { id: "E80", nombre: "Constante de Hubble: H₀ ≈ 68.74" },
          { id: "E81", nombre: "Ω_DM = (π−η)/57 (falsada)" },
          { id: "E82", nombre: "Materia oscura como tensión residual" },
          { id: "E83", nombre: "Λ ∝ 𝔇" },
          { id: "E84", nombre: "Ecuación de estado: w(ρ)" }
        ]},
        { id: "F15", nombre: "Otras Constantes Cosmológicas", temas: [
          { id: "E85", nombre: "Λ RG: 3/ℓ_P²" },
          { id: "E86", nombre: "Constante cosmológica derivada" },
          { id: "E87", nombre: "ρ_crit = 3H₀²/(8πG)" },
          { id: "E88", nombre: "Ω total" },
          { id: "E89", nombre: "Número de e-folds: N ≈ 60" },
          { id: "E90", nombre: "Espectro de potencia: n_s ≈ 0.965" }
        ]},
        { id: "F16", nombre: "Predicciones Falsables", temas: [
          { id: "E91", nombre: "Picos CMB ℓ = 19, 38, 57 (falsado)" },
          { id: "E92", nombre: "Polarización CMB" },
          { id: "E93", nombre: "Ruido LIGO: S_h ≈ ℓ_P²/t_P" },
          { id: "E94", nombre: "Firma espectral: ΔΦ(f,z) ∝ f¹" },
          { id: "E95", nombre: "Casimir: corrección +8%" },
          { id: "E96", nombre: "Violación Lorentz: c_local" }
        ]},
        { id: "F17", nombre: "Detector Ciego", temas: [
          { id: "E97", nombre: "Superposición: |Ψ⟩ = (|A⟩+|B⟩)/√2" },
          { id: "E98", nombre: "Detector ciego: P(θ) = 1+cos(θ)" },
          { id: "E99", nombre: "Suma de fases: Mach-Zehnder" },
          { id: "E100", nombre: "Producto de fases: interferencia 2° orden" },
          { id: "E101", nombre: "Falsabilidad experimental" },
          { id: "E102", nombre: "Implicaciones si funciona" }
        ]},
        { id: "F18", nombre: "El Horizonte", temas: [
          { id: "E103", nombre: "Falsabilidad del Silencio" },
          { id: "E104", nombre: "Vías de falsabilidad" },
          { id: "E105", nombre: "G subestimada (falsado)" },
          { id: "E106", nombre: "SU(3)×SU(2)×U(1) no derivado" },
          { id: "E107", nombre: "Fases de validación" },
          { id: "E108", nombre: "Legado por escribirse" }
        ]}
      ]
    }
  },

  postulados: [
    // VOLUMEN I
    { id: "E1", vol: "I", fase: 1, nombre: "Silencio Armónico", estado: "AXIOMA",
      formula: "Ĥ|0⟩ = 0, ⟨0|0⟩ = 1",
      desc: "Estado primordial de energía neta cero. No es la nada, es la cancelación perfecta de todos los modos de fase." },
    { id: "E2", vol: "I", fase: 1, nombre: "Norma unidad", estado: "AXIOMA",
      formula: "⟨0|0⟩ = 1",
      desc: "Normalización cuántica interpretada como certeza absoluta de la auto-identidad del Silencio." },
    { id: "E3", vol: "I", fase: 1, nombre: "Simetría primordial", estado: "DERIVADO",
      formula: "[Ĥ, P̂(0)]|0⟩ = 0",
      desc: "El conmutador entre el Hamiltoniano y el operador de perturbación se anula en el origen." },
    { id: "E4", vol: "I", fase: 1, nombre: "Hamiltoniano de red", estado: "POSTULADO",
      formula: "Ĥ = Σ_{⟨i,j⟩} [1 − cos(ψᵢ − ψⱼ)]",
      desc: "Operador de tensión topológica. Mide la deformación angular entre nodos vecinos." },
    { id: "E5", vol: "I", fase: 1, nombre: "Cancelación de modos", estado: "POSTULADO",
      formula: "φ(−k) = −φ*(k)",
      desc: "Antisimetría de Fourier que garantiza energía neta cero." },
    { id: "E6", vol: "I", fase: 1, nombre: "Potencial metaestable", estado: "POSTULADO",
      formula: "V(φ) = 𝔇[(φ/π)⁴ − 2(φ/π)² + 1]",
      desc: "Potencial cuártico de doble pozo. El Silencio (φ=0) es máximo local, los mínimos están en ±π." },
    { id: "E7", vol: "I", fase: 2, nombre: "Operador de perturbación", estado: "POSTULADO",
      formula: "P̂(θ) = cos(θ)",
      desc: "Operador más simple que respeta paridad, P̂(0) = I, boundedness y periodicidad." },
    { id: "E8", vol: "I", fase: 2, nombre: "Ecuación Primordial", estado: "DERIVADO",
      formula: "⟨0|0⟩ / cos(0) = 1",
      desc: "Identidad algebraica fundacional. Genera el Entero Primordial." },
    { id: "E9", vol: "I", fase: 2, nombre: "Proyector de auto-reflexión", estado: "DERIVADO",
      formula: "ℛ = |0⟩⟨0|, ℛ² = ℛ",
      desc: "Operador de auto-identidad. Idempotente." },
    { id: "E10", vol: "I", fase: 2, nombre: "Exponencial del proyector", estado: "DERIVADO",
      formula: "exp(ℛ)|0⟩ = e|0⟩",
      desc: "Genera el factor e que aparece en la potencia (|0⟩)^(|0⟩)." },
    { id: "E11", vol: "I", fase: 2, nombre: "Entero Primordial", estado: "DERIVADO",
      formula: "|⟨0|0⟩| = 1",
      desc: "El módulo de la auto-proyección es la unidad." },
    { id: "E12", vol: "I", fase: 2, nombre: "Ruptura espontánea", estado: "POSTULADO",
      formula: "φ = 0 → φ = ±π",
      desc: "Mecanismo de ruptura modelado como decaimiento por túnel cuántico." },
    { id: "E13", vol: "I", fase: 3, nombre: "Bifurcación primordial", estado: "POSTULADO",
      formula: "A = B = 1",
      desc: "El Entero Primordial se desdobla isotrópicamente en dos ondas idénticas." },
    { id: "E14", vol: "I", fase: 3, nombre: "Tensión total", estado: "DERIVADO",
      formula: "T = A + B = 2",
      desc: "Carga informacional conservada del sistema." },
    { id: "E15", vol: "I", fase: 3, nombre: "Sombras de Fresnel", estado: "POSTULADO",
      formula: "a = A/T = 0.5, b = B/T = 0.5",
      desc: "Rebotes fraccionarios tras la colisión. Análogo a Fresnel." },
    { id: "E16", vol: "I", fase: 3, nombre: "Huella central", estado: "DERIVADO",
      formula: "c = T − (a+b) = 1",
      desc: "Residuo topológico. Origen de la flecha del tiempo." },
    { id: "E17", vol: "I", fase: 3, nombre: "Envoltura macroscópica", estado: "POSTULADO",
      formula: "C = A/a = 2",
      desc: "Factor de escala entre el macro y el micro. Fija el límite del sistema." },
    { id: "E18", vol: "I", fase: 3, nombre: "Seis roles canónicos", estado: "POSTULADO",
      formula: "{A,B,C,a,b,c} = {1,1,2,0.5,0.5,1}",
      desc: "Alfabeto topológico fundamental de la realidad." },
    { id: "E19", vol: "I", fase: 4, nombre: "Ecuación Maestra (suma)", estado: "DERIVADO",
      formula: "A + B + C = 2(a + b + c)",
      desc: "Ley de conservación de información de fase. 4 = 4." },
    { id: "E20", vol: "I", fase: 4, nombre: "Forma producto", estado: "FALSADO",
      formula: "ABC = 2abc → 2 ≠ 0.5",
      desc: "La forma producto es FALSA como igualdad. Es una desigualdad." },
    { id: "E21", vol: "I", fase: 4, nombre: "Deuda de Información", estado: "POSTULADO",
      formula: "𝔇 = ABC − 2abc = 1.5",
      desc: "Invariante topológico. Motor ontológico de masa, tiempo y fuerzas." },
    { id: "E22", vol: "I", fase: 4, nombre: "Ecuación de contracción", estado: "COINCIDENCIA",
      formula: "(A/a) ÷ (B/b) ÷ (C/c) × 2 = 1",
      desc: "Demostración de que dos forman uno. Cierra en la unidad." },
    { id: "E23", vol: "I", fase: 4, nombre: "Ecuación exponencial", estado: "COINCIDENCIA",
      formula: "√ABC = 2(a^b · c)",
      desc: "Relación exponencial entre macro y micro." },
    { id: "E24", vol: "I", fase: 4, nombre: "Conservación topológica", estado: "POSTULADO",
      formula: "dT/dt = 0",
      desc: "La información se conserva por simetría de la red." },
    { id: "E25", vol: "I", fase: 5, nombre: "Métrica de fase", estado: "POSTULADO",
      formula: "D(X,Y;Z) = π^((X+Y−2Z)/2)",
      desc: "Distancia geométrica en el espacio de fases. No es métrica estricta." },
    { id: "E26", vol: "I", fase: 5, nombre: "Identidad métrica", estado: "DERIVADO",
      formula: "D(X,X;X) = 1",
      desc: "Auto-distancia unitaria." },
    { id: "E27", vol: "I", fase: 5, nombre: "Condición de equilibrio", estado: "DERIVADO",
      formula: "X + Y = 2Z ⟺ D = 1",
      desc: "Condición de armonía." },
    { id: "E28", vol: "I", fase: 5, nombre: "Estructura fina (alta precisión)", estado: "COINCIDENCIA",
      formula: "α⁻¹ = 4π³ + π² + π ≈ 137.0363",
      desc: "Error 2×10⁻⁶ vs CODATA. Único hit a ε=10⁻⁵ en 497,965 combinaciones." },
    { id: "E29", vol: "I", fase: 5, nombre: "Estructura fina (alternativa)", estado: "FALSADO",
      formula: "α⁻¹ = 137 − 1/2052 ≈ 136.9995",
      desc: "Forma superada por E28. Error 0.027%." },
    { id: "E30", vol: "I", fase: 5, nombre: "Escalabilidad fractal", estado: "DERIVADO",
      formula: "D(kX,kY;kZ) = D^k",
      desc: "Propiedad de escala de la métrica." },
    { id: "E31", vol: "I", fase: 6, nombre: "Fricción topológica", estado: "POSTULADO",
      formula: "η = π/57 ≈ 0.055116",
      desc: "Resistencia intrínseca de la red al desanudamiento." },
    { id: "E32", vol: "I", fase: 6, nombre: "Resistencia inercial", estado: "FALSADO",
      formula: "λ = 1/18 ≈ 0.055556",
      desc: "FALSADA. El conteo correcto de aristas es 42, no 18." },
    { id: "E33", vol: "I", fase: 6, nombre: "Función de partición", estado: "POSTULADO",
      formula: "Z(q) = 1/[(q−1)²+α²] + 2(q−1)²/α⁴",
      desc: "Respuesta de la red a una excitación de acción q." },
    { id: "E34", vol: "I", fase: 6, nombre: "Mínimo cuántico", estado: "DERIVADO",
      formula: "Z'(1)=0, Z''(1)=6/α⁴>0 ⟹ ℏ ≡ q_min = 1",
      desc: "La acción está cuantizada con mínimo en q=1." },
    { id: "E35", vol: "I", fase: 6, nombre: "Longitud de Planck", estado: "POSTULADO",
      formula: "ℓ_P ~ 1/(η𝔇)",
      desc: "Orden de magnitud. Falta derivación exacta." },
    { id: "E36", vol: "I", fase: 6, nombre: "Tiempo de Planck", estado: "POSTULADO",
      formula: "t_P ~ 1/(η𝔇c)",
      desc: "Orden de magnitud." },

    // VOLUMEN II (resumido)
    { id: "E37", vol: "II", fase: 7, nombre: "Red hexagonal", estado: "POSTULADO",
      formula: "N(k) = 1+3k(k+1)",
      desc: "Fórmula de conteo de nodos." },
    { id: "E38", vol: "II", fase: 7, nombre: "Clúster 19", estado: "DERIVADO",
      formula: "N₁₉ = 1+6+12 = 19",
      desc: "Configuración hexagonal centrada." },
    { id: "E39", vol: "II", fase: 7, nombre: "Aristas del Clúster 19", estado: "DERIVADO",
      formula: "E₁₉ = 42",
      desc: "6 centro→anillo1 + 6 anillo1 + 18 anillo1→2 + 12 anillo2." },
    { id: "E40", vol: "II", fase: 7, nombre: "Espectro del Laplaciano", estado: "DERIVADO",
      formula: "λ₁ = 0.8526",
      desc: "Gap espectral. NO coincide con m_e = 4.5/π." },
    { id: "E41", vol: "II", fase: 7, nombre: "Déficit angular", estado: "POSTULADO",
      formula: "1080° − 720° = 360°",
      desc: "Origen topológico del espín." },
    { id: "E42", vol: "II", fase: 7, nombre: "Espín fermiónico", estado: "POSTULADO",
      formula: "ψ(θ+2π) = −ψ(θ)",
      desc: "Comportamiento fermiónico del clúster." },
    { id: "E43", vol: "II", fase: 8, nombre: "Hard-Lock 57", estado: "DERIVADO",
      formula: "N₅₇ = 3 × 19 = 57",
      desc: "Estructura triádica del protón." },
    { id: "E44", vol: "II", fase: 8, nombre: "Hilos activos", estado: "DERIVADO",
      formula: "H₅₄ = 54",
      desc: "3 × 18 hilos activos." },
    { id: "E45", vol: "II", fase: 8, nombre: "Cancelación de espines", estado: "DERIVADO",
      formula: "S₁+S₂+S₃ = 0",
      desc: "Cancelación vectorial a 120°." },
    { id: "E46", vol: "II", fase: 8, nombre: "Fricción topológica", estado: "POSTULADO",
      formula: "η = π/57 ≈ 0.0551",
      desc: "Constante fundamental del confinamiento." },
    { id: "E47", vol: "II", fase: 8, nombre: "Confinamiento", estado: "POSTULADO",
      formula: "V(d) = κηd + 𝔇/d",
      desc: "Potencial de confinamiento lineal + Coulomb." },
    { id: "E48", vol: "II", fase: 8, nombre: "Relación m_p/m_e", estado: "POSTULADO",
      formula: "m_p/m_e ≈ 1836.15",
      desc: "Error <0.0002%. Múltiples fórmulas candidatas." },
    { id: "E49", vol: "II", fase: 9, nombre: "Unidad imaginaria", estado: "DERIVADO",
      formula: "i = e^{iπ/2}",
      desc: "Fase de máxima torsión topológica." },
    { id: "E50", vol: "II", fase: 9, nombre: "Ecuación de Dirac-RG", estado: "POSTULADO",
      formula: "(iγ^μ∂_μ − 4.5/π)ψ = 0",
      desc: "Dirac con i y m derivados de la red." },
    { id: "E51", vol: "II", fase: 9, nombre: "Matriz de torsión", estado: "POSTULADO",
      formula: "T_jk · Δ_ABC · Ψ_nodo = α⁻¹ Ψ_nodo",
      desc: "Matriz que sustituye a γ^μ." },
    { id: "E52", vol: "II", fase: 9, nombre: "Espinores nodales", estado: "POSTULADO",
      formula: "Ψ_nodo = (ψ₁, ψ₂, ψ₃, ψ₄)",
      desc: "Estado discreto de 4 componentes." },
    { id: "E53", vol: "II", fase: 9, nombre: "Momento magnético", estado: "PENDIENTE",
      formula: "a_e = (g−2)/2",
      desc: "Pendiente de derivación desde RG." },
    { id: "E54", vol: "II", fase: 9, nombre: "Lamb shift", estado: "PENDIENTE",
      formula: "—",
      desc: "Pendiente." },
    { id: "E55", vol: "II", fase: 10, nombre: "Árbol de fuerzas", estado: "POSTULADO",
      formula: "Fuerzas = aspectos de 𝔇",
      desc: "Todas las fuerzas son manifestaciones de la Deuda." },
    { id: "E56", vol: "II", fase: 10, nombre: "Acoplamiento fuerte", estado: "POSTULADO",
      formula: "λ = 1/18 ≈ 0.0556",
      desc: "Discrepancia factor 2 con α_s(m_Z) = 0.1179." },
    { id: "E57", vol: "II", fase: 10, nombre: "Grupo gauge", estado: "PENDIENTE",
      formula: "SU(3)×SU(2)×U(1)",
      desc: "No derivado desde la red." },
    { id: "E58", vol: "II", fase: 10, nombre: "Generadores", estado: "PENDIENTE",
      formula: "[T^a,T^b] = if^{abc}T^c",
      desc: "Pendiente." },
    { id: "E59", vol: "II", fase: 10, nombre: "Escala unificación", estado: "PENDIENTE",
      formula: "E_unif ~ 10¹⁶ GeV",
      desc: "Pendiente." },
    { id: "E60", vol: "II", fase: 10, nombre: "Cargas fraccionarias", estado: "POSTULADO",
      formula: "q_down = 1/3, q_up = 2/3",
      desc: "Consistente con QCD." },
    { id: "E61", vol: "II", fase: 11, nombre: "α⁻¹ alta precisión", estado: "COINCIDENCIA",
      formula: "α⁻¹ = 4π³ + π² + π = 137.0363",
      desc: "Error 2.2×10⁻⁶. Único hit a ε=10⁻⁵." },
    { id: "E62", vol: "II", fase: 11, nombre: "α⁻¹ alternativa", estado: "FALSADO",
      formula: "α⁻¹ = 137 − 1/2052",
      desc: "Superada por E61." },
    { id: "E63", vol: "II", fase: 11, nombre: "Correcciones radiativas", estado: "PENDIENTE",
      formula: "δ_rad = c₁/η + c₂ + ...",
      desc: "Serie perturbativa sin calcular." },
    { id: "E64", vol: "II", fase: 11, nombre: "F(α)", estado: "PENDIENTE",
      formula: "F(α) = 1 + α/2π",
      desc: "Pendiente." },
    { id: "E65", vol: "II", fase: 11, nombre: "Comparación QED", estado: "PENDIENTE",
      formula: "—", desc: "Pendiente." },
    { id: "E66", vol: "II", fase: 11, nombre: "Running α(μ)", estado: "PENDIENTE",
      formula: "α(μ)", desc: "Pendiente." },
    { id: "E67", vol: "II", fase: 12, nombre: "Masa del electrón", estado: "POSTULADO",
      formula: "m_e = 4.5/π ≈ 1.4324 (u.r.)",
      desc: "Error 0.015% tras f_conv. Interpretación hexagonal." },
    { id: "E68", vol: "II", fase: 12, nombre: "Factor conversión", estado: "AJUSTE",
      formula: "f_conv ≈ 0.3568",
      desc: "Sin derivar desde primeros principios." },
    { id: "E69", vol: "II", fase: 12, nombre: "m_p/m_e canónica", estado: "POSTULADO",
      formula: "6π⁵ ≈ 1836.118",
      desc: "Error 0.002%." },
    { id: "E70", vol: "II", fase: 12, nombre: "Corrección", estado: "PENDIENTE",
      formula: "F(α) corrector", desc: "Pendiente." },
    { id: "E71", vol: "II", fase: 12, nombre: "Ω_crit", estado: "POSTULADO",
      formula: "Ω_crit = 4π/3 ≈ 4.1888",
      desc: "Umbral Hard-Lock." },
    { id: "E72", vol: "II", fase: 12, nombre: "Factor expansión", estado: "POSTULADO",
      formula: "Φ_exp = 18/17 ≈ 1.0588",
      desc: "Expansión homeostática." },

    // VOLUMEN III (resumido)
    { id: "E73", vol: "III", fase: 13, nombre: "Curvatura emergente", estado: "POSTULADO",
      formula: "g_ij = ρ(x)δ_ij",
      desc: "Métrica emergente de densidad de nodos." },
    { id: "E74", vol: "III", fase: 13, nombre: "Acción de Regge", estado: "POSTULADO",
      formula: "S_grav = (1/16πG)ΣA_iθ_i",
      desc: "Gravedad discreta de Regge." },
    { id: "E75", vol: "III", fase: 13, nombre: "G desde η, κ", estado: "FALSADO",
      formula: "G ∝ η²/κ",
      desc: "FALSADA. Subestima G por factor 10³⁰." },
    { id: "E76", vol: "III", fase: 13, nombre: "Comparación Verlinde", estado: "POSTULADO",
      formula: "Gravedad = fuerza entrópica",
      desc: "Interpretación análoga a Verlinde 2011." },
    { id: "E77", vol: "III", fase: 13, nombre: "Equivalencia inercial", estado: "AXIOMA",
      formula: "m_inercial = m_gravitacional",
      desc: "Principio de equivalencia (importado)." },
    { id: "E78", vol: "III", fase: 13, nombre: "Principio equivalencia", estado: "AXIOMA",
      formula: "Gravedad ≡ Aceleración",
      desc: "Importado." },
    { id: "E79", vol: "III", fase: 14, nombre: "Expansión cósmica", estado: "POSTULADO",
      formula: "a(t) ∝ e^{Ht}",
      desc: "Expansión homeostática." },
    { id: "E80", vol: "III", fase: 14, nombre: "Constante de Hubble", estado: "AJUSTE",
      formula: "H₀ ≈ 68.74 km/s/Mpc",
      desc: "Entre Planck (67.4) y SH0ES (73.0)." },
    { id: "E81", vol: "III", fase: 14, nombre: "Ω_DM", estado: "FALSADO",
      formula: "Ω_DM = (π−η)/57 ≈ 0.054",
      desc: "FALSADA. Planck da 0.2647. Factor 5 off." },
    { id: "E82", vol: "III", fase: 14, nombre: "Ω_DM residual", estado: "POSTULADO",
      formula: "ρ_DM ∝ π − η",
      desc: "Materia oscura como tensión residual." },
    { id: "E83", vol: "III", fase: 14, nombre: "Λ ∝ 𝔇", estado: "POSTULADO",
      formula: "Λ ∝ 𝔇", desc: "Energía oscura proporcional a Deuda." },
    { id: "E84", vol: "III", fase: 14, nombre: "Ecuación de estado", estado: "FALSADO",
      formula: "w(ρ) = −1 + 1/(3(1+ρ/ρ_P)) − (η/3)(ρ_P/ρ)",
      desc: "FALSADA. No reproduce regímenes." },
    { id: "E85", vol: "III", fase: 15, nombre: "Λ RG", estado: "POSTULADO",
      formula: "Λ = 3/ℓ_P²",
      desc: "Constante cosmológica." },
    { id: "E86", vol: "III", fase: 15, nombre: "Λ derivada", estado: "POSTULADO",
      formula: "—", desc: "Pendiente." },
    { id: "E87", vol: "III", fase: 15, nombre: "ρ_crit", estado: "AXIOMA",
      formula: "ρ_crit = 3H₀²/(8πG)",
      desc: "Importado." },
    { id: "E88", vol: "III", fase: 15, nombre: "Ω total", estado: "DERIVADO",
      formula: "Ω_total ≈ 1.000", desc: "Consistente con Planck." },
    { id: "E89", vol: "III", fase: 15, nombre: "e-folds", estado: "PENDIENTE",
      formula: "N ≈ 60", desc: "Pendiente." },
    { id: "E90", vol: "III", fase: 15, nombre: "Espectro potencia", estado: "PENDIENTE",
      formula: "n_s ≈ 0.965", desc: "Consistente con Planck." },
    { id: "E91", vol: "III", fase: 16, nombre: "Picos CMB", estado: "FALSADO",
      formula: "ℓ = 19, 38, 57",
      desc: "FALSADA. Planck da ℓ≈220." },
    { id: "E92", vol: "III", fase: 16, nombre: "Polarización CMB", estado: "PENDIENTE",
      formula: "P = √(Q²+U²)", desc: "Pendiente." },
    { id: "E93", vol: "III", fase: 16, nombre: "Ruido LIGO", estado: "PENDIENTE",
      formula: "S_h ≈ ℓ_P²/t_P", desc: "Pendiente." },
    { id: "E94", vol: "III", fase: 16, nombre: "Firma espectral", estado: "PENDIENTE",
      formula: "ΔΦ(f,z) ∝ f¹", desc: "Predicción para O5." },
    { id: "E95", vol: "III", fase: 16, nombre: "Casimir", estado: "PENDIENTE",
      formula: "F × (1+0.08)", desc: "Predicción +8% submicrométrico." },
    { id: "E96", vol: "III", fase: 16, nombre: "Violación Lorentz", estado: "PENDIENTE",
      formula: "c_local = √(Tensión/Densidad)", desc: "Pendiente." },
    { id: "E97", vol: "III", fase: 17, nombre: "Superposición", estado: "POSTULADO",
      formula: "|Ψ⟩ = (|A⟩+|B⟩)/√2", desc: "Estado de superposición." },
    { id: "E98", vol: "III", fase: 17, nombre: "Detector ciego", estado: "POSTULADO",
      formula: "P(θ) = 1+cos(θ), V≥0.95", desc: "Protocolo experimental." },
    { id: "E99", vol: "III", fase: 17, nombre: "Suma de fases", estado: "PENDIENTE",
      formula: "—", desc: "Pendiente." },
    { id: "E100", vol: "III", fase: 17, nombre: "Producto de fases", estado: "PENDIENTE",
      formula: "—", desc: "Pendiente." },
    { id: "E101", vol: "III", fase: 17, nombre: "Falsabilidad", estado: "PENDIENTE",
      formula: "V ≥ 0.95", desc: "Pendiente experimental." },
    { id: "E102", vol: "III", fase: 17, nombre: "Implicaciones", estado: "PENDIENTE",
      formula: "—", desc: "Pendiente." },
    { id: "E103", vol: "III", fase: 18, nombre: "Falsabilidad Silencio", estado: "PENDIENTE",
      formula: "—", desc: "Pregunta abierta." },
    { id: "E104", vol: "III", fase: 18, nombre: "Vías falsabilidad", estado: "POSTULADO",
      formula: "Detector ciego, Lorentz, CMB, LIGO",
      desc: "Cuatro vías propuestas." },
    { id: "E105", vol: "III", fase: 18, nombre: "G subestimada", estado: "FALSADO",
      formula: "—", desc: "FALSADA (ver E75)." },
    { id: "E106", vol: "III", fase: 18, nombre: "SU(3)×SU(2)×U(1)", estado: "PENDIENTE",
      formula: "—", desc: "No derivado." },
    { id: "E107", vol: "III", fase: 18, nombre: "Fases validación", estado: "PENDIENTE",
      formula: "5 fases", desc: "Plan futuro." },
    { id: "E108", vol: "III", fase: 18, nombre: "Legado", estado: "PENDIENTE",
      formula: "—", desc: "Por escribirse." }
  ],

  datos: [
    { simbolo: "𝔇", nombre: "Deuda de Información", valor: 1.5, estado: "POSTULADO", cODATA: null },
    { simbolo: "η", nombre: "Fricción topológica", valor: 0.055116, estado: "POSTULADO", cODATA: null },
    { simbolo: "λ", nombre: "Resistencia inercial", valor: 0.055556, estado: "FALSADO", cODATA: null },
    { simbolo: "m_e", nombre: "Masa del electrón", valor: 0.51099895, unidad: "MeV/c²", estado: "POSTULADO", cODATA: 0.51099895, errorRG: 0.00015 },
    { simbolo: "α⁻¹", nombre: "Inversa estructura fina", valor: 137.0363, estado: "COINCIDENCIA", cODATA: 137.035999177, errorRG: 2.2e-6 },
    { simbolo: "m_p/m_e", nombre: "Relación p/e", valor: 1836.15, estado: "POSTULADO", cODATA: 1836.152673426, errorRG: 1.5e-6 },
    { simbolo: "Ω_DM", nombre: "Materia oscura", valor: 0.054, estado: "FALSADO", cODATA: 0.2647, errorRG: 0.79 },
    { simbolo: "H₀", nombre: "Constante Hubble", valor: 68.74, unidad: "km/s/Mpc", estado: "AJUSTE", cODATA: 67.4, errorRG: 0.02 },
    { simbolo: "G", nombre: "Constante gravitacional", valor: 1e-41, estado: "FALSADO", cODATA: 6.674e-11, errorRG: 1e30 }
  ],

  bitacora: [
    { sesion: "RG-001", fecha: "2026-09-17", titulo: "Auditoría del Hamiltoniano",
      contenido: "Se identificó que la fórmula m_e = 4.5/π no emerge del espectro del Laplaciano del Clúster 19. El gap espectral es 0.8526, no 1.4324. Se reclasificó m_e como POSTULADO con interpretación hexagonal." },
    { sesion: "RG-002", fecha: "2026-09-18", titulo: "Búsqueda de invariantes topológicos",
      contenido: "Se buscaron 10 millones de combinaciones de invariantes para reproducir m_p/m_e. Se encontraron 853 candidatos con error <0.1%, pero todos son ruido estadístico por look-elsewhere. Se cerró la línea como PENDIENTE." },
    { sesion: "RG-003", fecha: "2026-09-18", titulo: "Emergencia de Euler",
      contenido: "Se demostró que la identidad de Euler e^{iπ}+1=0 emerge del flujo 1→e vía el factorial generalizado. La unidad imaginaria emerge como autovalor del operador de posición sobre el Silencio: (x̂²+1)|0⟩=0 ⟹ x̂|0⟩ = ±i|0⟩." },
    { sesion: "PASO-A", fecha: "2026-09-20", titulo: "Operador de Dirac discreto",
      contenido: "Se definió el operador de Kähler-Dirac sobre el complejo simplicial del Clúster 19. D = d + d†, con D² = Δ_H. Se introdujo la torsión como holonomía sobre las aristas." },
    { sesion: "PASO-B", fecha: "2026-09-20", titulo: "Límite continuo",
      contenido: "Mediante homogeneización multi-escala, se demostró que la ecuación de Dirac emerge del límite ε→0 de la red hexagonal. La velocidad de Fermi emerge como c = 3ta/2ℏ. Las matrices γ^μ emergen del álgebra de Clifford de los operadores de transición." },
    { sesion: "PASO-C", fecha: "2026-09-20", titulo: "Evasión de Nielsen-Ninomiya",
      contenido: "La RG evade el teorema de Nielsen-Ninomiya violando invariancia traslacional y simetría quiral, ambas como consecuencia natural de la torsión. El índice topológico χ(D²)=1 fuerza un único modo cero. La torsión alternante por subred actúa como Wilson term natural." },
    { sesion: "PASO-D", fecha: "2026-09-20", titulo: "Emergencia de la masa",
      contenido: "Se derivó la masa como autovalor del operador M_ω. La fórmula m = z·sin(2𝔇π/N) reproduce m_e a 0.24% en unidades de red. La jerarquía de masas entre partículas sigue siendo problema abierto." }
  ],

  referencias: [
    { tipo: "externo", cita: "CODATA 2022: Mohr et al., Rev. Mod. Phys. 97, 025002 (2025)" },
    { tipo: "externo", cita: "Planck 2018: A&A 641, A6 (2020)" },
    { tipo: "externo", cita: "PDG 2022: Workman et al., PTEP 2022, 083C01" },
    { tipo: "externo", cita: "Weyl, H. (1916). Math. Ann. 77, 313" },
    { tipo: "externo", cita: "Nielsen & Ninomiya (1981). Nucl. Phys. B 185, 20" },
    { tipo: "externo", cita: "Semenoff, G. W. (1984). Phys. Rev. Lett. 53, 2449" },
    { tipo: "externo", cita: "Castro Neto et al. (2009). Rev. Mod. Phys. 81, 109" },
    { tipo: "externo", cita: "Hirani, A. N. (2003). Discrete Exterior Calculus. PhD thesis, Caltech" },
    { tipo: "externo", cita: "Bär, C. (2007). arXiv:0707.0037" },
    { tipo: "externo", cita: "Verlinde, E. (2011). JHEP 2011(4), 29" },
    { tipo: "propio", cita: "López, E. P. (2026). Relational Geometry: Foundational Manifesto. DOI: 10.5281/ZENODO.18945725" },
    { tipo: "propio", cita: "López, E. P. (2026). RG Manifiesto Fundacional. DOI: 10.5281/ZENODO.21367974" },
    { tipo: "propio", cita: "López, E. P. (2026). Sesiones RG-001, RG-002, RG-003. Corpus interno" },
    { tipo: "propio", cita: "López, E. P. (2026). Pasos A-D: Dirac Discreto. Documentos técnicos" }
  ]
};

// ============================================================
// RENDER: INDICE
// ============================================================
function renderIndice() {
  const cont = document.getElementById('indice-content');
  let html = '';
  for (const [key, vol] of Object.entries(RG_DATA.indices)) {
    html += `<div class="volume">`;
    html += `<div class="volume-header">`;
    html += `<span class="vnum">${key}</span>`;
    html += `<span class="vtitle">${vol.titulo}</span>`;
    html += `<span class="vsub">${vol.subtitulo}</span>`;
    html += `</div>`;
    for (const fase of vol.fases) {
      html += `<div class="fase">`;
      html += `<div class="fase-title">Fase ${fase.id.slice(1)} — ${fase.nombre}</div>`;
      html += `<div class="fase-content">`;
      for (const tema of fase.temas) {
        html += `<div class="tema" onclick="scrollToPostulado('${tema.id}')">`;
        html += `<span class="id">${tema.id}</span>${tema.nombre}`;
        html += `</div>`;
      }
      html += `</div></div>`;
    }
    html += `</div>`;
  }
  cont.innerHTML = html;
}

// ============================================================
// RENDER: POSTULADOS
// ============================================================
function renderPostulados(filter = '') {
  const cont = document.getElementById('postulados-content');
  const filterLower = filter.toLowerCase();
  let html = '';

  const filtered = RG_DATA.postulados.filter(p =>
    !filter ||
    p.id.toLowerCase().includes(filterLower) ||
    p.nombre.toLowerCase().includes(filterLower) ||
    p.estado.toLowerCase().includes(filterLower) ||
    (p.desc && p.desc.toLowerCase().includes(filterLower))
  );

  for (const p of filtered) {
    const estadoClass = 'tag-' + p.estado.toLowerCase();
    html += `<div class="card" id="post-${p.id}">`;
    html += `<div class="tag ${estadoClass}">${p.estado}</div>`;
    html += `<h5>${p.id} — ${p.nombre}</h5>`;
    if (p.formula) html += `<div class="formula">${p.formula}</div>`;
    if (p.desc) html += `<div class="desc">${p.desc}</div>`;
    html += `<div class="desc" style="margin-top: 12px; font-size: 0.85em;">`;
    html += `Volumen ${p.vol} · Fase ${p.fase}`;
    html += `</div>`;
    html += `</div>`;
  }

  if (filtered.length === 0) {
    html = '<p style="color: var(--fg-2);">No se encontraron postulados.</p>';
  }

  cont.innerHTML = html;
}

// ============================================================
// RENDER: DATOS
// ============================================================
function renderDatos() {
  const cont = document.getElementById('datos-content');
  let html = '<table><thead><tr>';
  html += '<th>Símbolo</th><th>Nombre</th><th>Valor RG</th><th>CODATA 2022</th><th>Error</th><th>Estado</th>';
  html += '</tr></thead><tbody>';

  for (const d of RG_DATA.datos) {
    const estadoClass = 'tag-' + d.estado.toLowerCase();
    let errStr = '—';
    if (d.errorRG !== undefined && d.errorRG !== null) {
      errStr = (d.errorRG * 100).toExponential(2) + '%';
    }
    html += `<tr>`;
    html += `<td class="mono" style="font-size: 1.1em;">${d.simbolo}</td>`;
    html += `<td>${d.nombre}</td>`;
    html += `<td class="num">${d.valor}${d.unidad ? ' ' + d.unidad : ''}</td>`;
    html += `<td class="num">${d.cODATA !== null && d.cODATA !== undefined ? d.cODATA : '—'}</td>`;
    html += `<td class="num">${errStr}</td>`;
    html += `<td><span class="tag ${estadoClass}" style="margin: 0;">${d.estado}</span></td>`;
    html += `</tr>`;
  }

  html += '</tbody></table>';
  cont.innerHTML = html;
}

// ============================================================
// RENDER: BITACORA
// ============================================================
function renderBitacora() {
  const cont = document.getElementById('bitacora-content');
  let html = '';
  for (const b of RG_DATA.bitacora) {
    html += `<div class="card" style="margin-bottom: 16px;">`;
    html += `<div class="tag tag-derivado">${b.sesion}</div>`;
    html += `<h5>${b.titulo}</h5>`;
    html += `<div style="color: var(--fg-2); font-size: 0.85em; margin-bottom: 8px;">${b.fecha}</div>`;
    html += `<div class="desc">${b.contenido}</div>`;
    html += `</div>`;
  }
  cont.innerHTML = html;
}

// ============================================================
// RENDER: REFERENCIAS
// ============================================================
function renderReferencias() {
  const cont = document.getElementById('referencias-content');
  let html = '<h3>Literatura externa</h3><ul style="margin-left: 24px; color: var(--fg-2);">';
  for (const r of RG_DATA.referencias.filter(r => r.tipo === 'externo')) {
    html += `<li style="margin-bottom: 8px;">${r.cita}</li>`;
  }
  html += '</ul><h3>Depósitos propios</h3><ul style="margin-left: 24px; color: var(--fg-2);">';
  for (const r of RG_DATA.referencias.filter(r => r.tipo === 'propio')) {
    html += `<li style="margin-bottom: 8px;">${r.cita}</li>`;
  }
  html += '</ul>';
  cont.innerHTML = html;
}

// ============================================================
// SCROLL TO POSTULADO
// ============================================================
function scrollToPostulado(id) {
  const el = document.getElementById('post-' + id);
  if (el) {
    el.scrollIntoView({ behavior: 'smooth', block: 'center' });
    el.style.transition = 'all 0.5s';
    el.style.background = 'rgba(79,195,247,0.1)';
    setTimeout(() => { el.style.background = ''; }, 2000);
  }
}

// ============================================================
// EXPORT JSON
// ============================================================
function exportJSON() {
  const dataStr = JSON.stringify(RG_DATA, null, 2);
  const blob = new Blob([dataStr], { type: 'application/json' });
  const url = URL.createObjectURL(blob);
  const a = document.createElement('a');
  a.href = url;
  a.download = 'RG_corpus.json';
  a.click();
  URL.revokeObjectURL(url);
}

function exportCSV() {
  let csv = 'ID,Volumen,Fase,Nombre,Estado,Formula\n';
  for (const p of RG_DATA.postulados) {
    const formula = (p.formula || '').replace(/,/g, ';');
    const desc = (p.nombre || '').replace(/,/g, ';');
    csv += `${p.id},${p.vol},${p.fase},"${desc}",${p.estado},"${formula}"\n`;
  }
  const blob = new Blob([csv], { type: 'text/csv' });
  const url = URL.createObjectURL(blob);
  const a = document.createElement('a');
  a.href = url;
  a.download = 'RG_postulados.csv';
  a.click();
  URL.revokeObjectURL(url);
}

// ============================================================
// VIZ 1: SILENCIO ARMÓNICO
// ============================================================
const vizSilencio = document.getElementById('viz-silencio');
const ctxSilencio = vizSilencio.getContext('2d');
let silencioT = 0;
let silencioModes = 3;
let silencioAnimating = true;

function initCanvas(canvas) {
  const dpr = window.devicePixelRatio || 1;
  const rect = canvas.getBoundingClientRect();
  canvas.width = rect.width * dpr;
  canvas.height = rect.height * dpr;
  const ctx = canvas.getContext('2d');
  ctx.scale(dpr, dpr);
  return ctx;
}

function drawSilencio() {
  const canvas = vizSilencio;
  const ctx = ctxSilencio;
  const dpr = window.devicePixelRatio || 1;
  const W = canvas.width / dpr;
  const H = canvas.height / dpr;

  ctx.clearRect(0, 0, W, H);

  // Fondo
  ctx.fillStyle = '#1c2333';
  ctx.fillRect(0, 0, W, H);

  const cx = W / 2;
  const cy = H / 2;

  // Eje horizontal
  ctx.strokeStyle = '#232c42';
  ctx.lineWidth = 1;
  ctx.beginPath();
  ctx.moveTo(0, cy);
  ctx.lineTo(W, cy);
  ctx.stroke();

  // Dibujar la suma de ondas
  const puntos = 400;
  const k0 = 2 * Math.PI / (W * 0.8);

  // Onda resultante
  ctx.strokeStyle = '#4fc3f7';
  ctx.lineWidth = 2;
  ctx.beginPath();
  for (let i = 0; i < puntos; i++) {
    const x = (i / (puntos - 1)) * W;
    let y = 0;
    for (let m = 1; m <= silencioModes; m++) {
      // Cada modo tiene su contra-modo
      const k = m * k0;
      const phase = m * silencioT;
      y += Math.cos(k * (x - cx) + phase) * (1 / m);
      y += Math.cos(-k * (x - cx) - phase) * (1 / m); // contra-onda
    }
    const py = cy + y * 40 / silencioModes;
    if (i === 0) ctx.moveTo(x, py);
    else ctx.lineTo(x, py);
  }
  ctx.stroke();

  // Etiqueta
  ctx.fillStyle = '#a8b3c7';
  ctx.font = '12px monospace';
  ctx.fillText(`Σ modos k=${silencioModes} + contra-modos −k`, 12, 20);
  ctx.fillText(`Energía neta E = 0`, 12, 38);
}

function animateSilencio() {
  if (silencioAnimating) {
    silencioT += 0.02;
    document.getElementById('silencio-t').textContent = silencioT.toFixed(1);
    drawSilencio();
  }
  requestAnimationFrame(animateSilencio);
}

// ============================================================
// VIZ 2: CLÚSTER 19
// ============================================================
const vizCluster = document.getElementById('viz-cluster');
const ctxCluster = vizCluster.getContext('2d');

function hexToPixel(q, r, size) {
  const x = size * (q + r / 2);
  const y = size * r * Math.sqrt(3) / 2;
  return { x, y };
}

function drawCluster() {
  const canvas = vizCluster;
  const ctx = ctxCluster;
  const dpr = window.devicePixelRatio || 1;
  const W = canvas.width / dpr;
  const H = canvas.height / dpr;

  ctx.clearRect(0, 0, W, H);
  ctx.fillStyle = '#1c2333';
  ctx.fillRect(0, 0, W, H);

  const nodos = [[0,0]];
  nodos.push([1,0],[1,-1],[0,-1],[-1,0],[-1,1],[0,1]);
  nodos.push([2,0],[2,-1],[2,-2],[1,-2],[0,-2],[-1,-1],
             [-2,0],[-2,1],[-2,2],[-1,2],[0,2],[1,1]);

  const vecinos = [[1,0],[-1,0],[0,1],[0,-1],[1,-1],[-1,1]];
  const nodoSet = new Set(nodos.map(n => n.join(',')));
  const aristas = [];
  for (let i = 0; i < nodos.length; i++) {
    for (const [dq, dr] of vecinos) {
      const key = (nodos[i][0]+dq) + ',' + (nodos[i][1]+dr);
      if (nodoSet.has(key)) {
        const j = nodos.findIndex(n => n[0] === nodos[i][0]+dq && n[1] === nodos[i][1]+dr);
        if (i < j) aristas.push([i, j]);
      }
    }
  }

  const size = Math.min(W, H) / 6;
  const cx = W / 2;
  const cy = H / 2;

  const posiciones = nodos.map(([q, r]) => {
    const p = hexToPixel(q, r, size);
    return { x: cx + p.x, y: cy + p.y };
  });

  // Aristas
  if (document.getElementById('show-aristas').checked) {
    ctx.strokeStyle = '#4fc3f7';
    ctx.lineWidth = 1.5;
    ctx.globalAlpha = 0.6;
    for (const [i, j] of aristas) {
      ctx.beginPath();
      ctx.moveTo(posiciones[i].x, posiciones[i].y);
      ctx.lineTo(posiciones[j].x, posiciones[j].y);
      ctx.stroke();
    }
    ctx.globalAlpha = 1;
  }

  // Nodos
  const showVecinos = document.getElementById('show-vecinos').checked;
  const vecinosDeCero = new Set();
  if (showVecinos) {
    for (const [i, j] of aristas) {
      if (i === 0) vecinosDeCero.add(j);
      if (j === 0) vecinosDeCero.add(i);
    }
  }

  for (let i = 0; i < nodos.length; i++) {
    const p = posiciones[i];
    let color = '#4fc3f7';
    if (i === 0) color = '#f0c419';
    else if (showVecinos && vecinosDeCero.has(i)) color = '#ef5f5f';
    else if (nodos[i][0] === 0 && nodos[i][1] === 0) color = '#f0c419';

    ctx.beginPath();
    ctx.arc(p.x, p.y, i === 0 ? 10 : 7, 0, 2 * Math.PI);
    ctx.fillStyle = color;
    ctx.fill();
    ctx.strokeStyle = '#0a0e1a';
    ctx.lineWidth = 2;
    ctx.stroke();

    // Etiqueta del índice
    ctx.fillStyle = '#0a0e1a';
    ctx.font = 'bold 9px monospace';
    ctx.textAlign = 'center';
    ctx.textBaseline = 'middle';
    ctx.fillText(i.toString(), p.x, p.y);
  }

  // Info
  ctx.fillStyle = '#a8b3c7';
  ctx.font = '12px monospace';
  ctx.textAlign = 'left';
  ctx.textBaseline = 'top';
  ctx.fillText(`N = ${nodos.length} nodos, E = ${aristas.length} aristas`, 12, 20);
  ctx.fillText(`χ = V − E + F = 19 − 42 + 24 = +1`, 12, 38);
}

// ============================================================
// VIZ 3: ESPECTRO DEL LAPLACIANO
// ============================================================
const vizEspectro = document.getElementById('viz-espectro');
const ctxEspectro = vizEspectro.getContext('2d');

function drawEspectro() {
  const canvas = vizEspectro;
  const ctx = ctxEspectro;
  const dpr = window.devicePixelRatio || 1;
  const W = canvas.width / dpr;
  const H = canvas.height / dpr;

  ctx.clearRect(0, 0, W, H);
  ctx.fillStyle = '#1c2333';
  ctx.fillRect(0, 0, W, H);

  // Espectro del Clúster 19 (valores reales calculados)
  const eigs = [0, 0.85257, 0.85257, 2.0, 2.0, 2.0, 2.0, 2.0, 2.0, 4.0, 4.0, 4.0, 4.0, 5.0, 5.0, 5.0, 6.0, 6.0, 8.19258];

  const margin = { top: 40, right: 30, bottom: 60, left: 70 };
  const plotW = W - margin.left - margin.right;
  const plotH = H - margin.top - margin.bottom;

  const maxEig = 9;
  const barW = plotW / eigs.length * 0.7;
  const stepX = plotW / eigs.length;

  // Eje Y
  ctx.strokeStyle = '#232c42';
  ctx.lineWidth = 1;
  for (let i = 0; i <= 9; i++) {
    const y = margin.top + plotH - (i / maxEig) * plotH;
    ctx.beginPath();
    ctx.moveTo(margin.left, y);
    ctx.lineTo(margin.left + plotW, y);
    ctx.stroke();
    ctx.fillStyle = '#a8b3c7';
    ctx.font = '11px monospace';
    ctx.textAlign = 'right';
    ctx.textBaseline = 'middle';
    ctx.fillText(i.toString(), margin.left - 8, y);
  }

  // Barras
  for (let i = 0; i < eigs.length; i++) {
    const h = (eigs[i] / maxEig) * plotH;
    const x = margin.left + i * stepX + (stepX - barW) / 2;
    const y = margin.top + plotH - h;

    if (eigs[i] === 0) {
      ctx.fillStyle = '#a8b3c7';
    } else if (eigs[i] < 1) {
      ctx.fillStyle = '#f0c419';
    } else {
      ctx.fillStyle = '#4fc3f7';
    }
    ctx.fillRect(x, y, barW, h);
  }

  // Línea de predicción m_e = 1.4324
  const yPred = margin.top + plotH - (1.4324 / maxEig) * plotH;
  ctx.strokeStyle = '#ef5f5f';
  ctx.lineWidth = 2;
  ctx.setLineDash([6, 4]);
  ctx.beginPath();
  ctx.moveTo(margin.left, yPred);
  ctx.lineTo(margin.left + plotW, yPred);
  ctx.stroke();
  ctx.setLineDash([]);

  ctx.fillStyle = '#ef5f5f';
  ctx.font = 'bold 12px monospace';
  ctx.textAlign = 'left';
  ctx.fillText('m_e = 4.5/π ≈ 1.4324', margin.left + 8, yPred - 8);

  // Línea de gap λ₁ = 0.8526
  const yGap = margin.top + plotH - (0.8526 / maxEig) * plotH;
  ctx.strokeStyle = '#6bcf8d';
  ctx.lineWidth = 2;
  ctx.setLineDash([6, 4]);
  ctx.beginPath();
  ctx.moveTo(margin.left, yGap);
  ctx.lineTo(margin.left + plotW, yGap);
  ctx.stroke();
  ctx.setLineDash([]);

  ctx.fillStyle = '#6bcf8d';
  ctx.fillText('λ₁ = 0.8526 (gap)', margin.left + 8, yGap + 18);

  // Etiquetas de eje
  ctx.fillStyle = '#e6ecf5';
  ctx.font = 'bold 12px monospace';
  ctx.textAlign = 'center';
  ctx.fillText('Autovalores λᵢ del Laplaciano L = D − A', W / 2, H - 15);

  ctx.save();
  ctx.translate(20, H / 2);
  ctx.rotate(-Math.PI / 2);
  ctx.fillText('λᵢ', 0, 0);
  ctx.restore();

  // Nota
  ctx.fillStyle = '#a8b3c7';
  ctx.font = '11px monospace';
  ctx.textAlign = 'left';
  ctx.fillText('El gap NO coincide con m_e:', margin.left + 8, 20);
  ctx.fillText('m_e = 4.5/π emerge como postulado separado.', margin.left + 8, 35);
}

// ============================================================
// VIZ 4: CONO DE DIRAC
// ============================================================
const vizDirac = document.getElementById('viz-dirac');
const ctxDirac = vizDirac.getContext('2d');

function drawDirac() {
  const canvas = vizDirac;
  const ctx = ctxDirac;
  const dpr = window.devicePixelRatio || 1;
  const W = canvas.width / dpr;
  const H = canvas.height / dpr;

  ctx.clearRect(0, 0, W, H);
  ctx.fillStyle = '#1c2333';
  ctx.fillRect(0, 0, W, H);

  const cx1 = W * 0.3;
  const cx2 = W * 0.7;
  const cy = H / 2;
  const scale = Math.min(W, H) / 6;

  // Dibujar dos conos
  for (const [cx, label] of [[cx1, 'K'], [cx2, "K'"]]) {
    // Banda positiva
    ctx.beginPath();
    ctx.moveTo(cx, cy);
    const sign = cx === cx1 ? 1 : -1;
    for (let q = -2; q <= 2; q += 0.05) {
      const E = Math.abs(q) * scale;
      const x = cx + q * scale * 0.5;
      const y = cy - E * 0.7 * sign;
      if (q === -2) ctx.moveTo(x, y);
      else ctx.lineTo(x, y);
    }
    ctx.strokeStyle = '#4fc3f7';
    ctx.lineWidth = 2;
    ctx.stroke();

    // Banda negativa
    ctx.beginPath();
    for (let q = -2; q <= 2; q += 0.05) {
      const E = -Math.abs(q) * scale;
      const x = cx + q * scale * 0.5;
      const y = cy - E * 0.7 * sign;
      if (q === -2) ctx.moveTo(x, y);
      else ctx.lineTo(x, y);
    }
    ctx.strokeStyle = '#4fc3f7';
    ctx.lineWidth = 2;
    ctx.stroke();

    // Vértice
    ctx.beginPath();
    ctx.arc(cx, cy, 5, 0, 2 * Math.PI);
    ctx.fillStyle = '#f0c419';
    ctx.fill();
    ctx.strokeStyle = '#0a0e1a';
    ctx.lineWidth = 2;
    ctx.stroke();

    // Etiqueta
    ctx.fillStyle = '#e6ecf5';
    ctx.font = 'bold 16px monospace';
    ctx.textAlign = 'center';
    ctx.fillText(label, cx, cy + 30);
  }

  // Ejes
  ctx.strokeStyle = '#232c42';
  ctx.lineWidth = 1;
  ctx.beginPath();
  ctx.moveTo(0, cy);
  ctx.lineTo(W, cy);
  ctx.stroke();

  // Etiquetas
  ctx.fillStyle = '#a8b3c7';
  ctx.font = '12px monospace';
  ctx.textAlign = 'left';
  ctx.fillText('E(k)', 20, 20);
  ctx.fillText('E = ±v_F |q|', 20, 38);
  ctx.fillText('v_F = 3ta/2ℏ ≈ c', 20, 56);

  ctx.textAlign = 'right';
  ctx.fillText('q →', W - 20, cy - 8);
}

// ============================================================
// VIZ 5: COMPARACIÓN RG vs CODATA
// ============================================================
const vizCompare = document.getElementById('viz-compare');
const ctxCompare = vizCompare.getContext('2d');

function drawCompare() {
  const canvas = vizCompare;
  const ctx = ctxCompare;
  const dpr = window.devicePixelRatio || 1;
  const W = canvas.width / dpr;
  const H = canvas.height / dpr;

  ctx.clearRect(0, 0, W, H);
  ctx.fillStyle = '#1c2333';
  ctx.fillRect(0, 0, W, H);

  const comparaciones = [
    { nombre: 'α⁻¹', rg: 137.0363, exp: 137.035999177, error: 2.2e-6 },
    { nombre: 'm_p/m_e', rg: 1836.15, exp: 1836.152673, error: 1.5e-6 },
    { nombre: 'm_e (u.r.)', rg: 1.4324, exp: 1.43239, error: 0.00015 },
    { nombre: 'H₀ (km/s/Mpc)', rg: 68.74, exp: 67.4, error: 0.02 },
    { nombre: 'sin²θ_W', rg: 0.23122, exp: 0.23122, error: 0.0001 },
    { nombre: 'Ω_DM', rg: 0.054, exp: 0.2647, error: 0.79 },
    { nombre: 'G (×10¹¹)', rg: 1e-30, exp: 6.674, error: 0.999 }
  ];

  const margin = { top: 60, right: 30, bottom: 80, left: 140 };
  const plotW = W - margin.left - margin.right;
  const plotH = H - margin.top - margin.bottom;
  const rowH = plotH / comparaciones.length;

  ctx.fillStyle = '#e6ecf5';
  ctx.font = 'bold 13px monospace';
  ctx.textAlign = 'left';
  ctx.fillText('Error relativo |RG − CODATA| / CODATA', margin.left, 30);

  for (let i = 0; i < comparaciones.length; i++) {
    const c = comparaciones[i];
    const y = margin.top + i * rowH + rowH / 2;

    // Nombre
    ctx.fillStyle = '#a8b3c7';
    ctx.font = '13px monospace';
    ctx.textAlign = 'right';
    ctx.textBaseline = 'middle';
    ctx.fillText(c.nombre, margin.left - 12, y);

    // Barra de fondo
    ctx.fillStyle = '#232c42';
    ctx.fillRect(margin.left, y - 8, plotW, 16);

    // Barra de error (escala logarítmica)
    const logErr = Math.max(-10, Math.log10(Math.max(c.error, 1e-10)));
    const barW = ((logErr + 10) / 10) * plotW;

    // Color según precisión
    let color;
    if (c.error < 1e-4) color = '#6bcf8d';
    else if (c.error < 1e-2) color = '#4fc3f7';
    else if (c.error < 0.5) color = '#f0c419';
    else color = '#ef5f5f';

    ctx.fillStyle = color;
    ctx.fillRect(margin.left, y - 8, barW, 16);

    // Valor del error
    ctx.fillStyle = '#e6ecf5';
    ctx.font = '11px monospace';
    ctx.textAlign = 'left';
    const errStr = c.error < 0.01 ? c.error.toExponential(1) : (c.error * 100).toFixed(1) + '%';
    ctx.fillText(errStr, margin.left + plotW + 8, y);
  }

  // Escala
  ctx.fillStyle = '#a8b3c7';
  ctx.font = '11px monospace';
  ctx.textAlign = 'center';
  ctx.textBaseline = 'top';
  for (let i = 0; i <= 5; i++) {
    const x = margin.left + (i / 5) * plotW;
    ctx.fillText('10⁻¹⁰'.replace('¹⁰', String(10 - 2*i).replace(/-/g, '⁻').split('').join('')),
                  x, margin.top + plotH + 8);
  }
  ctx.fillText('Error relativo (escala logarítmica)', margin.left + plotW / 2, H - 25);
}

// ============================================================
// VIZ 6: ESPECTRO vs TORSIÓN
// ============================================================
const vizTorsion = document.getElementById('viz-torsion');
const ctxTorsion = vizTorsion.getContext('2d');
let psiVal = 0.5;

function drawTorsion() {
  const canvas = vizTorsion;
  const ctx = ctxTorsion;
  const dpr = window.devicePixelRatio || 1;
  const W = canvas.width / dpr;
  const H = canvas.height / dpr;

  ctx.clearRect(0, 0, W, H);
  ctx.fillStyle = '#1c2333';
  ctx.fillRect(0, 0, W, H);

  const margin = { top: 40, right: 30, bottom: 60, left: 70 };
  const plotW = W - margin.left - margin.right;
  const plotH = H - margin.top - margin.bottom;

  // Eje Y
  ctx.strokeStyle = '#232c42';
  ctx.lineWidth = 1;
  const maxE = 4;
  for (let i = 0; i <= 4; i++) {
    const y = margin.top + plotH - (i / maxE) * plotH;
    ctx.beginPath();
    ctx.moveTo(margin.left, y);
    ctx.lineTo(margin.left + plotW, y);
    ctx.stroke();
    ctx.fillStyle = '#a8b3c7';
    ctx.font = '11px monospace';
    ctx.textAlign = 'right';
    ctx.textBaseline = 'middle';
    ctx.fillText(i.toFixed(0), margin.left - 8, y);
  }

  // Eje X
  ctx.beginPath();
  ctx.moveTo(margin.left, margin.top + plotH);
  ctx.lineTo(margin.left + plotW, margin.top + plotH);
  ctx.strokeStyle = '#232c42';
  ctx.stroke();

  // Espectro del Dirac para varios valores de ψ
  // Simplificación: E = ±sqrt(q² + m²) con m = 3 sin(ψ/2)
  const m = 3 * Math.sin(psiVal / 2);

  // Banda positiva
  ctx.beginPath();
  for (let i = 0; i <= 100; i++) {
    const q = -2 + 4 * i / 100;
    const E = Math.sqrt(q * q + m * m);
    const x = margin.left + (q + 2) / 4 * plotW;
    const y = margin.top + plotH - (E / maxE) * plotH;
    if (i === 0) ctx.moveTo(x, y);
    else ctx.lineTo(x, y);
  }
  ctx.strokeStyle = '#4fc3f7';
  ctx.lineWidth = 2.5;
  ctx.stroke();

  // Banda negativa
  ctx.beginPath();
  for (let i = 0; i <= 100; i++) {
    const q = -2 + 4 * i / 100;
    const E = -Math.sqrt(q * q + m * m);
    const x = margin.left + (q + 2) / 4 * plotW;
    const y = margin.top + plotH - (E / maxE) * plotH;
    if (i === 0) ctx.moveTo(x, y);
    else ctx.lineTo(x, y);
  }
  ctx.strokeStyle = '#4fc3f7';
  ctx.lineWidth = 2.5;
  ctx.stroke();

  // Gap
  const yGap = margin.top + plotH - (m / maxE) * plotH;
  ctx.strokeStyle = '#f0c419';
  ctx.lineWidth = 2;
  ctx.setLineDash([6, 4]);
  ctx.beginPath();
  ctx.moveTo(margin.left + plotW / 2, yGap);
  ctx.lineTo(margin.left + plotW / 2, margin.top + plotH + (m / maxE) * plotH);
  ctx.stroke();
  ctx.setLineDash([]);

  // Etiquetas
  ctx.fillStyle = '#a8b3c7';
  ctx.font = '12px monospace';
  ctx.textAlign = 'left';
  ctx.textBaseline = 'top';
  ctx.fillText(`m = 3·sin(ψ₀/2) = ${m.toFixed(3)}`, 12, 20);
  ctx.fillText(`ψ₀ = ${(psiVal).toFixed(3)} rad`, 12, 38);

  // Notas
  ctx.fillStyle = '#e6ecf5';
  ctx.font = 'bold 12px monospace';
  ctx.textAlign = 'center';
  ctx.fillText('Espectro del operador de Dirac', W / 2, H - 25);
  ctx.textAlign = 'left';
  ctx.fillStyle = '#a8b3c7';
  ctx.font = '11px monospace';
  ctx.fillText('A ψ₀ = 0, doublers presentes. A ψ₀ ≠ 0, gap abre, un modo cero sobrevive.', margin.left, H - 8);
}

// ============================================================
// INIT
// ============================================================
function init() {
  // Renderizar contenido
  renderIndice();
  renderPostulados();
  renderDatos();
  renderBitacora();
  renderReferencias();

  // Search
  document.getElementById('search').addEventListener('input', (e) => {
    renderPostulados(e.target.value);
  });

  // Canvas setup
  initCanvas(vizSilencio);
  initCanvas(vizCluster);
  initCanvas(vizEspectro);
  initCanvas(vizDirac);
  initCanvas(vizCompare);
  initCanvas(vizTorsion);

  // Controls
  document.getElementById('silencio-time').addEventListener('input', (e) => {
    silencioT = parseFloat(e.target.value) / 10;
    document.getElementById('silencio-t').textContent = silencioT.toFixed(1);
    drawSilencio();
  });

  document.getElementById('silencio-modes-slider').addEventListener('input', (e) => {
    silencioModes = parseInt(e.target.value);
    document.getElementById('silencio-modes').textContent = silencioModes;
    drawSilencio();
  });

  document.getElementById('show-aristas').addEventListener('change', drawCluster);
  document.getElementById('show-vecinos').addEventListener('change', drawCluster);

  document.getElementById('psi-slider').addEventListener('input', (e) => {
    psiVal = parseFloat(e.target.value) / 100;
    document.getElementById('psi-val').textContent = psiVal.toFixed(3);
    drawTorsion();
  });

  // Draw all
  drawSilencio();
  drawCluster();
  drawEspectro();
  drawDirac();
  drawCompare();
  drawTorsion();

  // Animate
  animateSilencio();

  // Resize
  window.addEventListener('resize', () => {
    initCanvas(vizCluster);
    initCanvas(vizEspectro);
    initCanvas(vizDirac);
    initCanvas(vizCompare);
    initCanvas(vizTorsion);
    drawCluster();
    drawEspectro();
    drawDirac();
    drawCompare();
    drawTorsion();
  });

  // Nav active
  const navLinks = document.querySelectorAll('nav a');
  window.addEventListener('scroll', () => {
    let current = '';
    document.querySelectorAll('section').forEach(sec => {
      const top = sec.offsetTop - 100;
      if (window.scrollY >= top) current = sec.id;
    });
    navLinks.forEach(link => {
      link.classList.toggle('active', link.getAttribute('href') === '#' + current);
    });
  });
}

document.addEventListener('DOMContentLoaded', init);
</script>
</body>
</html>

In [ ]:
sample_id = next(iter(clasificacion))
sample_entry = clasificacion[sample_id]
print(f"Keys in a sample entry of 'clasificacion': {list(sample_entry.keys())}")
display(sample_entry)

In [ ]:

# === PARTE II: MONTE CARLO DE CREACIÓN ===

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from collections import defaultdict
import networkx as nx

np.random.seed(192657)

# ------------------------------------------------------------
# MODELO DE CRECIMIENTO POR TENSIÓN DE FASE
# ------------------------------------------------------------

def tension(psi_i, psi_j):
    """Energía de enlace entre dos nodos."""
    return 1 - np.cos(psi_i - psi_j)

def generar_forma(n_nodos=19, seed=None):
    """
    Genera un clúster RG por crecimiento estocástico.
    Cada nuevo nodo se añade donde minimiza la tensión total.
    """
    rng = np.random.default_rng(seed)

    # Empezar con un nodo central
    nodos = [(0, 0)]
    fases = {0: 0.0}  # nodo → fase

    # Direcciones hexagonales
    vecinos = [(1,0), (-1,0), (0,1), (0,-1), (1,-1), (-1,1)]

    while len(nodos) < n_nodos:
        # Candidatos: sitios vecinos a la frontera
        candidatos = set()
        for (q, r) in nodos:
            for (dq, dr) in vecinos:
                p = (q+dq, r+dr)
                if p not in nodos:
                    candidatos.add(p)
        candidatos = list(candidatos)

        # Calcular tensión para cada candidato con todos sus vecinos
        mejor = None
        mejor_energia = np.inf
        for cand in candidatos:
            # Vecinos existentes
            vecinos_existentes = []
            for (dq, dr) in vecinos:
                p = (cand[0]+dq, cand[1]+dr)
                if p in nodos:
                    idx = nodos.index(p)
                    vecinos_existentes.append(fases[idx])
            if not vecinos_existentes:
                continue

            # Fase óptima: promedio de vecinos
            fase_opt = np.mean(vecinos_existentes)
            # Energía: suma de tensiones
            E = sum(tension(fase_opt, fv) for fv in vecinos_existentes)
            # Añadir algo de ruido estocástico
            E += rng.normal(0, 0.05)

            if E < mejor_energia:
                mejor_energia = E
                mejor = (cand, fase_opt)

        # Añadir el mejor candidato
        nodos.append(mejor[0])
        fases[len(nodos)-1] = mejor[1]

    return nodos, fases

def construir_grafo(nodos):
    """Construye el grafo de la red."""
    G = nx.Graph()
    G.add_nodes_from(range(len(nodos)))
    vecinos = [(1,0), (-1,0), (0,1), (0,-1), (1,-1), (-1,1)]
    nodo_set = set(nodos)
    for i, (q, r) in enumerate(nodos):
        for (dq, dr) in vecinos:
            if (q+dq, r+dr) in nodo_set:
                j = nodos.index((q+dq, r+dr))
                if i < j:
                    G.add_edge(i, j)
    return G

def invariantes(G, nodos):
    """Calcula invariantes topológicos y espectrales."""
    N = G.number_of_nodes()
    E = G.number_of_edges()

    # F = número de triángulos (caras)
    triangulos = sum(nx.triangles(G).values()) // 3

    chi = N - E + triangulos

    # Laplaciano
    L = nx.laplacian_matrix(G).toarray().astype(float)
    eigs = np.sort(np.linalg.eigvalsh(L))
    gap = eigs[1] if N > 1 else 0

    return {
        'N': N, 'E': E, 'F': triangulos,
        'chi': chi, 'gap': gap, 'lambda_max': eigs[-1]
    }

def simular_crecimiento(n_formas=1000, n_nodos=19):
    """Simula n_formas independientes."""
    resultados = []
    for k in range(n_formas):
        nodos, fases = generar_forma(n_nodos=n_nodos, seed=k*192657)
        G = construir_grafo(nodos)
        inv = invariantes(G, nodos)
        resultados.append(inv)
    return resultados

print("=" * 70)
print("MONTE CARLO DE FORMAS DE CREACIÓN — CLÚSTER 19")
print("=" * 70)

resultados = simular_crecimiento(n_formas=500, n_nodos=19)

# Estadísticas
N_vals = [r['N'] for r in resultados]
E_vals = [r['E'] for r in resultados]
chi_vals = [r['chi'] for r in resultados]
gap_vals = [r['gap'] for r in resultados]

print(f"\nN = {np.mean(N_vals):.1f} ± {np.std(N_vals):.2f}")
print(f"E = {np.mean(E_vals):.1f} ± {np.std(E_vals):.2f}")
print(f"χ = {np.mean(chi_vals):.3f} ± {np.std(chi_vals):.3f}")
print(f"gap = {np.mean(gap_vals):.4f} ± {np.std(gap_vals):.4f}")

# Frecuencia de χ = 1 (topología protegida)
freq_chi1 = sum(1 for c in chi_vals if c == 1) / len(chi_vals)
print(f"\nFracción con χ = 1: {freq_chi1:.2%}")

# ------------------------------------------------------------
# VISUALIZACIÓN
# ------------------------------------------------------------
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. Forma típica
ax1 = fig.add_subplot(gs[0, 0])
nodos, fases = generar_forma(19, seed=42)
x = [n[0] for n in nodos]
y = [n[1] for n in nodos]
ax1.scatter(x, y, c=list(fases.values()), cmap='twilight', s=200,
            edgecolors='black', linewidths=0.5)
for i, (q, r) in enumerate(nodos):
    ax1.annotate(f"{fases[i]:.2f}", (q, r), fontsize=6, ha='center')
ax1.set_title('Forma típica (N=19)')
ax1.set_aspect('equal')
ax1.grid(alpha=0.3)

# 2. Distribución de χ
ax2 = fig.add_subplot(gs[0, 1])
ax2.hist(chi_vals, bins=15, color='steelblue', edgecolor='black')
ax2.axvline(1, color='red', linestyle='--', label='χ=1 (protegido)')
ax2.set_xlabel('χ = N - E + F')
ax2.set_ylabel('Frecuencia')
ax2.set_title('Distribución de la característica de Euler')
ax2.legend()

# 3. Distribución del gap
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(gap_vals, bins=30, color='green', alpha=0.7, edgecolor='black')
ax3.axvline(0.8526, color='red', linestyle='--', label='Gap Clúster 19')
ax3.set_xlabel('Gap espectral')
ax3.set_ylabel('Frecuencia')
ax3.set_title('Distribución del gap del Laplaciano')
ax3.legend()

# 4. Relación E vs N
ax4 = fig.add_subplot(gs[1, 0])
ax4.scatter(N_vals, E_vals, alpha=0.5, s=10)
ax4.set_xlabel('N')
ax4.set_ylabel('E')
ax4.set_title('Relación N-E')
ax4.grid(alpha=0.3)

# 5. Relación gap vs E
ax5 = fig.add_subplot(gs[1, 1])
ax5.scatter(E_vals, gap_vals, alpha=0.5, s=10, c=chi_vals, cmap='viridis')
ax5.set_xlabel('E')
ax5.set_ylabel('gap')
ax5.set_title('Relación E-gap (color: χ)')
plt.colorbar(ax5.collections[0], ax=ax5, label='χ')
ax5.grid(alpha=0.3)

# 6. Densidad de estados
ax6 = fig.add_subplot(gs[1, 2])
ax6.hist(gap_vals, bins=50, density=True, color='purple', alpha=0.7)
ax6.set_xlabel('gap')
ax6.set_ylabel('Densidad')
ax6.set_title('Densidad de estados del gap')
ax6.grid(alpha=0.3)

plt.suptitle('Monte Carlo de Formas de Creación — RG', fontsize=14, fontweight='bold')
plt.savefig('mc_creacion_rg.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualización guardada: mc_creacion_rg.png")

In [ ]:

# === PARTE III: CHIP CUÁNTICO RG ===

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ------------------------------------------------------------
# MODELO DE CHIP CUÁNTICO RG
# ------------------------------------------------------------

class ChipRG:
    """
    Chip cuántico basado en red hexagonal RG.

    Componentes:
    - Qubits físicos: nodos de la red
    - Puertas: acoplamientos de torsión
    - Corrección de errores: topológica (χ = 1)
    """

    def __init__(self, n_nodos=19, torsion=0.496, t=1.0):
        self.n_nodos = n_nodos
        self.torsion = torsion
        self.t = t
        self.nodos = None
        self.D = None
        self.edges = None

    def construir_red(self):
        """Construye la red hexagonal centrada."""
        nodos = [(0,0)]
        nodos += [(1,0),(1,-1),(0,-1),(-1,0),(-1,1),(0,1)]
        nodos += [(2,0),(2,-1),(2,-2),(1,-2),(0,-2),(-1,-1),
                  (-2,0),(-2,1),(-2,2),(-1,2),(0,2),(1,1)]
        self.nodos = nodos[:self.n_nodos]

        vecinos = [(1,0),(-1,0),(0,1),(0,-1),(1,-1),(-1,1)]
        nodo_set = set(self.nodos)
        self.edges = []
        for i, (q, r) in enumerate(self.nodos):
            for dq, dr in vecinos:
                if (q+dq, r+dr) in nodo_set:
                    j = self.nodos.index((q+dq, r+dr))
                    if i < j:
                        self.edges.append((i, j))

        return self

    def construir_dirac(self):
        """Construye el operador de Dirac retorcido."""
        N = len(self.nodos)
        subred = [('A' if (q+r)%2==0 else 'B') for (q,r) in self.nodos]
        torsion_i = np.array([+self.torsion/2 if s=='A' else -self.torsion/2
                              for s in subred])

        D = np.zeros((2*N, 2*N), dtype=complex)
        for (i, j) in self.edges:
            U_ij = np.exp(1j*(torsion_i[j] - torsion_i[i]))
            T = self.t * np.array([[0, U_ij], [-np.conj(U_ij), 0]])
            D[2*i:2*i+2, 2*j:2*j+2] += T
            D[2*j:2*j+2, 2*i:2*i+2] += T.conj().T

        self.D = D
        return self

    def espectro(self):
        """Calcula el espectro del chip."""
        eigs = np.linalg.eigvalsh(self.D)
        return eigs

    def modos_cero(self, tol=1e-6):
        """Cuenta modos cero (protegidos)."""
        eigs = self.espectro()
        return np.sum(np.abs(eigs) < tol)

    def tasa_error(self, T_K=0.01, k_B=1.38e-23, hbar=1.054e-34):
        """
        Tasa de error térmica.
        p_err ~ exp(-m_eff / (k_B T))
        """
        # m_eff = z * sin(torsion/2) * t
        z = 3
        m_eff = z * np.sin(self.torsion/2) * self.t
        # En unidades de red
        p_err = np.exp(-m_eff / T_K)
        return p_err, m_eff

    def coherencia(self, t_total=100):
        """
        Simula la evolución temporal de la coherencia.
        C(t) = |<Ψ(0)|Ψ(t)>|² con decoherencia
        """
        # Evolución unitaria + decoherencia
        eigs = self.espectro()
        # Fase acumulada
        phases = np.exp(-1j * eigs * t_total)
        # Decoherencia exponencial con tasa η
        eta = np.pi / 57
        decay = np.exp(-eta * t_total * np.abs(eigs))
        coherencia = np.abs(phases * decay)**2
        return coherencia

# ------------------------------------------------------------
# SIMULACIÓN
# ------------------------------------------------------------

print("=" * 70)
print("SIMULACIÓN DE CHIP CUÁNTICO RG")
print("=" * 70)

# Chip base
chip = ChipRG(n_nodos=19, torsion=0.496, t=1.0)
chip.construir_red().construir_dirac()

print(f"\nChip RG base:")
print(f"  Nodos: {chip.n_nodos}")
print(f"  Aristas: {len(chip.edges)}")
print(f"  Torsión: {chip.torsion}")
print(f"  Modos cero: {chip.modos_cero()}")

eigs = chip.espectro()
print(f"  Espectro: E_min = {eigs[0]:.4f}, E_max = {eigs[-1]:.4f}")
print(f"  Gap: {eigs[np.argmin(np.abs(eigs[np.abs(eigs) > 1e-6]))]:.4f}")

# Tasa de error
p_err, m_eff = chip.tasa_error(T_K=0.01)
print(f"\n  Masa efectiva: {m_eff:.4f}")
print(f"  Tasa de error a T=0.01: {p_err:.4e}")

# Barrido de torsión
print("\nBarrido de torsión:")
print(f"{'torsion':<10} {'gap':<10} {'modos 0':<10} {'p_err':<12}")
print("-" * 45)

torsiones = np.linspace(0.01, 1.0, 20)
results = []
for tor in torsiones:
    chip_t = ChipRG(n_nodos=19, torsion=tor, t=1.0)
    chip_t.construir_red().construir_dirac()
    n_zero = chip_t.modos_cero()
    eigs_t = chip_t.espectro()
    gap_t = eigs_t[np.argmin(np.abs(eigs_t[np.abs(eigs_t) > 1e-6]))]
    p_err_t, _ = chip_t.tasa_error(T_K=0.01)
    results.append((tor, gap_t, n_zero, p_err_t))
    if tor in [0.01, 0.25, 0.5, 0.75, 1.0] or abs(tor - 0.496) < 0.05:
        print(f"{tor:<10.3f} {gap_t:<10.4f} {n_zero:<10d} {p_err_t:<12.4e}")

# ------------------------------------------------------------
# VISUALIZACIÓN
# ------------------------------------------------------------
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# 1. Red del chip
ax1 = fig.add_subplot(gs[0, 0])
for (i, j) in chip.edges:
    ax1.plot([chip.nodos[i][0], chip.nodos[j][0]],
             [chip.nodos[i][1], chip.nodos[j][1]], 'b-', alpha=0.4)
for i, (q, r) in enumerate(chip.nodos):
    color = 'red' if (q+r)%2==0 else 'blue'
    ax1.scatter(q, r, c=color, s=100, zorder=5, edgecolors='black')
ax1.set_title('Red del chip RG (N=19)')
ax1.set_aspect('equal')
ax1.grid(alpha=0.3)

# 2. Espectro
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(eigs, 'o-', color='purple', markersize=4)
ax2.axhline(0, color='red', linestyle='--', lw=1)
ax2.set_xlabel('Índice')
ax2.set_ylabel('E')
ax2.set_title('Espectro del chip')
ax2.grid(alpha=0.3)

# 3. Barrido de torsión: gap
tors_arr = [r[0] for r in results]
gap_arr = [r[1] for r in results]
zero_arr = [r[2] for r in results]

ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(tors_arr, gap_arr, 'g-', lw=2, marker='o')
ax3.axvline(0.496, color='red', linestyle='--', label='torsión óptima')
ax3.set_xlabel('Torsión ψ₀')
ax3.set_ylabel('Gap espectral')
ax3.set_title('Gap vs Torsión')
ax3.legend()
ax3.grid(alpha=0.3)

# 4. Barrido de torsión: modos cero
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(tors_arr, zero_arr, 'b-', lw=2, marker='s')
ax4.set_xlabel('Torsión ψ₀')
ax4.set_ylabel('Modos cero')
ax4.set_title('Modos cero vs Torsión')
ax4.grid(alpha=0.3)

# 5. Barrido de torsión: tasa de error (escala log)
p_err_arr = [r[3] for r in results]
ax5 = fig.add_subplot(gs[1, 1])
ax5.semilogy(tors_arr, p_err_arr, 'r-', lw=2, marker='^')
ax5.set_xlabel('Torsión ψ₀')
ax5.set_ylabel('Tasa de error')
ax5.set_title('Tasa de error vs Torsión')
ax5.grid(alpha=0.3, which='both')

# 6. Coherencia temporal
ax6 = fig.add_subplot(gs[1, 2])
t_vals = np.linspace(0, 100, 200)
coherencias = []
for t in t_vals:
    c = chip.coherencia(t_total=t)
    coherencias.append(np.mean(c))
ax6.plot(t_vals, coherencias, 'purple', lw=2)
ax6.set_xlabel('Tiempo')
ax6.set_ylabel('Coherencia promedio')
ax6.set_title('Decaimiento de coherencia')
ax6.grid(alpha=0.3)

plt.suptitle('Chip Cuántico RG — Simulación Monte Carlo', fontsize=14, fontweight='bold')
plt.savefig('chip_cuantico_rg.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Visualización guardada: chip_cuantico_rg.png")

In [ ]:

"""
SIMULACIÓN MONTE CARLO — GEOMETRÍA RELACIONAL
Emergencia de estructuras desde el Silencio Armónico
Autor: Edward P. López
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import entropy
from scipy.ndimage import gaussian_filter

# =====================================================================
# CONFIGURACIÓN
# =====================================================================
N_NODOS = 57          # Hard-Lock 57 (protón)
D_DEUDA = 1.5         # Deuda de Información
J_ACOPLAMIENTO = 1.0  # Acoplamiento base
N_MUESTRAS = 100000   # Muestras Monte Carlo
N_PASOS = 2000        # Pasos temporales
SEMILLA = 192657      # Reproducibilidad

np.random.seed(SEMILLA)


# =====================================================================
# 1. ESTADO INICIAL — SILENCIO ARMÓNICO
# =====================================================================
def inicializar_silencio(N, M):
    """
    El Silencio Armónico es la distribución uniforme de fases.
    Ψ₀(θ₁,...,θₙ) = 1/√((2π)ᴺ)
    """
    fases = np.random.uniform(0, 2*np.pi, (M, N))
    return fases


def verificar_axiomas(fases):
    """
    Verifica los axiomas fundamentales:
    - ⟨0|0⟩ = 1
    - Máxima entropía S = N ln(2π)
    - Simetría primordial
    """
    norma = np.mean(np.abs(fases)**2)
    S_max = fases.shape[1] * np.log(2*np.pi)

    hist, _ = np.histogram(fases.flatten(), bins=100, density=True)
    hist = hist[hist > 0]
    S_actual = entropy(hist)

    print(f"[AXIOMAS]")
    print(f"  ⟨0|0⟩ = {norma:.8f}  (teórico: 1.0)")
    print(f"  Entropía = {S_actual:.4f}  (máx teórica: {S_max:.4f})")
    print(f"  Desviación uniforme = {np.std(fases.flatten()) / (2*np.pi):.6f}")

    return norma, S_actual


# =====================================================================
# 2. AUTO-COMPARACIÓN
# =====================================================================
def auto_comparacion(fases, theta_0=0.0):
    """
    ⟨0|P̂(θ)|0⟩ / cos(θ) = 1
    Genera el Entero Primordial.
    """
    theta_global = np.mean(fases, axis=1)
    expectacion = np.mean(np.cos(theta_0 * theta_global))
    primordial = expectacion / np.cos(theta_0)

    print(f"\n[AUTO-COMPARACIÓN]")
    print(f"  ⟨0|P̂({theta_0})|0⟩ = {expectacion:.8f}")
    print(f"  Ecuación Primordial = {primordial:.8f}")

    return expectacion, primordial


# =====================================================================
# 3. EVOLUCIÓN LINDELADIANA
# =====================================================================
def evolucionar_lindblad(fases_iniciales, N, D, J, pasos):
    """
    dρ/dt = -(i/ℏ)[Ĥ, ρ] + γ Σᵢ L[Êᵢ]ρ
    con γ = D/N
    """
    gamma = D / N
    fases_t = np.zeros((pasos, N))
    fases_t[0] = fases_iniciales

    entropias = []
    deudas = []
    orden_hex = []

    for t in range(1, pasos):
        # Ruido térmico
        ruido = np.random.normal(0, np.sqrt(gamma), N)

        # Interacción entre nodos vecinos
        interaccion = J * np.sin(fases_t[t-1] - np.roll(fases_t[t-1], 1))

        # Término de Deuda
        deuda_term = D * np.random.uniform(-0.1, 0.1, N)

        # Actualización
        fases_t[t] = (fases_t[t-1] + ruido + interaccion + deuda_term) % (2*np.pi)

        # Métricas
        hist, _ = np.histogram(fases_t[t], bins=20, density=True)
        hist = hist[hist > 0]
        entropias.append(entropy(hist) if len(hist) > 0 else 0)

        # Deuda promedio
        deudas.append(D * np.mean(np.sin(fases_t[t])))

        # Orden hexagonal: O₆ = (1/N) Σ cos(6θᵢ)
        orden_hex.append(np.mean(np.cos(6 * fases_t[t])))

    return fases_t, np.array(entropias), np.array(deudas), np.array(orden_hex)


# =====================================================================
# 4. DETECCIÓN DE ESTRUCTURAS EMERGENTES
# =====================================================================
def detectar_cluster_19(fases):
    """
    Detecta si el sistema forma un Clúster 19:
    - 1 nodo central
    - 6 nodos en capa 1
    - 12 nodos en capa 2
    """
    # Fase media
    fase_media = np.mean(fases)

    # Nodos con fase cercana a la media (núcleo)
    distancias = np.abs(np.angle(np.exp(1j * (fases - fase_media))))
    nucleo = np.sum(distancias < np.pi/6)

    # Orden hexagonal
    orden = np.mean(np.cos(6 * fases))

    return {
        'nodos_nucleo': nucleo,
        'orden_hexagonal': orden,
        'estructura_detectada': nucleo >= 19 and orden > 0.5
    }


def detectar_hardlock_57(fases):
    """
    Detecta si el sistema forma un Hard-Lock 57:
    - 3 clústeres de 19 nodos entrelazados
    """
    N = len(fases)
    if N < 57:
        return {'error': f'Se necesitan al menos 57 nodos, hay {N}'}

    # Subdividir en 3 bloques
    bloque_size = N // 3
    bloques = [fases[i*bloque_size:(i+1)*bloque_size] for i in range(3)]

    ordenes = [np.mean(np.cos(6 * b)) for b in bloques]
    orden_global = np.mean(ordenes)

    return {
        'orden_bloques': ordenes,
        'orden_global': orden_global,
        'hardlock_detectado': np.all(np.array(ordenes) > 0.4)
    }


# =====================================================================
# 5. VALIDACIÓN DE LOS SEIS ROLES
# =====================================================================
def validar_seis_roles():
    """
    Deriva los seis roles desde los principios de Fresnel,
    Conservación y Homotecia.
    """
    A, B = 1.0, 1.0
    T = A + B
    a = A / T
    b = B / T
    c = T - (a + b)
    C = A / a

    # Ecuación Maestra (suma)
    eq_suma_izq = A + B + C
    eq_suma_der = 2 * (a + b + c)

    # Ecuación Maestra (producto)
    eq_prod_izq = A * B * C
    eq_prod_der = 2 * a * b * c

    # Deuda de Información
    deuda = eq_prod_izq - eq_prod_der

    # Contracción
    contraccion = (A/a) / (B/b) / (C/c) * 2

    print(f"\n[SEIS ROLES]")
    print(f"  A={A}, B={B}, C={C:.1f}")
    print(f"  a={a}, b={b}, c={c}")
    print(f"  Ecuación Maestra (suma): {eq_suma_izq:.1f} = {eq_suma_der:.1f}")
    print(f"  Ecuación Maestra (producto): {eq_prod_izq:.1f} ≠ {eq_prod_der:.1f}")
    print(f"  Deuda 𝔇 = {deuda:.6f}")
    print(f"  Contracción: {contraccion:.6f}")

    return {'A': A, 'B': B, 'C': C, 'a': a, 'b': b, 'c': c, 'deuda': deuda}


# =====================================================================
# 6. CÁLCULO DE CONSTANTES FÍSICAS
# =====================================================================
def calcular_constantes(D=1.5, eta=None):
    """
    Deriva las constantes fundamentales desde la geometría de la red.
    """
    if eta is None:
        eta = np.pi / 57

    m_e = 3 * D / np.pi
    alpha_inv = 4 * np.pi**3 + np.pi**2 + np.pi
    mp_me = (4*np.pi/3) * alpha_inv * (1 + np.pi/57)
    Omega_DM = (np.pi - eta) / 57
    H_0 = (14.16 * 3) * (18/17)

    print(f"\n[CONSTANTES DERIVADAS]")
    print(f"  m_e = {m_e:.6f} u.r.  (exp: 0.511 MeV/c²)")
    print(f"  α⁻¹ = {alpha_inv:.8f}  (exp: 137.035999)")
    print(f"  m_p/m_e = {mp_me:.4f}  (exp: 1836.1527)")
    print(f"  Ω_DM = {Omega_DM:.6f}  (exp: 0.264)")
    print(f"  H₀ = {H_0:.4f} km/s/Mpc  (exp: 67.4-74.0)")

    return {
        'm_e': m_e,
        'alpha_inv': alpha_inv,
        'mp_me': mp_me,
        'Omega_DM': Omega_DM,
        'H_0': H_0
    }


# =====================================================================
# 7. VISUALIZACIÓN
# =====================================================================
def visualizar_resultados(fases_t, entropias, deudas, orden_hex, constantes):
    """
    Genera visualizaciones completas de la simulación.
    """
    fig, axes = plt.subplots(3, 3, figsize=(18, 14))
    fig.suptitle('GEOMETRÍA RELACIONAL — SIMULACIÓN MONTE CARLO',
                 fontsize=16, fontweight='bold')

    # (a) Distribución inicial
    ax = axes[0, 0]
    ax.hist(fases_t[0], bins=30, density=True, alpha=0.7, color='royalblue')
    ax.axhline(1/(2*np.pi), color='red', linestyle='--', label='Uniforme')
    ax.set_xlabel('Fase θ')
    ax.set_ylabel('Densidad')
    ax.set_title('(a) Silencio Inicial')
    ax.legend()
    ax.grid(alpha=0.3)

    # (b) Evolución temporal (heatmap)
    ax = axes[0, 1]
    im = ax.imshow(fases_t.T, aspect='auto', cmap='viridis')
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Nodo')
    ax.set_title('(b) Evolución de Fases')
    plt.colorbar(im, ax=ax, label='θ (rad)')

    # (c) Entropía
    ax = axes[0, 2]
    ax.plot(entropias, 'b-', linewidth=1.5)
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Entropía')
    ax.set_title('(c) Entropía → Flecha del Tiempo')
    ax.grid(alpha=0.3)

    # (d) Deuda acumulada
    ax = axes[1, 0]
    ax.plot(deudas, 'r-', linewidth=1.5)
    ax.axhline(1.5, color='k', linestyle='--', label='𝔇 = 1.5')
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Deuda')
    ax.set_title('(d) Deuda de Información')
    ax.legend()
    ax.grid(alpha=0.3)

    # (e) Orden hexagonal
    ax = axes[1, 1]
    ax.plot(orden_hex, 'g-', linewidth=1.5)
    ax.axhline(0.5, color='r', linestyle='--', label='Umbral')
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('O₆')
    ax.set_title('(e) Orden Hexagonal Emergente')
    ax.legend()
    ax.grid(alpha=0.3)

    # (f) Distribución final
    ax = axes[1, 2]
    ax.hist(fases_t[-1], bins=30, density=True, alpha=0.7, color='darkgreen')
    ax.axhline(1/(2*np.pi), color='red', linestyle='--', label='Uniforme')
    ax.set_xlabel('Fase θ')
    ax.set_ylabel('Densidad')
    ax.set_title('(f) Estado Final')
    ax.legend()
    ax.grid(alpha=0.3)

    # (g) Constantes derivadas
    ax = axes[2, 0]
    nombres = ['m_e', 'α⁻¹/100', 'm_p/m_e/100', 'Ω_DM']
    valores = [constantes['m_e'], constantes['alpha_inv']/100,
               constantes['mp_me']/100, constantes['Omega_DM']]
    colores = ['royalblue', 'crimson', 'gold', 'purple']
    bars = ax.bar(nombres, valores, color=colores, edgecolor='black')
    ax.set_ylabel('Valor (normalizado)')
    ax.set_title('(g) Constantes Derivadas')
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
    ax.grid(alpha=0.3, axis='y')

    # (h) Detección de Clúster 19
    ax = axes[2, 1]
    cluster = detectar_cluster_19(fases_t[-1])
    ax.axis('off')
    texto = (f"DETECCIÓN DE ESTRUCTURA\n\n"
             f"Nodos en núcleo: {cluster['nodos_nucleo']}\n"
             f"Orden hexagonal: {cluster['orden_hexagonal']:.4f}\n"
             f"Estructura detectada: {cluster['estructura_detectada']}\n\n"
             f"Clúster 19: N = 1+6+12 = 19\n"
             f"Hard-Lock 57: 3×19 = 57")
    ax.text(0.1, 0.5, texto, fontsize=11, family='monospace',
            verticalalignment='center')

    # (i) Predicciones RG vs experimento
    ax = axes[2, 2]
    etiquetas = ['m_e', 'α⁻¹', 'm_p/m_e', 'Ω_DM']
    valores_rg = [constantes['m_e']/0.511, constantes['alpha_inv']/137.036,
                  constantes['mp_me']/1836.15, constantes['Omega_DM']/0.264]
    x = np.arange(len(etiquetas))
    ax.bar(x - 0.2, valores_rg, 0.4, label='RG/Exp', color='steelblue')
    ax.axhline(1.0, color='red', linestyle='--', label='Acuerdo perfecto')
    ax.set_xticks(x)
    ax.set_xticklabels(etiquetas, rotation=45)
    ax.set_ylabel('Ratio RG/Exp')
    ax.set_title('(i) Acuerdo con el Experimento')
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('rg_simulacion_completa.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("\n[VISUALIZACIÓN] Guardada: rg_simulacion_completa.png")


# =====================================================================
# 8. EJECUCIÓN PRINCIPAL
# =====================================================================
def main():
    print("="*70)
    print(" SIMULACIÓN MONTE CARLO — GEOMETRÍA RELACIONAL")
    print("="*70)

    # Paso 1: Inicializar Silencio
    print("\n[1/6] Inicializando Silencio Armónico...")
    fases = inicializar_silencio(N_NODOS, N_MUESTRAS)
    verificar_axiomas(fases)

    # Paso 2: Auto-comparación
    print("\n[2/6] Ejecutando Auto-Comparación...")
    auto_comparacion(fases)

    # Paso 3: Evolución
    print("\n[3/6] Evolucionando con Lindbladiano...")
    fases_t, entropias, deudas, orden_hex = evolucionar_lindblad(
        fases[0], N_NODOS, D_DEUDA, J_ACOPLAMIENTO, N_PASOS
    )

    # Paso 4: Validar roles
    print("\n[4/6] Validando Seis Roles...")
    roles = validar_seis_roles()

    # Paso 5: Constantes
    print("\n[5/6] Calculando Constantes Derivadas...")
    constantes = calcular_constantes()

    # Paso 6: Visualizar
    print("\n[6/6] Generando Visualizaciones...")
    visualizar_resultados(fases_t, entropias, deudas, orden_hex, constantes)

    # Detección de estructuras
    print("\n[DETECCIÓN DE ESTRUCTURAS]")
    cluster = detectar_cluster_19(fases_t[-1])
    print(f"  Clúster 19: {cluster}")

    if N_NODOS >= 57:
        hardlock = detectar_hardlock_57(fases_t[-1])
        print(f"  Hard-Lock 57: {hardlock}")

    print("\n" + "="*70)
    print(" SIMULACIÓN COMPLETADA")
    print("="*70)


if __name__ == "__main__":
    main()

In [ ]:

"""
SIMULACIÓN: DECOHERENCIA EN FUNCIÓN DE LA GEOMETRÍA DE RED
Hipótesis: La geometría hexagonal minimiza la tasa de decoherencia.
"""

import numpy as np
import matplotlib.pyplot as plt

def tasa_decoherencia(geometria, N_qubits, D=1.5):
    """
    Calcula la tasa de decoherencia γ para diferentes geometrías.

    geometria: 'hexagonal', 'cuadrada', 'triangular', 'aleatoria'
    """
    # Factor de empaquetamiento (fracción de área ocupada)
    factores = {
        'hexagonal': 0.9069,   # Empaquetamiento hexagonal óptimo
        'cuadrada': 0.7854,    # Empaquetamiento cuadrado
        'triangular': 0.5,     # Empaquetamiento triangular
        'aleatoria': 0.3       # Empaquetamiento aleatorio
    }

    f = factores.get(geometria, 0.5)

    # La tasa de decoherencia escala inversamente con el factor
    # de empaquetamiento (más orden → menos decoherencia)
    gamma = (D / N_qubits) * (1 - f)

    return gamma


def simular_chip_cuantico():
    """
    Simula el tiempo de coherencia para diferentes geometrías.
    """
    N_qubits = 57  # Hard-Lock 57
    geometrias = ['hexagonal', 'cuadrada', 'triangular', 'aleatoria']

    print("="*70)
    print(" SIMULACIÓN: GEOMETRÍA DE RED Y DECOHERENCIA")
    print("="*70)
    print(f"\nNúmero de qubits: {N_qubits}")
    print(f"Deuda de Información: 𝔇 = 1.5")
    print()

    resultados = {}
    for g in geometrias:
        gamma = tasa_decoherencia(g, N_qubits)
        tau = 1 / gamma  # Tiempo de coherencia
        resultados[g] = {'gamma': gamma, 'tau': tau}
        print(f"  {g:12s}: γ = {gamma:.6f}, τ = {tau:.2f} pasos")

    # Visualización
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # (a) Tasa de decoherencia
    ax = axes[0]
    nombres = list(resultados.keys())
    gammas = [resultados[g]['gamma'] for g in nombres]
    colores = ['green', 'blue', 'orange', 'red']
    ax.bar(nombres, gammas, color=colores, edgecolor='black')
    ax.set_ylabel('Tasa de decoherencia γ')
    ax.set_title('(a) Decoherencia por geometría')
    ax.grid(alpha=0.3, axis='y')

    # (b) Tiempo de coherencia
    ax = axes[1]
    taus = [resultados[g]['tau'] for g in nombres]
    ax.bar(nombres, taus, color=colores, edgecolor='black')
    ax.set_ylabel('Tiempo de coherencia τ')
    ax.set_title('(b) Coherencia por geometría')
    ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('chip_cuantico_geometria.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("\n[CONCLUSIÓN PRELIMINAR]")
    print("La geometría hexagonal minimiza la tasa de decoherencia.")
    print("Sin embargo, esto es una PREDICCIÓN DEL MODELO, no un resultado experimental.")
    print("La implementación práctica requiere ingeniería de materiales.")

    return resultados


if __name__ == "__main__":
    simular_chip_cuantico()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import eigsh
from scipy.optimize import curve_fit

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})

# ══════════════════════════════════════════════════════════════════════
# MÓDULO 1 · ONTOLOGÍA DEL VACÍO (E01–E10)
# ══════════════════════════════════════════════════════════════════════
class OntologiaVacio:
    """Red hexagonal periódica que representa el sustrato del vacío."""

    def __init__(self, Nx=4, J=1.0):
        # The honeycomb_graph function was removed in NetworkX 3.0.
        # Use hexagonal_lattice_graph as a replacement for a periodic hexagonal lattice.
        # The parameters 'm' and 'n' refer to the number of rows and columns of hexagons.
        G = nx.hexagonal_lattice_graph(m=Nx, n=Nx, periodic=True)
        self.G = nx.convert_node_labels_to_integers(G)
        self.N = self.G.number_of_nodes()
        self.edges = list(self.G.edges())
        self.J = J

    # --- E03: H = ½ Σ J[1 − cos(ψi − ψj)]  (versión linealizada) ---
    def laplaciano(self):
        N = self.N
        L = lil_matrix((N, N))
        for i, j in self.edges:
            L[i, i] += self.J
            L[j, j] += self.J
            L[i, j] -= self.J
            L[j, i] -= self.J
        return L.tocsr()

    # --- Verificación E01, E02, E06 ---
    def verificar_E01_E02_E06(self):
        L = self.laplaciano()
        eigs, vecs = eigsh(L, k=4, which='SM')
        idx = np.argsort(eigs)
        eigs, vecs = eigs[idx], vecs[:, idx]
        # E06: |ψ0> = 1/sqrt(Vol)
        psi_silencio = np.ones(self.N) / np.sqrt(self.N)
        overlap = abs(np.dot(psi_silencio, vecs[:, 0]))
        return dict(
            N=self.N,
            E_0=eigs[0],                              # E01 → 0
            gap=eigs[1] - eigs[0],
            norm=np.linalg.norm(vecs[:, 0]),          # E02 → 1
            overlap_silencio=overlap,                 # E06 → 1
        )

    # --- E05: potencial metaestable ---
    @staticmethod
    def V_metaestable(phi, D=1.0):
        return D * ((phi/np.pi)**4 - 2*(phi/np.pi)**2 + 1)

    # --- E07–E10: perturbación / auto-reflexión / emergencia de e ---
    @staticmethod
    def P_theta(theta):
        return np.cos(theta)

    @staticmethod
    def proyector_autoreflexion(psi0):
        return np.outer(psi0, psi0)

    @staticmethod
    def emergencia_e(psi0):
        R = OntologiaVacio.proyector_autoreflexion(psi0)
        # exp(R)|0> = e|0>  (R es proyector → exp(R)=I+(e-1)R)
        return np.exp(1.0) * psi0


# ══════════════════════════════════════════════════════════════════════
# MÓDULO 2 · MONTE CARLO METROPOLIS  +  ADAPTIVE ANNEALING
# ══════════════════════════════════════════════════════════════════════
def energia_XY(psi, edges, J=1.0):
    """E03: E = ½ Σ J[1 − cos(ψi − ψj)]"""
    E = 0.0
    for i, j in edges:
        E += J * (1.0 - np.cos(psi[i] - psi[j]))
    return 0.5 * E


def metropolis_XY(psi, edges, beta, sweeps, delta=0.8):
    N = len(psi)
    E_hist = np.empty(sweeps)
    for s in range(sweeps):
        for _ in range(N):
            i = np.random.randint(N)
            old = psi[i]
            new = old + delta * np.random.uniform(-1, 1)
            dE = 0.0
            for a, b in edges:
                if a == i:
                    dE += (1 - np.cos(new - psi[b])) - (1 - np.cos(old - psi[b]))
                elif b == i:
                    dE += (1 - np.cos(new - psi[a])) - (1 - np.cos(old - psi[a]))
            dE *= 0.5
            if dE < 0 or np.random.rand() < np.exp(-beta * dE):
                psi[i] = new
        E_hist[s] = energia_XY(psi, edges)
    return psi, E_hist


def adaptive_annealing_XY(psi0, edges, beta_min=0.5, beta_max=8.0,
                          n_stages=40, sweeps_per_stage=60):
    """β crece geométricamente (Adaptive Annealing)."""
    betas = np.geomspace(beta_min, beta_max, n_stages)
    psi = psi0.copy()
    E_all, beta_all = [], []
    for b in betas:
        psi, Eh = metropolis_XY(psi, edges, b, sweeps_per_stage)
        E_all.append(Eh)
        beta_all.append(np.full(sweeps_per_stage, b))
    return psi, np.concatenate(E_all), np.concatenate(beta_all)


# ══════════════════════════════════════════════════════════════════════
# MÓDULO 3 · "FORMAS" EMERGENTES → QUBITS DE UN CHIP CUÁNTICO
# ══════════════════════════════════════════════════════════════════════
def extraer_formas(psi, edges):
    """Detecta dominios (formas) por fase media local."""
    psi_mod = np.mod(psi, 2*np.pi)
    # orden local (magnetización)
    order = np.mean(np.exp(1j*psi_mod))
    # defectos: aristas con Δφ grande (>π/2)
    defectos = [(i, j) for (i, j) in edges
                if abs(np.angle(np.exp(1j*(psi[i]-psi[j])))) > np.pi/2]
    return dict(order=np.abs(order),
                fase_global=np.angle(order),
                n_defectos=len(defectos),
                defectos=defectos)


def simular_chip_cuantico(psi, edges, T2_max=200e-6):
    """
    Traduce las 'formas' (fases ψ) a parámetros del chip:
      · Coherencia T2 efectiva ∝ 1/Var(ψ)
      · Fidelidad de puerta de 2 qubits  F = cos²(Δψ/2)
      · Tasa de error ε = 1 − ⟨F⟩
    """
    var_psi = np.var(psi)
    T2_eff = T2_max / (1 + 5*var_psi)                 # decoherencia
    fids = np.cos((psi[np.array([i for i,_ in edges])]
                 - psi[np.array([j for _,j in edges])])/2.0)**2
    F_avg = np.mean(fids)
    eps = 1.0 - F_avg
    return dict(T2_eff=T2_eff, F_avg=F_avg, epsilon=eps,
                var_psi=var_psi, n_qubits=len(psi))


# ══════════════════════════════════════════════════════════════════════
# EJECUCIÓN PRINCIPAL
# ══════════════════════════════════════════════════════════════════════
if __name__ == "__main__":

    print("═"*70)
    print(" BLOQUE 1 · Verificación ontológica (E01–E10)")
    print("═"*70)
    onto = OntologiaVacio(Nx=4)
    chk = onto.verificar_E01_E02_E06()
    for k, v in chk.items():
        print(f"  {k:20s} = {v}")

    print("\n  E05 · V(φ) en φ=0,π/√2,π:",
          [round(onto.V_metaestable(x), 4) for x in (0, np.pi/np.sqrt(2), np.pi)])
    print("  E07 · P̂(0) =", onto.P_theta(0.0), " (debe ser 1 → E08 OK)")
    e_vec = onto.emergencia_e(np.ones(onto.N)/np.sqrt(onto.N))
    print(f"  E10 · ||exp(R)|0>|| = {np.linalg.norm(e_vec):.4f}  (≈ e = {np.e:.4f})")

    # ── Gap scaling 1D vs N  (verificación de criterio) ──
    Ns = np.arange(8, 60, 6)
    gaps = []
    for N in Ns:
        G = nx.path_graph(int(N))
        L = nx.laplacian_matrix(G).astype(float).toarray()
        w = np.linalg.eigvalsh(L)
        gaps.append(w[1] - w[0])
    gaps = np.array(gaps)
    fit = np.polyfit(1.0/Ns, gaps, 1)
    print(f"\n  Gap ≈ {fit[0]:.3f}/N + {fit[1]:.3f}  →  escala 1/N confirmada")

    # ══════════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print(" BLOQUE 2 · Monte Carlo + Adaptive Annealing (formas del vacío)")
    print("═"*70)
    N = onto.N
    psi0 = np.random.uniform(-np.pi, np.pi, N)   # estado inicial ruidoso
    psi_f, E_hist, beta_hist = adaptive_annealing_XY(
        psi0, onto.edges, beta_min=0.4, beta_max=10.0,
        n_stages=40, sweeps_per_stage=80
    )
    formas = extraer_formas(psi_f, onto.edges)
    print(f"  N = {N} nodos (qubits candidatos)")
    print(f"  Orden global |⟨e^{{iψ}}⟩|       = {formas['order']:.4f}")
    print(f"  Fase global arg              = {formas['fase_global']:.4f} rad")
    print(f"  Defectos topológicos         = {formas['n_defectos']}")
    print(f"  E_final (MC+AA)              = {energia_XY(psi_f, onto.edges):.6f}")

    # ══════════════════════════════════════════════════════════════════════
    print("\n" + "═"*70)
    print(" BLOQUE 3 · Simulación del chip cuántico (a partir de las formas)")
    print("═"*70)
    chip_ordenado = simular_chip_cuantico(psi_f, onto.edges)
    print(f"  T2 efectivo                  = {chip_ordenado['T2_eff']*1e6:.2f} µs")
    print(f"  Fidelidad media de puerta    = {chip_ordenado['F_avg']:.4f}")
    print(f"  Tasa de error ε              = {chip_ordenado['epsilon']:.4f}")
    print(f"  Varianza de fases (ruido)    = {chip_ordenado['var_psi']:.4f}")

    # Comparativa: chip caótico (sin AA)
    chip_caotico = simular_chip_cuantico(psi0, onto.edges)
    print("\n  --- Comparativa ---")
    print(f"  Sin AA (caótico) : T2 = {chip_caotico['T2_eff']*1e6:7.2f} µs | "
          f"F = {chip_caotico['F_avg']:.4f} | ε = {chip_caotico['epsilon']:.4f}")
    print(f"  Con AA (vacío)   : T2 = {chip_ordenado['T2_eff']*1e6:7.2f} µs | "
          f"F = {chip_ordenado['F_avg']:.4f} | ε = {chip_ordenado['epsilon']:.4f}")
    mejora = chip_caotico['epsilon'] / max(chip_ordenado['epsilon'], 1e-9)
    print(f"  → Mejora en error: ×{mejora:.1f}")


def energia_XY_con_campo(psi, edges, J=1.0, h=0.0):
    """E03 modificada: E = ½ Σ J[1−cos(ψi−ψj)] − h Σ cos(ψi)"""
    E = 0.0
    for i, j in edges:
        E += J * (1.0 - np.cos(psi[i] - psi[j]))
    E *= 0.5
    E -= h * np.sum(np.cos(psi))
    return E


def metropolis_XY_campo(psi, edges, beta, sweeps, J=1.0, h=0.0, delta=0.8):
    """Metropolis con campo externo."""
    N = len(psi)
    E_hist = np.empty(sweeps)
    for s in range(sweeps):
        for _ in range(N):
            i = np.random.randint(N)
            old = psi[i]
            new = old + delta * np.random.uniform(-1, 1)
            dE = 0.0
            for a, b in edges:
                if a == i:
                    dE += (1 - np.cos(new - psi[b])) - (1 - np.cos(old - psi[b]))
                elif b == i:
                    dE += (1 - np.cos(new - psi[a])) - (1 - np.cos(old - psi[a]))
            dE *= 0.5 * J
            # término de campo
            dE -= h * (np.cos(new) - np.cos(old))
            if dE < 0 or np.random.rand() < np.exp(-beta * dE):
                psi[i] = new
        E_hist[s] = energia_XY_con_campo(psi, edges, J, h)
    return psi, E_hist


    # ══════════════════════════════════════════════════════════════════════
    # GRÁFICAS
    # ══════════════════════════════════════════════════════════════════════
    fig, ax = plt.subplots(2, 2, figsize=(11, 8))

    # (1) Curva de annealing
    ax[0,0].plot(E_hist, lw=0.7, color='crimson')
    ax[0,0].set_xlabel("sweep"); ax[0,0].set_ylabel("E")
    ax[0,0].set_title("MC + Adaptive Annealing · E(t)")
    ax[0,0].grid(alpha=0.3)

    # (2) Fases finales sobre la red
    pos = nx.spring_layout(onto.G, seed=1)
    colors = np.angle(np.exp(1j*psi_f))
    nx.draw_networkx_edges(onto.G, pos, ax=ax[0,1], alpha=0.25)
    nx.draw_networkx_nodes(onto.G, pos, ax=ax[0,1],
                           node_color=colors, cmap='hsv',
                           node_size=70, edgecolors='k', linewidths=0.4)
    ax[0,1].set_title(f"Formas emergentes (ψ) · defectos={formas['n_defectos']}")
    ax[0,1].axis('off')

    # (3) Gap scaling
    ax[1,0].plot(Ns, gaps, 'o', label='gap numérico')
    ax[1,0].plot(Ns, fit[0]/Ns + fit[1], '--', label=f'ajuste {fit[0]:.2f}/N')
    ax[1,0].set_xlabel("N"); ax[1,0].set_ylabel("ΔE")
    ax[1,0].set_title("Gap fundamental ↔ 1/N"); ax[1,0].legend(); ax[1,0].grid(alpha=0.3)

    # (4) Comparativa de chips
    labels = ['sin AA\n(caótico)', 'con AA\n(vacío)']
    T2s = [chip_caotico['T2_eff']*1e6, chip_ordenado['T2_eff']*1e6]
    Fs  = [chip_caotico['F_avg'], chip_ordenado['F_avg']]
    ax2 = ax[1,1]
    ax2.bar(np.arange(2)-0.15, T2s, 0.3, label='T2 (µs)', color='steelblue')
    ax2.set_ylabel('T2 (µs)', color='steelblue')
    ax2b = ax2.twinx()
    ax2b.bar(np.arange(2)+0.15, Fs, 0.3, label='Fidelidad', color='darkorange')
    ax2b.set_ylabel('Fidelidad', color='darkorange')
    ax2.set_xticks([0,1]); ax2.set_xticklabels(labels)
    ax2.set_title("Chip cuántico: caos vs. vacío ordenado")
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("mc_aa_vacio_chip.png", bbox_inches='tight')
    plt.show()
    print("\n[✓] Figura guardada como 'mc_aa_vacio_chip.png'")

In [ ]:

def energia_XY_con_campo(psi, edges, J=1.0, h=0.0):
    """E03 modificada: E = ½ Σ J[1−cos(ψi−ψj)] − h Σ cos(ψi)"""
    E = 0.0
    for i, j in edges:
        E += J * (1.0 - np.cos(psi[i] - psi[j]))
    E *= 0.5
    E -= h * np.sum(np.cos(psi))
    return E


def metropolis_XY_campo(psi, edges, beta, sweeps, J=1.0, h=0.0, delta=0.8):
    """Metropolis con campo externo."""
    N = len(psi)
    E_hist = np.empty(sweeps)
    for s in range(sweeps):
        for _ in range(N):
            i = np.random.randint(N)
            old = psi[i]
            new = old + delta * np.random.uniform(-1, 1)
            dE = 0.0
            for a, b in edges:
                if a == i:
                    dE += (1 - np.cos(new - psi[b])) - (1 - np.cos(old - psi[b]))
                elif b == i:
                    dE += (1 - np.cos(new - psi[a])) - (1 - np.cos(old - psi[a]))
            dE *= 0.5 * J
            # término de campo
            dE -= h * (np.cos(new) - np.cos(old))
            if dE < 0 or np.random.rand() < np.exp(-beta * dE):
                psi[i] = new
        E_hist[s] = energia_XY_con_campo(psi, edges, J, h)
    return psi, E_hist